# PQID Seed Generation Pipeline — Quality-Aware Regime

This notebook orchestrates the **quality-aware instruction-generation stack** for the 2026 PQID rebuild.

It does not replace the older thesis-era `generate_seeds.py` / `generate_paraphrases.py` workflow. Instead, it provides a dedicated, reproducible pipeline for:

1. building a **role-conditioned routing manifest** from the full enriched corpus,
2. deriving the two implemented supervision branches,
3. auditing the resulting role distribution,
4. calibrating the seed-draft temperature under a documented high-rigor protocol,
5. generating documented **production source-code seeds**,
6. generating documented **production teacher-text seeds**, and
7. generating documented **quality-aware paraphrases** across both branches.

The methodological rationale is documented in `PQID/SEED_GENERATION_DESIGN.md`.

Current implemented seed-draft settings:

- API interface: `Responses API`
- default teacher model: `gpt-5.4`
- temperature: `0.1`
- `max_output_tokens`: `220`
- concurrency: `12`
- branch structure: `source_code` plus `teacher_text`

Current documented paraphrase stage:

- current default model: `gpt-5.4-mini`
- current operational temperature: `0.2`
- paraphrases per seed: `5`
- intended upstream source: both quality-aware seed branches
- rationale: the paraphrase stage is a narrower reformulation task than seed drafting, so it currently uses the smaller model plus slightly higher temperature as an operational, not yet calibration-frozen, choice
- note: paraphrase-specific calibration is still a later dedicated task


In [ ]:
import json
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"

ENRICHED_CORPUS = PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl"
MASTER_CORPUS = PROCESSED_DIR / "pqid_2026_master_corpus.jsonl"
MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1.jsonl"
MANIFEST_REPORT = PROCESSED_DIR / "seed_role_manifest_v1_report.md"
SOURCE_CODE_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl"
TEACHER_TEXT_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl"
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl"
TEACHER_TEXT_MUTATION_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl"
PILOT_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_pilot_balanced.jsonl"
STUDY_PILOT_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_pilot_balanced_study.jsonl"
SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_v1.jsonl"
SEED_DRAFT_ERRORS = PROCESSED_DIR / "seed_drafts_quality_aware_v1_errors.jsonl"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_SEED_DRAFT_ERRORS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl"
PRODUCTION_TEACHER_TEXT_SEEDS = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"
PRODUCTION_TEACHER_TEXT_ERRORS = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl"
QUALITY_AWARE_SOURCE_CODE_PARAPHRASES = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl"
QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl"
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl"
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl"
BATCH_DIR = PROCESSED_DIR / "openai_batch_jobs"
SOURCE_CODE_SEED_BATCH_REQUEST_FILE = BATCH_DIR / "source_code_seed_requests_v1.jsonl"
SOURCE_CODE_SEED_BATCH_STATE_FILE = BATCH_DIR / "source_code_seed_batch_v1.json"
SOURCE_CODE_SEED_BATCH_OUTPUT_FILE = BATCH_DIR / "source_code_seed_batch_output_v1.jsonl"
SOURCE_CODE_SEED_BATCH_ERROR_FILE = BATCH_DIR / "source_code_seed_batch_error_v1.jsonl"
SOURCE_CODE_RETRY_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code_retry.jsonl"
SOURCE_CODE_RETRY_BATCH_REQUEST_FILE = BATCH_DIR / "source_code_seed_retry_requests_v1.jsonl"
SOURCE_CODE_RETRY_BATCH_STATE_FILE = BATCH_DIR / "source_code_seed_retry_batch_v1.json"
SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE = BATCH_DIR / "source_code_seed_retry_batch_output_v1.jsonl"
SOURCE_CODE_RETRY_BATCH_ERROR_FILE = BATCH_DIR / "source_code_seed_retry_batch_error_v1.jsonl"
TEACHER_TEXT_SEED_BATCH_REQUEST_FILE = BATCH_DIR / "teacher_text_seed_requests_v1.jsonl"
TEACHER_TEXT_SEED_BATCH_STATE_FILE = BATCH_DIR / "teacher_text_seed_batch_v1.json"
TEACHER_TEXT_SEED_BATCH_OUTPUT_FILE = BATCH_DIR / "teacher_text_seed_batch_output_v1.jsonl"
TEACHER_TEXT_SEED_BATCH_ERROR_FILE = BATCH_DIR / "teacher_text_seed_batch_error_v1.jsonl"
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl"
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = BATCH_DIR / "teacher_text_validation_seed_batch_v1.json"
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl"
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl"
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl"
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json"
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl"
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl"
SOURCE_CODE_PARAPHRASE_BATCH_REQUEST_FILE = BATCH_DIR / "source_code_paraphrase_requests_v1.jsonl"
SOURCE_CODE_PARAPHRASE_BATCH_STATE_FILE = BATCH_DIR / "source_code_paraphrase_batch_v1.json"
SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE = BATCH_DIR / "source_code_paraphrase_batch_output_v1.jsonl"
SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE = BATCH_DIR / "source_code_paraphrase_batch_error_v1.jsonl"
TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE = BATCH_DIR / "teacher_text_paraphrase_requests_v1.jsonl"
TEACHER_TEXT_PARAPHRASE_BATCH_STATE_FILE = BATCH_DIR / "teacher_text_paraphrase_batch_v1.json"
TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE = BATCH_DIR / "teacher_text_paraphrase_batch_output_v1.jsonl"
TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE = BATCH_DIR / "teacher_text_paraphrase_batch_error_v1.jsonl"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
TEACHER_TEXT_MODEL_COMPARISON_DIR = PROCESSED_DIR / "teacher_text_model_comparison"
TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_model_study.jsonl"

BUILD_MANIFEST_SCRIPT = SCRIPTS_DIR / "build_seed_role_manifest.py"
GENERATE_DRAFTS_SCRIPT = SCRIPTS_DIR / "generate_seed_drafts_quality_aware.py"
GENERATE_PARAPHRASES_SCRIPT = SCRIPTS_DIR / "generate_paraphrases_quality_aware.py"
PREPARE_SEED_BATCH_SCRIPT = SCRIPTS_DIR / "prepare_seed_drafts_quality_aware_batch.py"
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = SCRIPTS_DIR / "build_missing_seed_retry_manifest.py"
MATERIALIZE_SEED_BATCH_SCRIPT = SCRIPTS_DIR / "materialize_seed_drafts_quality_aware_batch.py"
PREPARE_PARAPHRASE_BATCH_SCRIPT = SCRIPTS_DIR / "prepare_paraphrases_quality_aware_batch.py"
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py"
RUN_BATCH_JOB_SCRIPT = SCRIPTS_DIR / "run_openai_batch_job.py"
PREFLIGHT_TEACHER_TEXT_SCRIPT = SCRIPTS_DIR / "preflight_teacher_text_production.py"

print("root:", ROOT)
print("enriched corpus:", ENRICHED_CORPUS.exists(), ENRICHED_CORPUS)
print("master corpus:", MASTER_CORPUS.exists(), MASTER_CORPUS)
print("build manifest script:", BUILD_MANIFEST_SCRIPT.exists())
print("generate drafts script:", GENERATE_DRAFTS_SCRIPT.exists())
print("generate paraphrases script:", GENERATE_PARAPHRASES_SCRIPT.exists())
print("prepare seed batch script:", PREPARE_SEED_BATCH_SCRIPT.exists())
print("build missing seed retry manifest script:", BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT.exists())
print("materialize seed batch script:", MATERIALIZE_SEED_BATCH_SCRIPT.exists())
print("prepare paraphrase batch script:", PREPARE_PARAPHRASE_BATCH_SCRIPT.exists())
print("materialize paraphrase batch script:", MATERIALIZE_PARAPHRASE_BATCH_SCRIPT.exists())
print("run batch job script:", RUN_BATCH_JOB_SCRIPT.exists())


## Local Secret Auto-Discovery

This cell mirrors the secret-discovery logic used elsewhere in the PQID workflow.

It searches common user directories for the named secret files:

- `OPENAI_API_KEY_PQID_GPT54_V2.txt`
- `OPENAI_API_KEY_PQID_GPT54_V1.txt`
- `OPENAI_API_KEY_PQID_V1.txt`
- `GITHUB_TOKEN_PQID_V1.txt`

If found, it sets the corresponding environment-variable file overrides in the current kernel:

- `OPENAI_API_KEY_FILE`
- `GITHUB_TOKEN_FILE`

This keeps absolute local secret paths out of the notebook source.

In [ ]:
import os

def discover_named_secret(filename: str):
    home = Path.home()
    search_roots = [
        home / ".pqid_secrets",
        home / "Desktop",
        home / "Documents",
        home / "Downloads",
        home / "AppData" / "Roaming",
        home / "AppData" / "Local",
    ]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for candidate in root.rglob(filename):
                if candidate.is_file():
                    return candidate
        except (OSError, PermissionError):
            continue
    return None

openai_secret = discover_named_secret("OPENAI_API_KEY_PQID_GPT54_V2.txt")
if openai_secret is None:
    openai_secret = discover_named_secret("OPENAI_API_KEY_PQID_GPT54_V1.txt")
if openai_secret is None:
    openai_secret = discover_named_secret("OPENAI_API_KEY_PQID_V1.txt")
github_secret = discover_named_secret("GITHUB_TOKEN_PQID_V1.txt")

if openai_secret and not os.environ.get("OPENAI_API_KEY_FILE", "").strip():
    os.environ["OPENAI_API_KEY_FILE"] = str(openai_secret)

if github_secret and not os.environ.get("GITHUB_TOKEN_FILE", "").strip():
    os.environ["GITHUB_TOKEN_FILE"] = str(github_secret)

print("OPENAI_API_KEY_FILE discovered:", bool(os.environ.get("OPENAI_API_KEY_FILE", "").strip()))
print("GITHUB_TOKEN_FILE discovered:", bool(os.environ.get("GITHUB_TOKEN_FILE", "").strip()))
print("openai named secret found:", openai_secret is not None)
print("github named secret found:", github_secret is not None)

## Preflight — GPT-5.4 API Readiness

Before running the live draft stage, verify that the **current notebook interpreter** can actually call the OpenAI API.

This stage checks:

1. whether the `openai` package is installed in the active kernel,
2. whether the API key is discoverable through the current PQID secret-loading logic,
3. which teacher model is configured for the pilot.

The quality-aware seed generator uses the **Responses API** with `gpt-5.4`.

Current pilot defaults in the live draft script:

- temperature: `0.1`
- `max_output_tokens`: `220`
- concurrency: `12`

Why `temperature = 0.1`:

- selected by the notebook's documented Stage F high-rigor confirmation protocol
- retained because it was the only non-dominated candidate under the predeclared automatic criteria
- treated as an automatic operational calibration result rather than a human-preference claim


In [ ]:
import importlib.util
import os

print("openai installed in current kernel:", importlib.util.find_spec("openai") is not None)
print("OPENAI_API_KEY present:", bool(os.environ.get("OPENAI_API_KEY", "").strip()))
print("OPENAI_API_KEY_FILE present:", bool(os.environ.get("OPENAI_API_KEY_FILE", "").strip()))
print("planned teacher model:", "gpt-5.4")

### Optional Install / Upgrade Cell

Run this only if the preflight cell reports that `openai` is missing in the current notebook kernel.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "openai"],
    check=True,
        capture_output=True,
        text=True,
)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("openai install/upgrade completed with return code:", result.returncode)

## Stage A — Build the Seed Role Manifest

This stage assigns each source record in the routing source to a seed-generation role such as:

- `gold_generation`
- `broad_generation`
- `mutation_robustness`
- `repair_or_explanation`
- `validation_diagnosis`

The manifest is the explicit bridge between the benchmark-readiness metadata and the later seed-generation prompts.

In this notebook, the **routing manifest** is built from the full enriched corpus so every record can be assigned the appropriate role. Master-corpus readiness metadata is overlaid where available. The first live generation branch then filters that routing manifest down to the subset that can be supervised directly with source code targets.

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        str(BUILD_MANIFEST_SCRIPT),
        "--input-file", str(ENRICHED_CORPUS),
        "--output-file", str(MANIFEST_FILE),
        "--report-file", str(MANIFEST_REPORT),
        "--source-artifact-name", ENRICHED_CORPUS.name,
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("seed-role manifest build completed with return code:", result.returncode)

In [ ]:
role_counts = Counter()
response_mode_counts = Counter()
target_mode_counts = Counter()
tier_counts = Counter()

with open(MANIFEST_FILE, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        role_counts[row["seed_role"]] += 1
        response_mode_counts[row["expected_response_mode"]] += 1
        target_mode_counts[row["target_supervision_mode"]] += 1
        tier_counts[row["readiness"].get("benchmark_suitability_tier_v2", "<missing>")] += 1

print("Seed role distribution")
for key, value in role_counts.most_common():
    print(f"  {key}: {value:,}")

print("\nExpected response modes")
for key, value in response_mode_counts.most_common():
    print(f"  {key}: {value:,}")

print("\nTarget supervision modes")
for key, value in target_mode_counts.most_common():
    print(f"  {key}: {value:,}")

print("\nManifest n/8 tier distribution")
for key, value in tier_counts.most_common():
    print(f"  {key}: {value:,}")

## Stage A2 — Derive the Live Supervision Branches

The full routing manifest covers **all readiness levels**, including both code-target and text-target supervision.

This cell derives the two current live branches explicitly:

- `source_code`
  - `gold_generation`
  - `broad_generation`
  - `repair_or_explanation`
- `teacher_text`
  - `validation_diagnosis`
  - `mutation_robustness`

The branch split is an implementation detail, not a scope reduction. The full corpus is only considered covered once both branches have been generated.


In [ ]:
source_code_rows = []
teacher_text_rows = []
with open(MANIFEST_FILE, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        mode = row.get("target_supervision_mode")
        if mode == "source_code":
            source_code_rows.append(row)
        elif mode == "teacher_text":
            teacher_text_rows.append(row)

with open(SOURCE_CODE_MANIFEST_FILE, "w", encoding="utf-8") as f:
    for row in source_code_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(TEACHER_TEXT_MANIFEST_FILE, "w", encoding="utf-8") as f:
    for row in teacher_text_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

source_code_role_counts = Counter(row["seed_role"] for row in source_code_rows)
teacher_text_role_counts = Counter(row["seed_role"] for row in teacher_text_rows)

print("source-code supervision manifest written:", SOURCE_CODE_MANIFEST_FILE)
print("source-code role distribution")
for key, value in source_code_role_counts.most_common():
    print(f"  {key}: {value:,}")
print("total source-code rows:", len(source_code_rows))

print("\nteacher-text supervision manifest written:", TEACHER_TEXT_MANIFEST_FILE)
print("teacher-text role distribution")
for key, value in teacher_text_role_counts.most_common():
    print(f"  {key}: {value:,}")
print("total teacher-text rows:", len(teacher_text_rows))

print("\nfull routed rows recovered:", len(source_code_rows) + len(teacher_text_rows))


## Stage A3 — Build a Balanced Pilot Manifest

The source-code branch is still imbalanced, so the first live teacher-model pilot should sample across its major roles instead of following corpus order.

This balanced pilot is intentionally small and deterministic.

In [ ]:
from collections import defaultdict

REVIEW_ROLE_QUOTAS = {
    "gold_generation": 2,
    "broad_generation": 2,
    "repair_or_explanation": 2,
}

STUDY_ROLE_QUOTAS = {
    "gold_generation": 6,
    "broad_generation": 6,
    "repair_or_explanation": 6,
}

def write_balanced_manifest(input_manifest, output_manifest, role_quotas):
    grouped_rows = defaultdict(list)
    with open(input_manifest, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            grouped_rows[row["seed_role"]].append(row)

    selected_rows = []
    for role, quota in role_quotas.items():
        rows = grouped_rows.get(role, [])
        if not rows:
            print(f"missing role in source-code manifest: {role}")
            continue
        selected_rows.extend(rows[: min(quota, len(rows))])

    selected_rows.sort(key=lambda row: (row["seed_role"], row["source_record"].get("circuit_hash", "")))

    with open(output_manifest, "w", encoding="utf-8") as f:
        for row in selected_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    role_counts = Counter(row["seed_role"] for row in selected_rows)
    return selected_rows, role_counts

pilot_rows, pilot_role_counts = write_balanced_manifest(SOURCE_CODE_MANIFEST_FILE, PILOT_MANIFEST_FILE, REVIEW_ROLE_QUOTAS)
print("balanced pilot manifest written:", PILOT_MANIFEST_FILE)
print("pilot role distribution")
for key, value in pilot_role_counts.items():
    print(f"  {key}: {value:,}")
print("total pilot rows:", len(pilot_rows))

study_rows, study_role_counts = write_balanced_manifest(SOURCE_CODE_MANIFEST_FILE, STUDY_PILOT_MANIFEST_FILE, STUDY_ROLE_QUOTAS)
print("\nstudy pilot manifest written:", STUDY_PILOT_MANIFEST_FILE)
print("study role distribution")
for key, value in study_role_counts.items():
    print(f"  {key}: {value:,}")
print("total study rows:", len(study_rows))

## Stage B — Pilot Draft Generation

This stage runs a **pilot-only draft pass** for the current `source_code` supervision branch. It is intentionally resume-safe and dry-run capable.

Recommended workflow:

1. keep `DRY_RUN = True` first,
2. inspect the printed prompt payload,
3. set `DRY_RUN = False` only when the role framing looks right,
4. start with the **balanced source-code pilot manifest** before scaling to the full source-code branch.

The default teacher model is `gpt-5.4`.

Active implemented generation settings:

- API interface: `Responses API`
- temperature: `0.1`
- `max_output_tokens`: `220`
- concurrency: `12`
- prompt style goal: role fidelity plus wording diversity
- current pilot size: `6` balanced examples

Temperature rationale:

- selected by the notebook's documented calibration ladder rather than by rule of thumb
- lower temperature was retained after the higher-rigor automatic comparison favored semantic quality over looser phrasing variation
- the production default is now frozen at `0.1` for the source-code seed draft stage


In [ ]:
MODEL = "gpt-5.4"
TEMPERATURE = 0.1
ACTIVE_MANIFEST_FILE = PILOT_MANIFEST_FILE if PILOT_MANIFEST_FILE.exists() else SOURCE_CODE_MANIFEST_FILE
MAX_RECORDS = 6
DRY_RUN = False

print("active manifest:", ACTIVE_MANIFEST_FILE)
print("temperature:", TEMPERATURE)

cmd = [
    sys.executable,
    str(GENERATE_DRAFTS_SCRIPT),
    "--manifest-file", str(ACTIVE_MANIFEST_FILE),
    "--source-file", str(ENRICHED_CORPUS),
    "--output-file", str(SEED_DRAFTS),
    "--log-file", str(SEED_DRAFT_ERRORS),
    "--model", MODEL,
    "--temperature", str(TEMPERATURE),
    "--max-records", str(MAX_RECORDS),
]

if DRY_RUN:
    cmd.append("--dry-run")

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("seed-draft stage completed with return code:", result.returncode)

In [ ]:
if not SEED_DRAFTS.exists():
    print("No seed draft file yet.")
    if SEED_DRAFT_ERRORS.exists():
        print("\nRecent error-log preview:\n")
        with open(SEED_DRAFT_ERRORS, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                row = json.loads(line)
                print(json.dumps({
                    "error_type": row.get("error_type"),
                    "error_message": row.get("error_message"),
                    "seed_role": row.get("seed_role"),
                }, ensure_ascii=False, indent=2))
                print()
    else:
        print("Run the pilot cell with DRY_RUN = False to materialize outputs.")
else:
    import re
    from collections import Counter

    draft_rows = []
    draft_role_counts = Counter()
    with open(SEED_DRAFTS, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            draft_rows.append(row)
            draft_role_counts[row["metadata"].get("seed_role", "<missing>")] += 1
    def normalize_seed_text(text: str) -> str:
        return re.sub(r"\s+", " ", text.strip().lower())

    opener_counts = Counter()
    normalized_counts = Counter()
    for row in draft_rows:
        text = row.get("input", "").strip()
        opener = " ".join(text.split()[:2]).lower() if text else "<empty>"
        opener_counts[opener] += 1
        normalized_counts[normalize_seed_text(text)] += 1

    print("Seed draft counts by role")
    for key, value in draft_role_counts.most_common():
        print(f"  {key}: {value:,}")

    duplicate_count = sum(1 for value in normalized_counts.values() if value > 1)
    repeated_openers = {key: value for key, value in opener_counts.items() if value > 1}
    print("\nDuplicate/variation audit")
    print(f"  exact normalized duplicates: {duplicate_count}")
    if repeated_openers:
        print("  repeated opening patterns:")
        for key, value in repeated_openers.items():
            print(f"    {key}: {value}")
    else:
        print("  repeated opening patterns: none")

    if len(draft_rows) <= 10:
        print("\nFull pilot draft review:\n")
        for idx, row in enumerate(draft_rows, start=1):
            meta = row.get("metadata", {})
            print(f"[{idx}] role = {meta.get('seed_role')} | response = {meta.get('seed_expected_response_mode')} | temp = {meta.get('seed_generation_temperature')}")
            print(row.get("input", ""))
            print()
    elif draft_rows:
        first = draft_rows[0]
        print("\nSample draft input:\n")
        print(first["input"])
        print("\nSample draft metadata slice:\n")
        print(json.dumps({
            "seed_role": first["metadata"].get("seed_role"),
            "seed_learning_objective": first["metadata"].get("seed_learning_objective"),
            "seed_expected_response_mode": first["metadata"].get("seed_expected_response_mode"),
            "seed_generation_stage": first["metadata"].get("seed_generation_stage"),
            "seed_generation_temperature": first["metadata"].get("seed_generation_temperature"),
            "seed_template_version": first["metadata"].get("seed_template_version"),
        }, indent=2))

## Stage C — Temperature Comparison Mini-Study

This stage generates matched pilot batches at `0.1`, `0.3`, and `0.5` using the same balanced pilot manifest.

The goal is to support the choice of `0.3` with examples rather than intuition alone. We compare:

- exact normalized duplicate rate,
- repeated opening patterns,
- and the side-by-side wording of matched pilot records.

Use a fresh comparison label whenever you want a new clean run.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"
ENRICHED_CORPUS = PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl"
STUDY_PILOT_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_pilot_balanced_study.jsonl"
GENERATE_DRAFTS_SCRIPT = SCRIPTS_DIR / "generate_seed_drafts_quality_aware.py"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"

TEMPERATURE_GRID = [0.1, 0.3, 0.5]
COMPARISON_LABEL = "tempstudy_v2"
COMPARISON_MODEL = "gpt-5.4"
COMPARISON_MAX_RECORDS = 18
COMPARISON_DRY_RUN = False

TEMPERATURE_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
if not STUDY_PILOT_MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Missing study pilot manifest: {STUDY_PILOT_MANIFEST_FILE}. Rerun Stage A3 before running the temperature study."
    )

for temp in TEMPERATURE_GRID:
    temp_suffix = f"{temp:.1f}".replace(".", "p")
    output_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{COMPARISON_LABEL}_temp_{temp_suffix}.jsonl"
    log_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{COMPARISON_LABEL}_temp_{temp_suffix}_errors.jsonl"
    print(f"\nrunning comparison batch at temperature={temp} -> {output_file.name}")
    cmd = [
        sys.executable,
        str(GENERATE_DRAFTS_SCRIPT),
        "--manifest-file", str(STUDY_PILOT_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--output-file", str(output_file),
        "--log-file", str(log_file),
        "--model", COMPARISON_MODEL,
        "--temperature", str(temp),
        "--max-records", str(COMPARISON_MAX_RECORDS),
    ]
    if COMPARISON_DRY_RUN:
        cmd.append("--dry-run")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("return code:", result.returncode)

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"

def normalize_seed_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())

comparison_rows = {}
for temp in [0.1, 0.3, 0.5]:
    temp_suffix = f"{temp:.1f}".replace(".", "p")
    output_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{COMPARISON_LABEL}_temp_{temp_suffix}.jsonl"
    if not output_file.exists():
        print(f"missing comparison output for temperature={temp}: {output_file}")
        continue
    rows = []
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    comparison_rows[temp] = rows

for temp, rows in comparison_rows.items():
    opener_counts = Counter()
    normalized_counts = Counter()
    for row in rows:
        text = row.get("input", "").strip()
        opener = " ".join(text.split()[:2]).lower() if text else "<empty>"
        opener_counts[opener] += 1
        normalized_counts[normalize_seed_text(text)] += 1
    duplicate_count = sum(1 for value in normalized_counts.values() if value > 1)
    repeated_openers = {key: value for key, value in opener_counts.items() if value > 1}
    print(f"\nTemperature {temp}")
    print(f"  rows: {len(rows)}")
    print(f"  exact normalized duplicates: {duplicate_count}")
    if repeated_openers:
        print("  repeated opening patterns:")
        for key, value in repeated_openers.items():
            print(f"    {key}: {value}")
    else:
        print("  repeated opening patterns: none")

indexed_rows = {}
for temp, rows in comparison_rows.items():
    indexed_rows[temp] = {}
    for row in rows:
        meta = row.get("metadata", {})
        key = (meta.get("circuit_hash"), meta.get("seed_role"))
        indexed_rows[temp][key] = row

common_keys = None
for temp, mapping in indexed_rows.items():
    keys = set(mapping.keys())
    common_keys = keys if common_keys is None else (common_keys & keys)
common_keys = sorted(common_keys or [])

if common_keys:
    print("\nSide-by-side prompt comparison (matched by circuit_hash + seed_role)\n")
    for idx, key in enumerate(common_keys, start=1):
        circuit_hash, role = key
        print(f"Sample {idx} | role={role} | circuit_hash={circuit_hash}")
        for temp in sorted(indexed_rows):
            row = indexed_rows[temp][key]
            print(f"  temp={temp}")
            print(f"    {row.get('input', '')}")
        print()

## Stage D — Advanced Temperature Calibration

This second calibration stage preserves the earlier broad comparison (`0.1 / 0.3 / 0.5`) and adds a finer-grained low-temperature study.

The goal is to compare:

- `0.1`
- `0.2`
- `0.3`

on the same matched `18`-example study manifest. This is the more relevant calibration once higher temperature (`0.5`) has already been shown to be less stable.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"
ENRICHED_CORPUS = PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl"
STUDY_PILOT_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_pilot_balanced_study.jsonl"
GENERATE_DRAFTS_SCRIPT = SCRIPTS_DIR / "generate_seed_drafts_quality_aware.py"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"

ADV_TEMPERATURE_GRID = [0.1, 0.2, 0.3]
ADV_COMPARISON_LABEL = "tempstudy_v3"
ADV_COMPARISON_MODEL = "gpt-5.4"
ADV_COMPARISON_MAX_RECORDS = 18
ADV_COMPARISON_DRY_RUN = False

TEMPERATURE_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
if not STUDY_PILOT_MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Missing study pilot manifest: {STUDY_PILOT_MANIFEST_FILE}. Rerun Stage A3 before running the advanced calibration study."
    )

for temp in ADV_TEMPERATURE_GRID:
    temp_suffix = f"{temp:.1f}".replace(".", "p")
    output_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{ADV_COMPARISON_LABEL}_temp_{temp_suffix}.jsonl"
    log_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{ADV_COMPARISON_LABEL}_temp_{temp_suffix}_errors.jsonl"
    print(f"\nrunning advanced calibration batch at temperature={temp} -> {output_file.name}")
    cmd = [
        sys.executable,
        str(GENERATE_DRAFTS_SCRIPT),
        "--manifest-file", str(STUDY_PILOT_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--output-file", str(output_file),
        "--log-file", str(log_file),
        "--model", str(ADV_COMPARISON_MODEL),
        "--temperature", str(temp),
        "--max-records", str(ADV_COMPARISON_MAX_RECORDS),
    ]
    if ADV_COMPARISON_DRY_RUN:
        cmd.append("--dry-run")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("return code:", result.returncode)

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
ADV_TEMPERATURE_GRID = [0.1, 0.2, 0.3]
ADV_COMPARISON_LABEL = "tempstudy_v3"

def normalize_seed_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())

comparison_rows = {}
for temp in ADV_TEMPERATURE_GRID:
    temp_suffix = f"{temp:.1f}".replace(".", "p")
    output_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{ADV_COMPARISON_LABEL}_temp_{temp_suffix}.jsonl"
    if not output_file.exists():
        print(f"missing comparison output for temperature={temp}: {output_file}")
        continue
    rows = []
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    comparison_rows[temp] = rows

for temp, rows in comparison_rows.items():
    opener_counts = Counter()
    normalized_counts = Counter()
    for row in rows:
        text = row.get("input", "").strip()
        opener = " ".join(text.split()[:2]).lower() if text else "<empty>"
        opener_counts[opener] += 1
        normalized_counts[normalize_seed_text(text)] += 1
    duplicate_count = sum(1 for value in normalized_counts.values() if value > 1)
    repeated_openers = {key: value for key, value in opener_counts.items() if value > 1}
    print(f"\nTemperature {temp}")
    print(f"  rows: {len(rows)}")
    print(f"  exact normalized duplicates: {duplicate_count}")
    if repeated_openers:
        print("  repeated opening patterns:")
        for key, value in repeated_openers.items():
            print(f"    {key}: {value}")
    else:
        print("  repeated opening patterns: none")

indexed_rows = {}
for temp, rows in comparison_rows.items():
    indexed_rows[temp] = {}
    for row in rows:
        meta = row.get("metadata", {})
        key = (meta.get("circuit_hash"), meta.get("seed_role"))
        indexed_rows[temp][key] = row

common_keys = None
for temp, mapping in indexed_rows.items():
    keys = set(mapping.keys())
    common_keys = keys if common_keys is None else (common_keys & keys)
common_keys = sorted(common_keys or [])

if common_keys:
    print("\nSide-by-side prompt comparison (matched by circuit_hash + seed_role)\n")
    for idx, key in enumerate(common_keys, start=1):
        circuit_hash, role = key
        print(f"Sample {idx} | role={role} | circuit_hash={circuit_hash}")
        for temp in sorted(indexed_rows):
            row = indexed_rows[temp][key]
            print(f"  temp={temp}")
            print(f"    {row.get('input', '')}")
        print()

## Stage E — Empirical Temperature Evaluation

This stage turns the temperature comparison into an explicit empirical audit rather than a descriptive reading of examples alone.

It evaluates both documented calibration studies:

- `Stage C` broad screen: `0.1 / 0.3 / 0.5`
- `Stage D` low-temperature refinement: `0.1 / 0.2 / 0.3`

The evaluation script computes:

- automatic semantic-fidelity indicators such as qubit-count alignment, measurement retention, parameter retention, global-phase retention, gate coverage, and unsupported-addition flags
- role-fidelity indicators for generation vs repair prompts
- lexical concentration diagnostics such as opener concentration and exact normalized duplicate rate
- pairwise exact sign tests on matched items for `overall_score`, `strict_pass`, and `unsupported_addition_count`

The goal is not to prove a temperature by fiat, but to make the calibration claim more statistically and semantically defensible.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
EVALUATE_TEMPERATURE_SCRIPT = SCRIPTS_DIR / "evaluate_seed_temperature_calibration.py"
TEMPERATURE_EVAL_REPORT = TEMPERATURE_COMPARISON_DIR / "seed_temperature_empirical_evaluation_v1.json"

cmd = [
    sys.executable,
    str(EVALUATE_TEMPERATURE_SCRIPT),
    "--comparison-dir", str(TEMPERATURE_COMPARISON_DIR),
    "--output-file", str(TEMPERATURE_EVAL_REPORT),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("return code:", result.returncode)
print("report file:", TEMPERATURE_EVAL_REPORT)


In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
TEMPERATURE_EVAL_REPORT = TEMPERATURE_COMPARISON_DIR / "seed_temperature_empirical_evaluation_v1.json"

if not TEMPERATURE_EVAL_REPORT.exists():
    raise FileNotFoundError(
        f"Missing empirical temperature evaluation report: {TEMPERATURE_EVAL_REPORT}. Run Stage E before auditing it."
    )

with open(TEMPERATURE_EVAL_REPORT, encoding="utf-8") as f:
    report = json.load(f)

print("temperature empirical evaluation report:", TEMPERATURE_EVAL_REPORT)
for study in report.get("studies", []):
    print(f"\n{study.get('name', '<unnamed study>')}")
    print("-" * len(study.get("name", "study")))
    if not study.get("available"):
        print("  not available")
        for path in study.get("missing_files", []):
            print("   missing:", path)
        continue
    print("  matched rows:", study.get("matched_rows"))
    print("  temperature summaries:")
    for temp in study.get("temps", []):
        summary = study.get("temperature_summaries", {}).get(str(temp))
        if not summary:
            continue
        print(
            "   "
            f" temp={temp} | overall={summary.get('overall_score_mean', 0.0):.4f} | "
            f"strict_pass={summary.get('strict_pass_rate', 0.0):.4f} | "
            f"drift={summary.get('drift_flag_rate', 0.0):.4f} | "
            f"unsupported_add_mean={summary.get('unsupported_addition_mean', 0.0):.4f} | "
            f"max_opener_share={summary.get('max_opener_share', 0.0):.4f}"
        )
    print("  pairwise overall_score sign tests:")
    for result in study.get("pairwise_sign_tests", []):
        if result.get("metric") != "overall_score":
            continue
        print(
            "   "
            f" {result.get('comparison')} | wins={result.get('wins_for_right')} | "
            f"losses={result.get('losses_for_right')} | ties={result.get('ties')} | "
            f"mean_diff={result.get('mean_signed_difference', 0.0):.4f} | "
            f"p={result.get('exact_sign_pvalue', 1.0):.4f}"
        )
    print("  flagged examples preview:")
    for temp in study.get("temps", []):
        examples = study.get("flagged_examples", {}).get(str(temp), [])
        print(f"   temp={temp} -> {len(examples)} preview items")
        for example in examples[:2]:
            print(
                "    - "
                f"{example.get('seed_role')} | {example.get('circuit_hash')} | "
                f"missing={example.get('missing_components')} | unsupported={example.get('unsupported_additions')}"
            )


## Stage F — High-Rigor Temperature Selection Protocol

This stage is the first calibration block explicitly designed for top-publication standards rather than quick operational tuning.

It adds three elements beyond the earlier mini-studies:

- a larger matched balanced manifest
- a predeclared selection rule
- a blinded human-annotation pack

Default confirmation design in this notebook:

- temperatures: `0.1`, `0.2`, `0.3`
- matched role-balanced manifest: `12 + 12 + 12 = 36`
- study label: `tempstudy_v4_highrigor`

Predeclared selection logic:

1. run the larger matched automatic study
2. eliminate any temperature that is strictly weaker than another on both `overall_score_mean` and `strict_pass_rate`
3. prepare the blinded human-annotation pack for the remaining candidates
4. choose the **lowest** temperature that is not materially worse than the best remaining candidate on human semantic fidelity and benchmark appropriateness
5. if multiple temperatures still remain tied, prefer the one with lower lexical concentration (`max_opener_share`) and better opener diversity

This stage is intentionally a protocol, not a post hoc justification layer.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
SOURCE_CODE_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl"
HIGHRIGOR_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_pilot_balanced_highrigor.jsonl"

HIGHRIGOR_ROLE_QUOTAS = {
    "gold_generation": 12,
    "broad_generation": 12,
    "repair_or_explanation": 12,
}

grouped_rows = defaultdict(list)
with open(SOURCE_CODE_MANIFEST_FILE, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        grouped_rows[row["seed_role"]].append(row)

selected_rows = []
for role, quota in HIGHRIGOR_ROLE_QUOTAS.items():
    rows = grouped_rows.get(role, [])
    if not rows:
        print(f"missing role in source-code manifest: {role}")
        continue
    selected_rows.extend(rows[: min(quota, len(rows))])

selected_rows.sort(key=lambda row: (row["seed_role"], row["source_record"].get("circuit_hash", "")))
with open(HIGHRIGOR_MANIFEST_FILE, "w", encoding="utf-8") as f:
    for row in selected_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

role_counts = Counter(row["seed_role"] for row in selected_rows)
print("high-rigor manifest written:", HIGHRIGOR_MANIFEST_FILE)
print("high-rigor role distribution")
for key, value in role_counts.items():
    print(f"  {key}: {value:,}")
print("total high-rigor rows:", len(selected_rows))


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"
ENRICHED_CORPUS = PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl"
HIGHRIGOR_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_pilot_balanced_highrigor.jsonl"
GENERATE_DRAFTS_SCRIPT = SCRIPTS_DIR / "generate_seed_drafts_quality_aware.py"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"

HIGHRIGOR_TEMPERATURE_GRID = [0.1, 0.2, 0.3]
HIGHRIGOR_COMPARISON_LABEL = "tempstudy_v4_highrigor"
HIGHRIGOR_COMPARISON_MODEL = "gpt-5.4"
HIGHRIGOR_COMPARISON_MAX_RECORDS = 36
HIGHRIGOR_COMPARISON_DRY_RUN = False

TEMPERATURE_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
if not HIGHRIGOR_MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Missing high-rigor study manifest: {HIGHRIGOR_MANIFEST_FILE}. Run the Stage F manifest build cell first."
    )

for temp in HIGHRIGOR_TEMPERATURE_GRID:
    temp_suffix = f"{temp:.1f}".replace(".", "p")
    output_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{HIGHRIGOR_COMPARISON_LABEL}_temp_{temp_suffix}.jsonl"
    log_file = TEMPERATURE_COMPARISON_DIR / f"seed_drafts_quality_aware_{HIGHRIGOR_COMPARISON_LABEL}_temp_{temp_suffix}_errors.jsonl"
    print(f"\nrunning high-rigor comparison batch at temperature={temp} -> {output_file.name}")
    cmd = [
        sys.executable,
        str(GENERATE_DRAFTS_SCRIPT),
        "--manifest-file", str(HIGHRIGOR_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--output-file", str(output_file),
        "--log-file", str(log_file),
        "--model", str(HIGHRIGOR_COMPARISON_MODEL),
        "--temperature", str(temp),
        "--max-records", str(HIGHRIGOR_COMPARISON_MAX_RECORDS),
    ]
    if HIGHRIGOR_COMPARISON_DRY_RUN:
        cmd.append("--dry-run")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("return code:", result.returncode)

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
EVALUATE_TEMPERATURE_SCRIPT = SCRIPTS_DIR / "evaluate_seed_temperature_calibration.py"
HIGHRIGOR_EVAL_REPORT = TEMPERATURE_COMPARISON_DIR / "seed_temperature_empirical_evaluation_v2_highrigor.json"

cmd = [
    sys.executable,
    str(EVALUATE_TEMPERATURE_SCRIPT),
    "--comparison-dir", str(TEMPERATURE_COMPARISON_DIR),
    "--output-file", str(HIGHRIGOR_EVAL_REPORT),
    "--study", "tempstudy_v4_highrigor|Stage F — High-Rigor Confirmation Study|0.1,0.2,0.3",
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("return code:", result.returncode)
print("report file:", HIGHRIGOR_EVAL_REPORT)


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
SCRIPTS_DIR = ROOT / "PQID/scripts/03_instruction_generation"
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
ANNOTATION_PACK_SCRIPT = SCRIPTS_DIR / "prepare_seed_temperature_annotation_pack.py"
ANNOTATION_PACK_PREFIX = TEMPERATURE_COMPARISON_DIR / "seed_temperature_annotation_pack_tempstudy_v4_highrigor"
ANNOTATION_RUBRIC = SCRIPTS_DIR / "TEMPERATURE_ANNOTATION_RUBRIC.md"

cmd = [
    sys.executable,
    str(ANNOTATION_PACK_SCRIPT),
    "--comparison-dir", str(TEMPERATURE_COMPARISON_DIR),
    "--label", "tempstudy_v4_highrigor",
    "--temps", "0.1,0.2,0.3",
    "--output-prefix", str(ANNOTATION_PACK_PREFIX),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("return code:", result.returncode)
print("annotation rubric:", ANNOTATION_RUBRIC)


In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
TEMPERATURE_COMPARISON_DIR = PROCESSED_DIR / "seed_temperature_comparison"
HIGHRIGOR_EVAL_REPORT = TEMPERATURE_COMPARISON_DIR / "seed_temperature_empirical_evaluation_v2_highrigor.json"
ANNOTATION_KEY_FILE = TEMPERATURE_COMPARISON_DIR / "seed_temperature_annotation_pack_tempstudy_v4_highrigor_key.json"

if not HIGHRIGOR_EVAL_REPORT.exists():
    raise FileNotFoundError(
        f"Missing Stage F evaluation report: {HIGHRIGOR_EVAL_REPORT}. Run the Stage F evaluation cell first."
    )

with open(HIGHRIGOR_EVAL_REPORT, encoding="utf-8") as f:
    report = json.load(f)

study = report.get("studies", [{}])[0]
print(study.get("name", "<unnamed study>"))
print("matched rows:", study.get("matched_rows"))
for temp in study.get("temps", []):
    summary = study.get("temperature_summaries", {}).get(str(temp), {})
    print(
        f"temp={temp} | overall={summary.get('overall_score_mean', 0.0):.4f} | "
        f"strict_pass={summary.get('strict_pass_rate', 0.0):.4f} | "
        f"max_opener_share={summary.get('max_opener_share', 0.0):.4f}"
    )
print("\npairwise overall_score sign tests")
for result in study.get("pairwise_sign_tests", []):
    if result.get("metric") != "overall_score":
        continue
    print(
        f"  {result.get('comparison')} | wins={result.get('wins_for_right')} | "
        f"losses={result.get('losses_for_right')} | ties={result.get('ties')} | "
        f"mean_diff={result.get('mean_signed_difference', 0.0):.4f} | "
        f"p={result.get('exact_sign_pvalue', 1.0):.4f}"
    )

if ANNOTATION_KEY_FILE.exists():
    with open(ANNOTATION_KEY_FILE, encoding="utf-8") as f:
        key_report = json.load(f)
    print("\nblinded annotation pack")
    print("  matched_rows    :", key_report.get("matched_rows"))
    print("  annotation_rows :", key_report.get("annotation_rows"))
    print("  temps           :", key_report.get("temps"))
else:
    print("\nannotation pack not found yet; run the Stage F annotation-pack cell.")


## Stage G — Production Source-Code Seed Draft Generation

This is the documented **full source-code seed generation** stage for the rebuilt PQID path.

It uses the entire `source_code` supervision manifest rather than the balanced pilot slice.

What this stage does:

- consumes `seed_role_manifest_v1_source_code.jsonl`
- keeps the frozen seed-draft temperature at `0.1`
- writes a production-oriented seed artifact separate from the small pilot file
- remains resume-safe so long-running generation can be restarted safely

Recommended use:

1. keep `PRODUCTION_DRY_RUN = True` for a first payload sanity check if you changed anything upstream
2. optionally set `PRODUCTION_MAX_RECORDS` to a small cap for a rehearsal run
3. set `PRODUCTION_MAX_RECORDS = None` for the full source-code branch
4. only treat these as final training seeds after the later critique/rewrite gate is in place

For the full-corpus production run, prefer the adjacent **Batch API** section once the prompts and settings are frozen. Keep the synchronous run cell for spot checks, pilot reruns, and narrow recovery work.


In [ ]:
PRODUCTION_MODEL = "gpt-5.4"
PRODUCTION_TEMPERATURE = 0.1
PRODUCTION_ACTIVE_MANIFEST = SOURCE_CODE_MANIFEST_FILE
PRODUCTION_MAX_RECORDS = None
PRODUCTION_DRY_RUN = False

print("production manifest:", PRODUCTION_ACTIVE_MANIFEST)
print("production output:", PRODUCTION_SEED_DRAFTS)
print("production temperature:", PRODUCTION_TEMPERATURE)

cmd = [
    sys.executable,
    str(GENERATE_DRAFTS_SCRIPT),
    "--manifest-file", str(PRODUCTION_ACTIVE_MANIFEST),
    "--source-file", str(ENRICHED_CORPUS),
    "--output-file", str(PRODUCTION_SEED_DRAFTS),
    "--log-file", str(PRODUCTION_SEED_DRAFT_ERRORS),
    "--model", PRODUCTION_MODEL,
    "--temperature", str(PRODUCTION_TEMPERATURE),
]

if PRODUCTION_MAX_RECORDS is not None:
    cmd.extend(["--max-records", str(PRODUCTION_MAX_RECORDS)])

if PRODUCTION_DRY_RUN:
    cmd.append("--dry-run")

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("production source-code seed generation completed with return code:", result.returncode)

In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_SEED_DRAFT_ERRORS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl"
SOURCE_CODE_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl"

if not PRODUCTION_SEED_DRAFTS.exists():
    print("No production seed draft file yet.")
    if PRODUCTION_SEED_DRAFT_ERRORS.exists():
        print("\nRecent error-log preview:\n")
        with open(PRODUCTION_SEED_DRAFT_ERRORS, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                row = json.loads(line)
                print(json.dumps({
                    "error_type": row.get("error_type"),
                    "error_message": row.get("error_message"),
                    "seed_role": row.get("seed_role"),
                }, ensure_ascii=False, indent=2))
                print()
    else:
        print("Run the production source-code seed cell to materialize outputs.")
else:
    role_counts = Counter()
    prompt_type_counts = Counter()
    temperature_counts = Counter()
    unique_hashes = set()
    rows = 0

    with open(PRODUCTION_SEED_DRAFTS, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            meta = row.get("metadata", {})
            role_counts[meta.get("seed_role", "<missing>")] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            temperature_counts[str(meta.get("seed_generation_temperature", "<missing>"))] += 1
            unique_hashes.add(meta.get("circuit_hash", ""))

    print("Production seed counts by role")
    for key, value in role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nPrompt types")
    for key, value in prompt_type_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nSeed draft temperatures present")
    for key, value in temperature_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nProduction seed coverage")
    print("  rows:", f"{rows:,}")
    print("  unique circuit_hash values:", f"{len(unique_hashes):,}")
    print(
        "  source-code manifest size:",
        sum(1 for _ in open(SOURCE_CODE_MANIFEST_FILE, encoding="utf-8") if _.strip()),
    )


### Stage G-Batch — Batch Execution for Source-Code Seeds

This is the preferred path for the **full source-code production run** once the prompt contract is frozen. It keeps the same seed-generation logic as Stage G, but moves transport to the Batch API so cost drops without changing the pedagogical design.

The workflow is:

1. prepare the request JSONL
2. submit or resume the batch job
3. download the raw batch output files
4. materialize them into the standard PQID seed artifact that the existing audit cell already understands

To preserve notebook reproducibility, the fixed-purpose create/wait cells below can be used instead of editing the manual submit cell in place.

Visible run order in this notebook section:

1. **Prepare request file**
2. **Optional manual submit cell**
3. **Fixed-path create cell**
4. **Fixed-path wait/download cell**
5. **Materialize standard PQID output**


#### How To Use Stage G-Batch

This section contains **two different paths**:

1. the normal source-code batch production path
2. an optional retry path used only if the audit still shows missing rows after the normal path

Normal path:

1. **Prepare request file**
2. **Create batch job**
3. **Wait for completion and download files**
4. **Materialize the normal PQID output file**
5. **Run the local audit cell directly below**

Only if that audit still shows missing rows, continue to the separate **Stage G-Retry** subsection below.


##### Step G-B1 — Prepare Source-Code Batch Request File

Run this first. It writes the batch request JSONL for the missing or remaining `source_code` seed rows.


In [ ]:
SOURCE_CODE_BATCH_MODEL = "gpt-5.4"
SOURCE_CODE_BATCH_TEMPERATURE = 0.1
SOURCE_CODE_BATCH_MAX_RECORDS = None

cmd = [
    sys.executable,
    str(PREPARE_SEED_BATCH_SCRIPT),
    "--manifest-file", str(SOURCE_CODE_MANIFEST_FILE),
    "--source-file", str(ENRICHED_CORPUS),
    "--request-file", str(SOURCE_CODE_SEED_BATCH_REQUEST_FILE),
    "--existing-output-file", str(PRODUCTION_SEED_DRAFTS),
    "--model", SOURCE_CODE_BATCH_MODEL,
    "--temperature", str(SOURCE_CODE_BATCH_TEMPERATURE),
]

if SOURCE_CODE_BATCH_MAX_RECORDS is not None:
    cmd.extend(["--max-records", str(SOURCE_CODE_BATCH_MAX_RECORDS)])

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("source-code seed batch preparation completed with return code:", result.returncode)

##### Optional Step G-B2a — Manual Submit / Inspect Cell

This is the original flexible control cell. It remains here for advanced use, but the fixed-path cells below are easier to reproduce.


In [ ]:
SOURCE_CODE_BATCH_CREATE = False
SOURCE_CODE_BATCH_WAIT = False
SOURCE_CODE_BATCH_ID = ""

if not SOURCE_CODE_BATCH_CREATE and not SOURCE_CODE_BATCH_ID:
    print("Set SOURCE_CODE_BATCH_CREATE = True to create a new batch, or set SOURCE_CODE_BATCH_ID to inspect/download an existing batch.")
else:
    cmd = [sys.executable, str(RUN_BATCH_JOB_SCRIPT)]
    if SOURCE_CODE_BATCH_CREATE:
        cmd.extend([
            "--request-file", str(SOURCE_CODE_SEED_BATCH_REQUEST_FILE),
            "--state-file", str(SOURCE_CODE_SEED_BATCH_STATE_FILE),
        ])
    else:
        cmd.extend(["--batch-id", SOURCE_CODE_BATCH_ID])

    if SOURCE_CODE_BATCH_WAIT:
        cmd.append("--wait")

    cmd.extend([
        "--download-output-file", str(SOURCE_CODE_SEED_BATCH_OUTPUT_FILE),
        "--download-error-file", str(SOURCE_CODE_SEED_BATCH_ERROR_FILE),
    ])

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("source-code seed batch command completed with return code:", result.returncode)

#### Fixed-Path Source-Code Batch Continuation

These cells avoid in-place parameter flipping. Use them when you want the notebook record to show a stable `create` step and a stable `wait/download` step.


##### Step G-B2 — Create Source-Code Batch Job

Run this after preparation if you want the stable fixed-path workflow. It uploads the request file, creates the batch job, and stores the batch id in the state file.


In [ ]:
if not SOURCE_CODE_SEED_BATCH_REQUEST_FILE.exists():
    print("Prepare the source-code batch request file first.")
else:
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(SOURCE_CODE_SEED_BATCH_REQUEST_FILE),
        "--state-file", str(SOURCE_CODE_SEED_BATCH_STATE_FILE),
        "--download-output-file", str(SOURCE_CODE_SEED_BATCH_OUTPUT_FILE),
        "--download-error-file", str(SOURCE_CODE_SEED_BATCH_ERROR_FILE),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("source-code seed batch creation completed with return code:", result.returncode)
    if SOURCE_CODE_SEED_BATCH_STATE_FILE.exists():
        state = json.loads(SOURCE_CODE_SEED_BATCH_STATE_FILE.read_text(encoding="utf-8"))
        print("source-code batch id:", state.get("batch_id"))
        print("source-code batch status:", state.get("status"))


##### Step G-B3 — Wait For Completion And Download Source-Code Batch Files

Run this after the create step. It reads the saved batch id from the state file, waits for completion, and downloads the batch output and error files.


In [ ]:
if not SOURCE_CODE_SEED_BATCH_STATE_FILE.exists():
    print("No source-code batch state file yet. Run the create cell first.")
else:
    state = json.loads(SOURCE_CODE_SEED_BATCH_STATE_FILE.read_text(encoding="utf-8"))
    batch_id = state.get("batch_id", "")
    if not batch_id:
        print("No batch_id recorded in the source-code batch state file.")
    else:
        cmd = [
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--batch-id", batch_id,
            "--state-file", str(SOURCE_CODE_SEED_BATCH_STATE_FILE),
            "--wait",
            "--download-output-file", str(SOURCE_CODE_SEED_BATCH_OUTPUT_FILE),
            "--download-error-file", str(SOURCE_CODE_SEED_BATCH_ERROR_FILE),
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if result.stdout.strip():
            print(result.stdout.strip())
        if result.stderr.strip():
            print("stderr")
            print(result.stderr.strip())
        print("source-code seed batch wait/download completed with return code:", result.returncode)

##### Step G-B4 — Materialize Standard Source-Code Seed Artifact

Run this after the wait/download step. It converts the downloaded Batch API results back into the standard PQID seed JSONL and error log used by the existing audit cell.


In [ ]:
if not SOURCE_CODE_SEED_BATCH_OUTPUT_FILE.exists():
    print("No downloaded source-code seed batch output yet.")
else:
    cmd = [
        sys.executable,
        str(MATERIALIZE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(SOURCE_CODE_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--batch-output-file", str(SOURCE_CODE_SEED_BATCH_OUTPUT_FILE),
        "--batch-error-file", str(SOURCE_CODE_SEED_BATCH_ERROR_FILE),
        "--output-file", str(PRODUCTION_SEED_DRAFTS),
        "--log-file", str(PRODUCTION_SEED_DRAFT_ERRORS),
        "--model", SOURCE_CODE_BATCH_MODEL,
        "--temperature", str(SOURCE_CODE_BATCH_TEMPERATURE),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("source-code seed batch materialization completed with return code:", result.returncode)

##### Step G-B5 — Audit Source-Code Seed Artifact

This is the **end of the normal Stage G-Batch path**.

If coverage is complete here, stop and move on to Stage H. Only continue into the retry subsection if this audit still shows missing rows.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_SEED_DRAFT_ERRORS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl"
SOURCE_CODE_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl"

if not PRODUCTION_SEED_DRAFTS.exists():
    print("No production seed draft file yet.")
    if PRODUCTION_SEED_DRAFT_ERRORS.exists():
        print("\nRecent error-log preview:\n")
        with open(PRODUCTION_SEED_DRAFT_ERRORS, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                row = json.loads(line)
                print(json.dumps({
                    "error_type": row.get("error_type"),
                    "error_message": row.get("error_message"),
                    "seed_role": row.get("seed_role"),
                }, ensure_ascii=False, indent=2))
                print()
    else:
        print("Run the production source-code seed cell to materialize outputs.")
else:
    role_counts = Counter()
    prompt_type_counts = Counter()
    temperature_counts = Counter()
    unique_hashes = set()
    rows = 0

    with open(PRODUCTION_SEED_DRAFTS, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            meta = row.get("metadata", {})
            role_counts[meta.get("seed_role", "<missing>")] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            temperature_counts[str(meta.get("seed_generation_temperature", "<missing>"))] += 1
            unique_hashes.add(meta.get("circuit_hash", ""))

    print("Production seed counts by role")
    for key, value in role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nPrompt types")
    for key, value in prompt_type_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nSeed draft temperatures present")
    for key, value in temperature_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nProduction seed coverage")
    print("  rows:", f"{rows:,}")
    print("  unique circuit_hash values:", f"{len(unique_hashes):,}")
    print(
        "  source-code manifest size:",
        sum(1 for _ in open(SOURCE_CODE_MANIFEST_FILE, encoding="utf-8") if _.strip()),
    )


#### Stage G-Retry — Recover Missing Or Truncated Source-Code Rows

This subsection is **optional**. Use it only if `Step G-B5` still shows missing rows after the normal Stage G-Batch path.

The current documented failure mode is truncated JSON caused by `max_output_tokens`, so this retry path builds a manifest of only the still-missing rows and reruns them with a higher output cap. The preparation script now also applies truncation-aware dynamic token allocation per request.


##### Step G-R1 — Build Source-Code Retry Manifest From Still-Missing Rows

Run this first if `Step G-B5` still reports `rows < source-code manifest size`. It compares the full source-code manifest against the current materialized output and writes a retry manifest containing only the still-missing keys.


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
SOURCE_CODE_MANIFEST_FILE = globals().get("SOURCE_CODE_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_SEED_DRAFT_ERRORS = globals().get("PRODUCTION_SEED_DRAFT_ERRORS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl")
SOURCE_CODE_RETRY_MANIFEST_FILE = globals().get("SOURCE_CODE_RETRY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code_retry.jsonl")
SOURCE_CODE_RETRY_BATCH_REQUEST_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_REQUEST_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_requests_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_STATE_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_STATE_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_v1.json")
SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_output_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_ERROR_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_ERROR_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_error_v1.jsonl")
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = globals().get("BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT", SCRIPTS_DIR / "build_missing_seed_retry_manifest.py")
PREPARE_SEED_BATCH_SCRIPT = globals().get("PREPARE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "prepare_seed_drafts_quality_aware_batch.py")
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get("MATERIALIZE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "materialize_seed_drafts_quality_aware_batch.py")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", SCRIPTS_DIR / "run_openai_batch_job.py")

cmd = [
    sys.executable,
    str(BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT),
    "--manifest-file", str(SOURCE_CODE_MANIFEST_FILE),
    "--output-file", str(PRODUCTION_SEED_DRAFTS),
    "--retry-manifest-file", str(SOURCE_CODE_RETRY_MANIFEST_FILE),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("source-code retry manifest build completed with return code:", result.returncode)

##### Step G-R2 — Prepare Source-Code Retry Batch Request File

Run this after the retry manifest is built. This retry path uses a higher base output-token cap because the missing row is currently caused by response truncation. The preparation script may raise the actual per-request cap further for large opaque circuits.


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
SOURCE_CODE_MANIFEST_FILE = globals().get("SOURCE_CODE_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_SEED_DRAFT_ERRORS = globals().get("PRODUCTION_SEED_DRAFT_ERRORS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl")
SOURCE_CODE_RETRY_MANIFEST_FILE = globals().get("SOURCE_CODE_RETRY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code_retry.jsonl")
SOURCE_CODE_RETRY_BATCH_REQUEST_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_REQUEST_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_requests_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_STATE_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_STATE_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_v1.json")
SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_output_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_ERROR_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_ERROR_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_error_v1.jsonl")
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = globals().get("BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT", SCRIPTS_DIR / "build_missing_seed_retry_manifest.py")
PREPARE_SEED_BATCH_SCRIPT = globals().get("PREPARE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "prepare_seed_drafts_quality_aware_batch.py")
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get("MATERIALIZE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "materialize_seed_drafts_quality_aware_batch.py")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", SCRIPTS_DIR / "run_openai_batch_job.py")

SOURCE_CODE_RETRY_BATCH_MODEL = "gpt-5.4"
SOURCE_CODE_RETRY_BATCH_TEMPERATURE = 0.1
SOURCE_CODE_RETRY_MAX_OUTPUT_TOKENS = 1200

if not SOURCE_CODE_RETRY_MANIFEST_FILE.exists():
    print("No retry manifest yet. Run Step G-R1 first.")
else:
    cmd = [
        sys.executable,
        str(PREPARE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(SOURCE_CODE_RETRY_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--request-file", str(SOURCE_CODE_RETRY_BATCH_REQUEST_FILE),
        "--model", SOURCE_CODE_RETRY_BATCH_MODEL,
        "--temperature", str(SOURCE_CODE_RETRY_BATCH_TEMPERATURE),
        "--max-output-tokens", str(SOURCE_CODE_RETRY_MAX_OUTPUT_TOKENS),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("source-code retry batch preparation completed with return code:", result.returncode)

##### Step G-R3 — Create Source-Code Retry Batch Job

Run this after preparing the retry request file. It creates a small retry batch containing only the still-missing rows.


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
SOURCE_CODE_MANIFEST_FILE = globals().get("SOURCE_CODE_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_SEED_DRAFT_ERRORS = globals().get("PRODUCTION_SEED_DRAFT_ERRORS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl")
SOURCE_CODE_RETRY_MANIFEST_FILE = globals().get("SOURCE_CODE_RETRY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code_retry.jsonl")
SOURCE_CODE_RETRY_BATCH_REQUEST_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_REQUEST_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_requests_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_STATE_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_STATE_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_v1.json")
SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_output_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_ERROR_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_ERROR_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_error_v1.jsonl")
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = globals().get("BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT", SCRIPTS_DIR / "build_missing_seed_retry_manifest.py")
PREPARE_SEED_BATCH_SCRIPT = globals().get("PREPARE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "prepare_seed_drafts_quality_aware_batch.py")
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get("MATERIALIZE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "materialize_seed_drafts_quality_aware_batch.py")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", SCRIPTS_DIR / "run_openai_batch_job.py")

if not SOURCE_CODE_RETRY_BATCH_REQUEST_FILE.exists():
    print("No source-code retry batch request file yet. Run Step G-B7 first.")
else:
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(SOURCE_CODE_RETRY_BATCH_REQUEST_FILE),
        "--state-file", str(SOURCE_CODE_RETRY_BATCH_STATE_FILE),
        "--download-output-file", str(SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE),
        "--download-error-file", str(SOURCE_CODE_RETRY_BATCH_ERROR_FILE),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("source-code retry batch creation completed with return code:", result.returncode)
    if SOURCE_CODE_RETRY_BATCH_STATE_FILE.exists():
        state = json.loads(SOURCE_CODE_RETRY_BATCH_STATE_FILE.read_text(encoding="utf-8"))
        print("source-code retry batch id:", state.get("batch_id"))
        print("source-code retry batch status:", state.get("status"))


##### Step G-R4 — Wait For Completion And Download Source-Code Retry Batch Files

Run this after the retry create step. It waits on the saved retry batch id and downloads the retry output and error files.


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
SOURCE_CODE_MANIFEST_FILE = globals().get("SOURCE_CODE_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_SEED_DRAFT_ERRORS = globals().get("PRODUCTION_SEED_DRAFT_ERRORS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl")
SOURCE_CODE_RETRY_MANIFEST_FILE = globals().get("SOURCE_CODE_RETRY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code_retry.jsonl")
SOURCE_CODE_RETRY_BATCH_REQUEST_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_REQUEST_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_requests_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_STATE_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_STATE_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_v1.json")
SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_output_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_ERROR_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_ERROR_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_error_v1.jsonl")
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = globals().get("BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT", SCRIPTS_DIR / "build_missing_seed_retry_manifest.py")
PREPARE_SEED_BATCH_SCRIPT = globals().get("PREPARE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "prepare_seed_drafts_quality_aware_batch.py")
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get("MATERIALIZE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "materialize_seed_drafts_quality_aware_batch.py")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", SCRIPTS_DIR / "run_openai_batch_job.py")

if not SOURCE_CODE_RETRY_BATCH_STATE_FILE.exists():
    print("No source-code retry batch state file yet. Run Step G-B8 first.")
else:
    state = json.loads(SOURCE_CODE_RETRY_BATCH_STATE_FILE.read_text(encoding="utf-8"))
    batch_id = state.get("batch_id", "")
    if not batch_id:
        print("No retry batch id recorded in the source-code retry state file.")
    else:
        cmd = [
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--batch-id", batch_id,
            "--state-file", str(SOURCE_CODE_RETRY_BATCH_STATE_FILE),
            "--wait",
            "--download-output-file", str(SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE),
            "--download-error-file", str(SOURCE_CODE_RETRY_BATCH_ERROR_FILE),
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if result.stdout.strip():
            print(result.stdout.strip())
        if result.stderr.strip():
            print("stderr")
            print(result.stderr.strip())
        print("source-code retry batch wait/download completed with return code:", result.returncode)

##### Step G-R5 — Materialize Source-Code Retry Results

Run this after the retry wait/download step. It appends the retry results into the same standard PQID source-code seed artifact. Then run the local retry audit directly below to confirm that Stage G is complete.


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
SOURCE_CODE_MANIFEST_FILE = globals().get("SOURCE_CODE_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_SEED_DRAFT_ERRORS = globals().get("PRODUCTION_SEED_DRAFT_ERRORS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl")
SOURCE_CODE_RETRY_MANIFEST_FILE = globals().get("SOURCE_CODE_RETRY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_source_code_retry.jsonl")
SOURCE_CODE_RETRY_BATCH_REQUEST_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_REQUEST_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_requests_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_STATE_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_STATE_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_v1.json")
SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_output_v1.jsonl")
SOURCE_CODE_RETRY_BATCH_ERROR_FILE = globals().get("SOURCE_CODE_RETRY_BATCH_ERROR_FILE", PROCESSED_DIR / "openai_batch_jobs/source_code_seed_retry_batch_error_v1.jsonl")
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = globals().get("BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT", SCRIPTS_DIR / "build_missing_seed_retry_manifest.py")
PREPARE_SEED_BATCH_SCRIPT = globals().get("PREPARE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "prepare_seed_drafts_quality_aware_batch.py")
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get("MATERIALIZE_SEED_BATCH_SCRIPT", SCRIPTS_DIR / "materialize_seed_drafts_quality_aware_batch.py")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", SCRIPTS_DIR / "run_openai_batch_job.py")

if not SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE.exists():
    print("No downloaded source-code retry batch output yet.")
else:
    cmd = [
        sys.executable,
        str(MATERIALIZE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(SOURCE_CODE_RETRY_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--batch-output-file", str(SOURCE_CODE_RETRY_BATCH_OUTPUT_FILE),
        "--batch-error-file", str(SOURCE_CODE_RETRY_BATCH_ERROR_FILE),
        "--output-file", str(PRODUCTION_SEED_DRAFTS),
        "--log-file", str(PRODUCTION_SEED_DRAFT_ERRORS),
        "--model", SOURCE_CODE_RETRY_BATCH_MODEL,
        "--temperature", str(SOURCE_CODE_RETRY_BATCH_TEMPERATURE),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("source-code retry batch materialization completed with return code:", result.returncode)

##### Step G-R6 — Re-Audit Source-Code Seed Artifact

Run this immediately after `Step G-R5`. This closes the optional retry path inside the same local subsection.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_SEED_DRAFT_ERRORS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_errors.jsonl"
SOURCE_CODE_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl"

if not PRODUCTION_SEED_DRAFTS.exists():
    print("No production seed draft file yet.")
    if PRODUCTION_SEED_DRAFT_ERRORS.exists():
        print("\nRecent error-log preview:\n")
        with open(PRODUCTION_SEED_DRAFT_ERRORS, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                row = json.loads(line)
                print(json.dumps({
                    "error_type": row.get("error_type"),
                    "error_message": row.get("error_message"),
                    "seed_role": row.get("seed_role"),
                }, ensure_ascii=False, indent=2))
                print()
    else:
        print("Run the production source-code seed cell to materialize outputs.")
else:
    role_counts = Counter()
    prompt_type_counts = Counter()
    temperature_counts = Counter()
    unique_hashes = set()
    rows = 0

    with open(PRODUCTION_SEED_DRAFTS, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            meta = row.get("metadata", {})
            role_counts[meta.get("seed_role", "<missing>")] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            temperature_counts[str(meta.get("seed_generation_temperature", "<missing>"))] += 1
            unique_hashes.add(meta.get("circuit_hash", ""))

    print("Production seed counts by role")
    for key, value in role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nPrompt types")
    for key, value in prompt_type_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nSeed draft temperatures present")
    for key, value in temperature_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nProduction seed coverage")
    print("  rows:", f"{rows:,}")
    print("  unique circuit_hash values:", f"{len(unique_hashes):,}")
    print(
        "  source-code manifest size:",
        sum(1 for _ in open(SOURCE_CODE_MANIFEST_FILE, encoding="utf-8") if _.strip()),
    )


#### Stage G-Post — Harmonize Prompt-Type Labels After Schema Rename

Run this subsection only if the Stage G audit shows a mixed prompt-type count such as `human_seed_quality_aware` plus `base_seed_quality_aware`. That situation means the artifact spans the rename boundary from the deprecated label to the canonical one.

This is a **release-cleanup** step, not a semantic rewrite. It only harmonizes lineage labels so the final artifact uses the documented canonical prompt type.


##### Step G-P1 — Normalize Prompt-Type Labels In The Source-Code Seed Artifact

This cell rewrites the deprecated alias `human_seed_quality_aware` to the canonical `base_seed_quality_aware` and keeps a backup of the pre-normalization file for auditability.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
SOURCE_CODE_PROMPT_TYPE_BACKUP_FILE = globals().get("SOURCE_CODE_PROMPT_TYPE_BACKUP_FILE", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1_pre_prompt_type_normalization.jsonl")
NORMALIZE_QUALITY_AWARE_PROMPT_TYPES_SCRIPT = globals().get("NORMALIZE_QUALITY_AWARE_PROMPT_TYPES_SCRIPT", SCRIPTS_DIR / "normalize_quality_aware_prompt_types.py")

cmd = [
    sys.executable,
    str(NORMALIZE_QUALITY_AWARE_PROMPT_TYPES_SCRIPT),
    "--input-file", str(PRODUCTION_SEED_DRAFTS),
    "--backup-file", str(SOURCE_CODE_PROMPT_TYPE_BACKUP_FILE),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("source-code prompt-type normalization completed with return code:", result.returncode)

##### Step G-P2 — Re-Audit Source-Code Seed Artifact After Prompt-Type Normalization

Run this immediately after `Step G-P1`. The expected outcome is that the prompt-type distribution collapses to a single canonical rebuild label.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
SOURCE_CODE_MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1_source_code.jsonl"

role_counts = Counter()
prompt_type_counts = Counter()
temperature_counts = Counter()
unique_hashes = set()
rows = 0

with open(PRODUCTION_SEED_DRAFTS, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        rows += 1
        meta = row.get("metadata", {})
        role_counts[meta.get("seed_role", "<missing>")] += 1
        prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
        temperature_counts[str(meta.get("seed_generation_temperature", "<missing>"))] += 1
        unique_hashes.add(meta.get("circuit_hash", ""))

print("Production seed counts by role")
for key, value in role_counts.most_common():
    print(f"  {key}: {value:,}")

print("\nPrompt types")
for key, value in prompt_type_counts.most_common():
    print(f"  {key}: {value:,}")

print("\nSeed draft temperatures present")
for key, value in temperature_counts.most_common():
    print(f"  {key}: {value:,}")

print("\nProduction seed coverage")
print("  rows:", f"{rows:,}")
print("  unique circuit_hash values:", f"{len(unique_hashes):,}")
print(
    "  source-code manifest size:",
    sum(1 for _ in open(SOURCE_CODE_MANIFEST_FILE, encoding="utf-8") if _.strip()),
)


## Stage H-Cal-A — Validation-Diagnosis Model Calibration Gate

This stage is a **cost-quality calibration gate** for the majority-corpus `validation_diagnosis` branch inside `teacher_text`. Run it **before** Stage H production so the dominant teacher-text model choice is evidence-based rather than implicit.

The output of this stage should determine whether the `validation_diagnosis` portion of Stage H stays on `gpt-5.4` or switches to a cheaper model such as `gpt-5.4-mini`.


### Stage H-Cal-A1 — Build Validation-Diagnosis Model-Comparison Manifest

This creates a deterministic matched `validation_diagnosis` study manifest for comparing teacher-text candidate models on the same source records.


In [ ]:
from collections import Counter
import json
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE = globals().get("TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_model_study.jsonl")

TEACHER_TEXT_MODEL_STUDY_ROLE = "validation_diagnosis"
TEACHER_TEXT_MODEL_STUDY_SIZE = 60

rows = []
with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            if row.get("seed_role") == TEACHER_TEXT_MODEL_STUDY_ROLE:
                rows.append(row)

selected = rows[:TEACHER_TEXT_MODEL_STUDY_SIZE]
with open(TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE, "w", encoding="utf-8") as f:
    for row in selected:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

role_counts = Counter(row["seed_role"] for row in selected)
print("validation-diagnosis model study manifest written:", TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE)
print("study rows:", len(selected))
print("role distribution")
for key, value in role_counts.most_common():
    print(f"  {key}: {value:,}")


### Stage H-Cal-A2 — Run Matched Validation-Diagnosis Model Comparison

Run this before any full Stage H production execution. It evaluates the same matched `validation_diagnosis` study manifest under two candidate models:

- `gpt-5.4`
- `gpt-5.4-mini`

The goal is to choose the dominant diagnosis-branch production model on evidence, not budget intuition alone.


In [ ]:
import subprocess
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE = globals().get("TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_model_study.jsonl")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")
GENERATE_DRAFTS_SCRIPT = globals().get("GENERATE_DRAFTS_SCRIPT", SCRIPTS_DIR / "generate_seed_drafts_quality_aware.py")

TEACHER_TEXT_MODEL_GRID = ["gpt-5.4", "gpt-5.4-mini"]
TEACHER_TEXT_MODEL_COMPARISON_TEMPERATURE = 0.1
TEACHER_TEXT_MODEL_COMPARISON_DRY_RUN = False

TEACHER_TEXT_MODEL_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

for model_name in TEACHER_TEXT_MODEL_GRID:
    safe_name = model_name.replace('.', 'p')
    output_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_model_compare_{safe_name}.jsonl"
    error_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_model_compare_{safe_name}_errors.jsonl"
    print(f"\nrunning teacher-text model comparison for {model_name}")
    cmd = [
        sys.executable,
        str(GENERATE_DRAFTS_SCRIPT),
        "--manifest-file", str(TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--output-file", str(output_file),
        "--log-file", str(error_file),
        "--model", model_name,
        "--temperature", str(TEACHER_TEXT_MODEL_COMPARISON_TEMPERATURE),
    ]
    if TEACHER_TEXT_MODEL_COMPARISON_DRY_RUN:
        cmd.append("--dry-run")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("return code:", result.returncode)

### Stage H-Cal-A3 — Audit Validation-Diagnosis Model Comparison

This is a descriptive comparison for the `validation_diagnosis` study outputs. Use it for quick qualitative inspection before the more formal statistical evaluation step.


In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())

rows_by_model = {}
for model_name in ["gpt-5.4", "gpt-5.4-mini"]:
    safe_name = model_name.replace('.', 'p')
    output_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_model_compare_{safe_name}.jsonl"
    error_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_model_compare_{safe_name}_errors.jsonl"
    if not output_file.exists():
        print(f"missing output for {model_name}: {output_file}")
        if error_file.exists():
            print("recent errors:")
            with open(error_file, encoding="utf-8") as f:
                for i, line in enumerate(f):
                    if i >= 3:
                        break
                    print(line.strip()[:1500])
        continue
    rows = []
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    rows_by_model[model_name] = rows

for model_name, rows in rows_by_model.items():
    opener_counts = Counter()
    role_counts = Counter()
    normalized_counts = Counter()
    avg_words = 0.0
    avg_out_words = 0.0
    for row in rows:
        text = row.get("input", "").strip()
        out = row.get("output", "").strip()
        meta = row.get("metadata", {})
        opener = " ".join(text.split()[:2]).lower() if text else "<empty>"
        opener_counts[opener] += 1
        role_counts[meta.get("seed_role", "<missing>")] += 1
        normalized_counts[normalize_text(text)] += 1
        avg_words += len(text.split())
        avg_out_words += len(out.split())
    if rows:
        avg_words /= len(rows)
        avg_out_words /= len(rows)
    print(f"\nModel {model_name}")
    print("  rows:", len(rows))
    print("  avg input words:", round(avg_words, 1))
    print("  avg output words:", round(avg_out_words, 1))
    print("  exact normalized duplicates:", sum(1 for v in normalized_counts.values() if v > 1))
    repeated_openers = {k:v for k,v in opener_counts.items() if v > 1}
    if repeated_openers:
        print("  repeated opening patterns:")
        for key, value in repeated_openers.items():
            print(f"    {key}: {value}")
    print("  role distribution:")
    for key, value in role_counts.most_common():
        print(f"    {key}: {value}")

common = None
indexed = {}
for model_name, rows in rows_by_model.items():
    indexed[model_name] = {}
    for row in rows:
        meta = row.get("metadata", {})
        key = (meta.get("circuit_hash"), meta.get("seed_role"))
        indexed[model_name][key] = row
    keys = set(indexed[model_name].keys())
    common = keys if common is None else (common & keys)

if common:
    print("\nMatched teacher-text sample comparison")
    for idx, key in enumerate(sorted(common)[:8], start=1):
        circuit_hash, role = key
        print(f"\nSample {idx} | role={role} | circuit_hash={circuit_hash}")
        for model_name in ["gpt-5.4", "gpt-5.4-mini"]:
            row = indexed[model_name][key]
            print(f"  model={model_name}")
            print(f"    input: {row.get('input', '')}")
            print(f"    output: {row.get('output', '')[:500]}")


### Stage H-Cal-A4 — Statistical Evaluation Of Validation-Diagnosis Model Comparison

Run this after `Stage H-Cal-A3`. It writes a structured evaluation report and prints matched automatic metrics with sign-test p-values and bootstrap confidence intervals, so the model choice can be defended with quantitative evidence rather than only heuristic reading.


In [ ]:
import json

if "render_teacher_text_model_eval_report" not in globals():
    def render_teacher_text_model_eval_report(report: dict) -> None:
        def fmt(value, digits: int = 4) -> str:
            if value is None:
                return "n/a"
            if isinstance(value, (int, float)):
                return f"{value:.{digits}f}"
            return str(value)

        def print_table(headers, rows) -> None:
            widths = [len(str(header)) for header in headers]
            for row in rows:
                for idx, cell in enumerate(row):
                    widths[idx] = max(widths[idx], len(str(cell)))
            header_line = " | ".join(str(header).ljust(widths[idx]) for idx, header in enumerate(headers))
            divider = "-+-".join("-" * widths[idx] for idx in range(len(headers)))
            print(header_line)
            print(divider)
            for row in rows:
                print(" | ".join(str(cell).ljust(widths[idx]) for idx, cell in enumerate(row)))

        print("teacher-text model calibration report")
        print("  study label:", report.get("study_label", ""))
        if report.get("role"):
            print("  role:", report.get("role"))
        print("  expected rows:", report.get("expected_rows"))
        print()

        summary_headers = [
            "model",
            "rows",
            "complete",
            "overall",
            "strict_pass",
            "specificity",
            "caution",
            "actionability",
            "overclaim_clean",
            "max_opener_share",
        ]
        summary_rows = []
        for model_name in report.get("models", []):
            summary = report.get("summaries", {}).get(model_name, {})
            repeated = summary.get("repeated_openers", {}) or {}
            rows = summary.get("rows") or 0
            max_opener_share = None
            if rows:
                max_count = max(repeated.values()) if repeated else 1
                max_opener_share = max_count / rows
            summary_rows.append([
                model_name,
                rows,
                fmt(summary.get("completion_rate")),
                fmt(summary.get("overall_score_mean")),
                fmt(summary.get("strict_pass_rate")),
                fmt(summary.get("source_specificity_score_mean")),
                fmt(summary.get("caution_score_mean")),
                fmt(summary.get("actionability_score_mean")),
                fmt(summary.get("overclaim_clean_rate")),
                fmt(max_opener_share),
            ])
        print("Model summary table")
        print_table(summary_headers, summary_rows)
        print()

        for pair in report.get("pairwise", []):
            left, right = pair.get("models", ["model_a", "model_b"])
            print(f"Pairwise matched comparison: {left} vs {right}")
            metric_headers = [
                "metric",
                "wins",
                "losses",
                "ties",
                "mean_diff",
                "p_value",
                "ci_low",
                "ci_high",
                "matched_rows",
            ]
            metric_rows = []
            for metric_name in [
                "overall_score",
                "strict_pass",
                "source_specificity_score",
                "caution_score",
                "actionability_score",
            ]:
                stats = pair.get("metrics", {}).get(metric_name, {})
                metric_rows.append([
                    metric_name,
                    stats.get("wins", "n/a"),
                    stats.get("losses", "n/a"),
                    stats.get("ties", "n/a"),
                    fmt(stats.get("mean_diff")),
                    fmt(stats.get("p_value_sign_test")),
                    fmt(stats.get("bootstrap_ci_low")),
                    fmt(stats.get("bootstrap_ci_high")),
                    stats.get("matched_rows", "n/a"),
                ])
            print_table(metric_headers, metric_rows)
            print()

    globals()["render_teacher_text_model_eval_report"] = render_teacher_text_model_eval_report

SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE = globals().get("TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_model_study.jsonl")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")
EVALUATE_TEACHER_TEXT_MODEL_CALIBRATION_SCRIPT = globals().get("EVALUATE_TEACHER_TEXT_MODEL_CALIBRATION_SCRIPT", SCRIPTS_DIR / "evaluate_teacher_text_model_calibration.py")
VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE = globals().get("VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE", TEACHER_TEXT_MODEL_COMPARISON_DIR / "teacher_text_model_calibration_validation_eval.json")

cmd = [
    sys.executable,
    str(EVALUATE_TEACHER_TEXT_MODEL_CALIBRATION_SCRIPT),
    "--source-file", str(ENRICHED_CORPUS),
    "--comparison-dir", str(TEACHER_TEXT_MODEL_COMPARISON_DIR),
    "--file-prefix", "teacher_text_model_compare",
    "--models", "gpt-5.4", "gpt-5.4-mini",
    "--manifest-file", str(TEACHER_TEXT_MODEL_STUDY_MANIFEST_FILE),
    "--study-label", "validation_diagnosis_model_gate",
    "--role", "validation_diagnosis",
    "--output-file", str(VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("validation-diagnosis statistical evaluation completed with return code:", result.returncode)
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
report = json.loads(VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE.read_text(encoding="utf-8"))
render_teacher_text_model_eval_report(report)


## Stage H-Cal-B — Mutation-Robustness Model Calibration Gate

This stage separately calibrates the `mutation_robustness` branch. Because this role is conceptually sharper and smaller than `validation_diagnosis`, it should not inherit the diagnosis-branch model choice by default.


### Stage H-Cal-B1 — Build Mutation-Robustness Model-Comparison Manifest

This creates a deterministic matched `mutation_robustness` study manifest for comparing teacher-text candidate models on the same mutation-stress records.


In [ ]:
from collections import Counter
import json
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
MUTATION_MODEL_STUDY_MANIFEST_FILE = globals().get("MUTATION_MODEL_STUDY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_model_study.jsonl")

MUTATION_MODEL_STUDY_ROLE = "mutation_robustness"
MUTATION_MODEL_STUDY_SIZE = 60

rows = []
with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            if row.get("seed_role") == MUTATION_MODEL_STUDY_ROLE:
                rows.append(row)

selected = rows[:MUTATION_MODEL_STUDY_SIZE]
with open(MUTATION_MODEL_STUDY_MANIFEST_FILE, "w", encoding="utf-8") as f:
    for row in selected:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

role_counts = Counter(row["seed_role"] for row in selected)
print("mutation-robustness model study manifest written:", MUTATION_MODEL_STUDY_MANIFEST_FILE)
print("study rows:", len(selected))
print("role distribution")
for key, value in role_counts.most_common():
    print(f"  {key}: {value:,}")


### Stage H-Cal-B2 — Run Matched Mutation-Robustness Model Comparison

Run this before any full Stage H production execution. It evaluates the same matched `mutation_robustness` study manifest under two candidate models:

- `gpt-5.4`
- `gpt-5.4-mini`


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
MUTATION_MODEL_STUDY_MANIFEST_FILE = globals().get("MUTATION_MODEL_STUDY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_model_study.jsonl")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")
GENERATE_DRAFTS_SCRIPT = globals().get("GENERATE_DRAFTS_SCRIPT", SCRIPTS_DIR / "generate_seed_drafts_quality_aware.py")

MUTATION_MODEL_GRID = ["gpt-5.4", "gpt-5.4-mini"]
MUTATION_MODEL_COMPARISON_TEMPERATURE = 0.1
MUTATION_MODEL_COMPARISON_DRY_RUN = False

TEACHER_TEXT_MODEL_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

for model_name in MUTATION_MODEL_GRID:
    safe_name = model_name.replace(".", "p")
    output_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_mutation_model_compare_{safe_name}.jsonl"
    error_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_mutation_model_compare_{safe_name}_errors.jsonl"
    print(f"\nrunning mutation-robustness model comparison for {model_name}")
    cmd = [
        sys.executable,
        str(GENERATE_DRAFTS_SCRIPT),
        "--manifest-file", str(MUTATION_MODEL_STUDY_MANIFEST_FILE),
        "--source-file", str(ENRICHED_CORPUS),
        "--output-file", str(output_file),
        "--log-file", str(error_file),
        "--model", model_name,
        "--temperature", str(MUTATION_MODEL_COMPARISON_TEMPERATURE),
    ]
    if MUTATION_MODEL_COMPARISON_DRY_RUN:
        cmd.append("--dry-run")
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("return code:", result.returncode)

### Stage H-Cal-B3 — Audit Mutation-Robustness Model Comparison

This is a descriptive comparison for the `mutation_robustness` study outputs. Use it for quick qualitative inspection before the more formal statistical evaluation step.


In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())

rows_by_model = {}
for model_name in ["gpt-5.4", "gpt-5.4-mini"]:
    safe_name = model_name.replace(".", "p")
    output_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_mutation_model_compare_{safe_name}.jsonl"
    error_file = TEACHER_TEXT_MODEL_COMPARISON_DIR / f"teacher_text_mutation_model_compare_{safe_name}_errors.jsonl"
    if not output_file.exists():
        print(f"missing output for {model_name}: {output_file}")
        if error_file.exists():
            print("recent errors:")
            with open(error_file, encoding="utf-8") as f:
                for i, line in enumerate(f):
                    if i >= 3:
                        break
                    print(line.strip()[:1500])
        continue
    rows = []
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    rows_by_model[model_name] = rows

for model_name, rows in rows_by_model.items():
    opener_counts = Counter()
    role_counts = Counter()
    normalized_counts = Counter()
    avg_words = 0.0
    avg_out_words = 0.0
    for row in rows:
        text = row.get("input", "").strip()
        out = row.get("output", "").strip()
        meta = row.get("metadata", {})
        opener = " ".join(text.split()[:2]).lower() if text else "<empty>"
        opener_counts[opener] += 1
        role_counts[meta.get("seed_role", "<missing>")] += 1
        normalized_counts[normalize_text(text)] += 1
        avg_words += len(text.split())
        avg_out_words += len(out.split())
    if rows:
        avg_words /= len(rows)
        avg_out_words /= len(rows)
    print(f"\nModel {model_name}")
    print("  rows:", len(rows))
    print("  avg input words:", round(avg_words, 1))
    print("  avg output words:", round(avg_out_words, 1))
    print("  exact normalized duplicates:", sum(1 for v in normalized_counts.values() if v > 1))
    repeated_openers = {k:v for k,v in opener_counts.items() if v > 1}
    if repeated_openers:
        print("  repeated opening patterns:")
        for key, value in repeated_openers.items():
            print(f"    {key}: {value}")
    print("  role distribution:")
    for key, value in role_counts.most_common():
        print(f"    {key}: {value}")

common = None
indexed = {}
for model_name, rows in rows_by_model.items():
    indexed[model_name] = {}
    for row in rows:
        meta = row.get("metadata", {})
        key = (meta.get("circuit_hash"), meta.get("seed_role"))
        indexed[model_name][key] = row
    keys = set(indexed[model_name].keys())
    common = keys if common is None else (common & keys)

if common:
    print("\nMatched mutation-robustness sample comparison")
    for idx, key in enumerate(sorted(common)[:8], start=1):
        circuit_hash, role = key
        print(f"\nSample {idx} | role={role} | circuit_hash={circuit_hash}")
        for model_name in ["gpt-5.4", "gpt-5.4-mini"]:
            row = indexed[model_name][key]
            print(f"  model={model_name}")
            print(f"    input: {row.get('input', '')}")
            print(f"    output: {row.get('output', '')[:500]}")


### Stage H-Cal-B4 — Statistical Evaluation Of Mutation-Robustness Model Comparison

Run this after `Stage H-Cal-B3`. It writes a structured evaluation report and prints matched automatic metrics with sign-test p-values and bootstrap confidence intervals for the mutation-robustness branch.


In [ ]:
import json

if "render_teacher_text_model_eval_report" not in globals():
    def render_teacher_text_model_eval_report(report: dict) -> None:
        def fmt(value, digits: int = 4) -> str:
            if value is None:
                return "n/a"
            if isinstance(value, (int, float)):
                return f"{value:.{digits}f}"
            return str(value)

        def print_table(headers, rows) -> None:
            widths = [len(str(header)) for header in headers]
            for row in rows:
                for idx, cell in enumerate(row):
                    widths[idx] = max(widths[idx], len(str(cell)))
            header_line = " | ".join(str(header).ljust(widths[idx]) for idx, header in enumerate(headers))
            divider = "-+-".join("-" * widths[idx] for idx in range(len(headers)))
            print(header_line)
            print(divider)
            for row in rows:
                print(" | ".join(str(cell).ljust(widths[idx]) for idx, cell in enumerate(row)))

        print("teacher-text model calibration report")
        print("  study label:", report.get("study_label", ""))
        if report.get("role"):
            print("  role:", report.get("role"))
        print("  expected rows:", report.get("expected_rows"))
        print()

        summary_headers = [
            "model",
            "rows",
            "complete",
            "overall",
            "strict_pass",
            "specificity",
            "caution",
            "actionability",
            "overclaim_clean",
            "max_opener_share",
        ]
        summary_rows = []
        for model_name in report.get("models", []):
            summary = report.get("summaries", {}).get(model_name, {})
            repeated = summary.get("repeated_openers", {}) or {}
            rows = summary.get("rows") or 0
            max_opener_share = None
            if rows:
                max_count = max(repeated.values()) if repeated else 1
                max_opener_share = max_count / rows
            summary_rows.append([
                model_name,
                rows,
                fmt(summary.get("completion_rate")),
                fmt(summary.get("overall_score_mean")),
                fmt(summary.get("strict_pass_rate")),
                fmt(summary.get("source_specificity_score_mean")),
                fmt(summary.get("caution_score_mean")),
                fmt(summary.get("actionability_score_mean")),
                fmt(summary.get("overclaim_clean_rate")),
                fmt(max_opener_share),
            ])
        print("Model summary table")
        print_table(summary_headers, summary_rows)
        print()

        for pair in report.get("pairwise", []):
            left, right = pair.get("models", ["model_a", "model_b"])
            print(f"Pairwise matched comparison: {left} vs {right}")
            metric_headers = [
                "metric",
                "wins",
                "losses",
                "ties",
                "mean_diff",
                "p_value",
                "ci_low",
                "ci_high",
                "matched_rows",
            ]
            metric_rows = []
            for metric_name in [
                "overall_score",
                "strict_pass",
                "source_specificity_score",
                "caution_score",
                "actionability_score",
            ]:
                stats = pair.get("metrics", {}).get(metric_name, {})
                metric_rows.append([
                    metric_name,
                    stats.get("wins", "n/a"),
                    stats.get("losses", "n/a"),
                    stats.get("ties", "n/a"),
                    fmt(stats.get("mean_diff")),
                    fmt(stats.get("p_value_sign_test")),
                    fmt(stats.get("bootstrap_ci_low")),
                    fmt(stats.get("bootstrap_ci_high")),
                    stats.get("matched_rows", "n/a"),
                ])
            print_table(metric_headers, metric_rows)
            print()

    globals()["render_teacher_text_model_eval_report"] = render_teacher_text_model_eval_report

SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
MUTATION_MODEL_STUDY_MANIFEST_FILE = globals().get("MUTATION_MODEL_STUDY_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_model_study.jsonl")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")
EVALUATE_TEACHER_TEXT_MODEL_CALIBRATION_SCRIPT = globals().get("EVALUATE_TEACHER_TEXT_MODEL_CALIBRATION_SCRIPT", SCRIPTS_DIR / "evaluate_teacher_text_model_calibration.py")
MUTATION_MODEL_EVAL_FILE = globals().get("MUTATION_MODEL_EVAL_FILE", TEACHER_TEXT_MODEL_COMPARISON_DIR / "teacher_text_model_calibration_mutation_eval.json")

cmd = [
    sys.executable,
    str(EVALUATE_TEACHER_TEXT_MODEL_CALIBRATION_SCRIPT),
    "--source-file", str(ENRICHED_CORPUS),
    "--comparison-dir", str(TEACHER_TEXT_MODEL_COMPARISON_DIR),
    "--file-prefix", "teacher_text_mutation_model_compare",
    "--models", "gpt-5.4", "gpt-5.4-mini",
    "--manifest-file", str(MUTATION_MODEL_STUDY_MANIFEST_FILE),
    "--study-label", "mutation_robustness_model_gate",
    "--role", "mutation_robustness",
    "--output-file", str(MUTATION_MODEL_EVAL_FILE),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("mutation-robustness statistical evaluation completed with return code:", result.returncode)
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
report = json.loads(MUTATION_MODEL_EVAL_FILE.read_text(encoding="utf-8"))
render_teacher_text_model_eval_report(report)


## Stage H-Cal-C — Export Teacher-Text Calibration Tables

Run this after `Stage H-Cal-A4` and `Stage H-Cal-B4`. It exports combined CSV and Markdown tables for the validation-diagnosis and mutation-robustness calibration studies, so the model-selection evidence is easy to reuse in the paper and release documentation.


In [ ]:
from pathlib import Path
import subprocess

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_MODEL_COMPARISON_DIR = globals().get("TEACHER_TEXT_MODEL_COMPARISON_DIR", PROCESSED_DIR / "teacher_text_model_comparison")
VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE = globals().get("VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE", TEACHER_TEXT_MODEL_COMPARISON_DIR / "teacher_text_model_calibration_validation_eval.json")
MUTATION_MODEL_EVAL_FILE = globals().get("MUTATION_MODEL_EVAL_FILE", TEACHER_TEXT_MODEL_COMPARISON_DIR / "teacher_text_model_calibration_mutation_eval.json")
EXPORT_TEACHER_TEXT_MODEL_CALIBRATION_TABLES_SCRIPT = globals().get("EXPORT_TEACHER_TEXT_MODEL_CALIBRATION_TABLES_SCRIPT", SCRIPTS_DIR / "export_teacher_text_model_calibration_tables.py")
TEACHER_TEXT_MODEL_TABLE_PREFIX = globals().get("TEACHER_TEXT_MODEL_TABLE_PREFIX", TEACHER_TEXT_MODEL_COMPARISON_DIR / "teacher_text_model_calibration")

cmd = [
    sys.executable,
    str(EXPORT_TEACHER_TEXT_MODEL_CALIBRATION_TABLES_SCRIPT),
    "--report-files",
    str(VALIDATION_DIAGNOSIS_MODEL_EVAL_FILE),
    str(MUTATION_MODEL_EVAL_FILE),
    "--output-prefix",
    str(TEACHER_TEXT_MODEL_TABLE_PREFIX),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())

summary_md = TEACHER_TEXT_MODEL_TABLE_PREFIX.with_name(TEACHER_TEXT_MODEL_TABLE_PREFIX.name + "_summary_table.md")
pairwise_md = TEACHER_TEXT_MODEL_TABLE_PREFIX.with_name(TEACHER_TEXT_MODEL_TABLE_PREFIX.name + "_pairwise_table.md")

print()
print("Summary table preview")
print(summary_md.read_text(encoding="utf-8").strip())
print()
print("Pairwise table preview")
print(pairwise_md.read_text(encoding="utf-8").strip())


## Stage H Guide — How To Read This Section

This part of the notebook is operationally important and expensive, so it should be read as a **decision tree**, not as a flat list of cells. The labels are intentionally structured, but the structure was not explained clearly enough before. This guide makes the numbering explicit so the notebook remains interpretable even after time has passed.

### Meaning Of The Labels

- `H-Cal-*`: calibration cells used to choose the teacher-text production model policy before the main run. These are evidence-gathering cells, not production cells.
- `H-Preflight`: integrity checks run before spending heavily. This is optional but recommended when the cost/risk is high.
- `H-Policy`: the policy-freeze step. This is where the notebook commits to the role-specific production mapping.
- `H`: the old synchronous production path. It is now deprecated and blocked for production use.
- `H-B*`: the **normal production batch path** for Stage H. This is the path to use under ordinary circumstances.
- `H-BR*`: the **batch recovery path**. Use this only if the normal batch path fails in a documented way, such as an oversized request file.

### Production Decision Tree

Normal production path:
1. Run `Stage H-Preflight` if you want a high-confidence pre-spend check.
2. Run `Stage H-Policy`.
3. Run `Step H-B1`.
4. Run `Step H-B2`.
5. Run `Step H-B3`.
6. Run `Step H-B4`.
7. Run `Step H-B5`.

Recovery path:
- Only enter `Stage H-BR` if the notebook explicitly tells you that the normal validation batch failed before accepting requests, typically because the request JSONL exceeded the Batch API size limit.
- If that happens, keep the successful `mutation_robustness` branch, run `H-BR1` through `H-BR4`, and then return to the normal path at `H-B4` and `H-B5`.

### Which Cells Are Safe To Ignore

- `Stage H` should be ignored for normal production. It is only a blocked legacy/debug path.
- `Optional Step H-B2a` is only an inspection helper. It does **not** create or submit jobs. It only prints the saved state if batch jobs already exist.

### Practical Interpretation Rule

If you want the teacher-text production run and do not have an abnormal failure in front of you, stay entirely inside:
- `Stage H-Policy`
- `Stage H-Batch`

Only enter `Stage H-BR` when the notebook explicitly tells you the normal validation batch failed at the validation/submission boundary.


## Stage H-Preflight — Teacher-Text Production Integrity Gate

This is the **last non-generative checkpoint** before the expensive teacher-text production run. It is additive: it does not write seeds, does not mutate the corpus, and does not change the policy. Its role is to answer one question before money is spent: *is the Stage H production setup internally coherent enough to trust?*

What this stage checks:
- exact coverage of the role-specific manifests against the parent `teacher_text` manifest
- overlap or duplicate `circuit_hash` values across the two role-specific manifests
- source-corpus alignment against the enriched corpus
- consistency between the frozen Stage H production policy and the calibration reports
- non-blocking metadata drift such as stale `generation_defaults.teacher_model` values inside manifests

How to interpret the result:
- if this stage reports no blocking issues, proceed to `Stage H-Policy`
- if it reports blocking issues, stop and fix them before any production generation
- if it reports only warnings, note them in the documentation but do not necessarily treat them as a reason to stop

Recommended use:
- run it whenever the teacher-text run is expensive enough that you want a documented pre-spend sanity gate
- rerun it if the manifests or production policy change materially


In [ ]:
from pathlib import Path
import subprocess

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
PREFLIGHT_TEACHER_TEXT_SCRIPT = globals().get(
    "PREFLIGHT_TEACHER_TEXT_SCRIPT",
    SCRIPTS_DIR / "preflight_teacher_text_production.py",
)
TEACHER_TEXT_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl",
)
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
ENRICHED_CORPUS = globals().get(
    "ENRICHED_CORPUS",
    PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
VALIDATION_CALIBRATION_REPORT = PROCESSED_DIR / "teacher_text_model_comparison/teacher_text_model_calibration_validation_eval.json"
MUTATION_CALIBRATION_REPORT = PROCESSED_DIR / "teacher_text_model_comparison/teacher_text_model_calibration_mutation_eval.json"
TEACHER_TEXT_PREFLIGHT_REPORT = PROCESSED_DIR / "teacher_text_production_preflight_report.json"

cmd = [
    sys.executable,
    str(PREFLIGHT_TEACHER_TEXT_SCRIPT),
    "--parent-manifest", str(TEACHER_TEXT_MANIFEST_FILE),
    "--validation-manifest", str(TEACHER_TEXT_VALIDATION_MANIFEST_FILE),
    "--mutation-manifest", str(TEACHER_TEXT_MUTATION_MANIFEST_FILE),
    "--source-file", str(ENRICHED_CORPUS),
    "--validation-report", str(VALIDATION_CALIBRATION_REPORT),
    "--mutation-report", str(MUTATION_CALIBRATION_REPORT),
    "--existing-output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
    "--output-report", str(TEACHER_TEXT_PREFLIGHT_REPORT),
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("teacher-text production preflight completed with return code:", result.returncode)


## Stage H-Policy — Freeze The Role-Specific Teacher-Text Production Policy

This is the **policy-commitment stage** for teacher-text generation. The goal here is not to generate outputs yet, but to freeze the model/temperature choice per role so the later production run is methodologically traceable to the calibration evidence.

Current frozen mapping:
- `validation_diagnosis` -> `gpt-5.4` at `0.1`
- `mutation_robustness` -> `gpt-5.4-mini` at `0.1`

What this stage does:
- materializes the role-specific production manifests
- prints the role counts for those manifests
- prints the frozen policy table so the production run is auditable later

Why this matters:
- it prevents the teacher-text branch from quietly inheriting one model for all roles
- it provides a clean paper trail from calibration to production
- it reduces the chance of later confusion when the batch path is resumed after interruption

Operational rule:
- run this once before `Stage H-Batch`
- if the printed counts or role mapping look wrong, stop here rather than proceeding to `H-B1`


In [ ]:
from collections import Counter
import json

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


TEACHER_TEXT_ROLE_POLICY = {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}

role_rows = {role: [] for role in TEACHER_TEXT_ROLE_POLICY}
with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        row = json.loads(line)
        role = row.get("seed_role")
        if role in role_rows:
            role_rows[role].append(row)

for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    manifest_file = cfg["manifest_file"]
    manifest_file.parent.mkdir(parents=True, exist_ok=True)
    with manifest_file.open("w", encoding="utf-8") as handle:
        for row in role_rows[role]:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

print("teacher-text role-specific production manifests written")
for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    print(f"  {role}: {cfg['manifest_file']}")

print("\nmanifest counts by role")
total_rows = 0
for role, rows in role_rows.items():
    total_rows += len(rows)
    print(f"  {role}: {len(rows):,}")
print("  combined teacher-text rows:", f"{total_rows:,}")

print("\nfrozen teacher-text production policy")
print("role                   | model        | temperature | manifest")
print("-----------------------+--------------+-------------+---------")
for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    print(
        f"{role:<23} | "
        f"{cfg['model']:<12} | "
        f"{cfg['temperature']:<11} | "
        f"{cfg['manifest_file'].name}"
    )


## Stage H — Deprecated Synchronous Teacher-Text Generation

This section exists only so the notebook preserves the historical synchronous implementation. It is **not** the agreed production path. It is kept for debugging or forensic reference, not for normal execution.

Why it should not be used for production:
- it is materially more expensive than the Batch API route
- it is slower for the majority-corpus teacher-text branch
- it is harder to resume cleanly after interruption
- it is easier to confuse with the real production flow if not explicitly marked as deprecated

Normal rule:
- ignore this section during standard Stage H execution
- use `Stage H-Batch` instead

The code cell directly below is intentionally blocked unless you explicitly set an override flag. That safeguard is there to prevent accidental production use.


In [ ]:
ALLOW_DEPRECATED_SYNC_STAGE_H = globals().get("ALLOW_DEPRECATED_SYNC_STAGE_H", False)
if not ALLOW_DEPRECATED_SYNC_STAGE_H:
    raise RuntimeError(
        "Deprecated synchronous Stage H is disabled. Use Stage H-Batch instead. "
        "If you explicitly need the legacy synchronous path for debugging, set "
        "ALLOW_DEPRECATED_SYNC_STAGE_H = True in a separate cell and rerun this cell intentionally."
    )

import json
import subprocess

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}


def ensure_teacher_text_role_manifests(policy):
    manifest_files = [cfg["manifest_file"] for cfg in policy.values()]
    if all(path.exists() for path in manifest_files):
        return

    role_rows = {role: [] for role in policy}
    with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            role = row.get("seed_role")
            if role in role_rows:
                role_rows[role].append(row)

    for role, cfg in policy.items():
        manifest_file = cfg["manifest_file"]
        manifest_file.parent.mkdir(parents=True, exist_ok=True)
        with manifest_file.open("w", encoding="utf-8") as handle:
            for row in role_rows[role]:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()
ensure_teacher_text_role_manifests(TEACHER_TEXT_ROLE_POLICY)

TEACHER_TEXT_MAX_RECORDS_PER_ROLE = None
TEACHER_TEXT_DRY_RUN = False

print("teacher-text output:", PRODUCTION_TEACHER_TEXT_SEEDS)
print("teacher-text error log:", PRODUCTION_TEACHER_TEXT_ERRORS)

for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    print("\nrole:", role)
    print("  manifest:", cfg["manifest_file"])
    print("  model:", cfg["model"])
    print("  temperature:", cfg["temperature"])

    cmd = [
        sys.executable,
        str(GENERATE_DRAFTS_SCRIPT),
        "--manifest-file", str(cfg["manifest_file"]),
        "--source-file", str(ENRICHED_CORPUS),
        "--output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
        "--log-file", str(PRODUCTION_TEACHER_TEXT_ERRORS),
        "--model", cfg["model"],
        "--temperature", str(cfg["temperature"]),
    ]

    if TEACHER_TEXT_MAX_RECORDS_PER_ROLE is not None:
        cmd.extend(["--max-records", str(TEACHER_TEXT_MAX_RECORDS_PER_ROLE)])

    if TEACHER_TEXT_DRY_RUN:
        cmd.append("--dry-run")

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print(f"teacher-text generation completed for {role} with return code:", result.returncode)


In [ ]:
import json

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)

from collections import Counter

def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}


def ensure_teacher_text_role_manifests(policy):
    manifest_files = [cfg["manifest_file"] for cfg in policy.values()]
    if all(path.exists() for path in manifest_files):
        return

    role_rows = {role: [] for role in policy}
    with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            role = row.get("seed_role")
            if role in role_rows:
                role_rows[role].append(row)

    for role, cfg in policy.items():
        manifest_file = cfg["manifest_file"]
        manifest_file.parent.mkdir(parents=True, exist_ok=True)
        with manifest_file.open("w", encoding="utf-8") as handle:
            for row in role_rows[role]:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()

if not PRODUCTION_TEACHER_TEXT_SEEDS.exists():
    print("No teacher-text seed draft file yet.")
    if PRODUCTION_TEACHER_TEXT_ERRORS.exists():
        print("\nRecent error-log preview:\n")
        with open(PRODUCTION_TEACHER_TEXT_ERRORS, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                row = json.loads(line)
                print(json.dumps({
                    "error_type": row.get("error_type"),
                    "error_message": row.get("error_message"),
                    "seed_role": row.get("seed_role"),
                }, ensure_ascii=False, indent=2))
                print()
    else:
        print("Run the production teacher-text seed cell to materialize outputs.")
else:
    role_counts = Counter()
    mode_counts = Counter()
    prompt_type_counts = Counter()
    temperature_counts = Counter()
    model_role_counts = Counter()
    unique_hashes = set()
    rows = 0

    with open(PRODUCTION_TEACHER_TEXT_SEEDS, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            meta = row.get("metadata", {})
            role = meta.get("seed_role", "<missing>")
            model = meta.get("generation_model", "<missing>")
            role_counts[role] += 1
            mode_counts[meta.get("seed_target_supervision_mode", "<missing>")] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            temperature_counts[str(meta.get("seed_generation_temperature", "<missing>"))] += 1
            model_role_counts[f"{role} | {model}"] += 1
            unique_hashes.add(meta.get("circuit_hash", ""))

    print("Teacher-text seed counts by role")
    for key, value in role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nPrompt types")
    for key, value in prompt_type_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nTarget supervision modes")
    for key, value in mode_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nSeed draft temperatures present")
    for key, value in temperature_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nGeneration models by role")
    for key, value in model_role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nTeacher-text manifest sizes by role")
    combined_manifest_rows = 0
    for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
        role_rows = sum(1 for _ in open(cfg["manifest_file"], encoding="utf-8") if _.strip()) if cfg["manifest_file"].exists() else 0
        combined_manifest_rows += role_rows
        print(f"  {role}: {role_rows:,}")

    print("\nTeacher-text seed coverage")
    print("  rows:", f"{rows:,}")
    print("  unique circuit_hash values:", f"{len(unique_hashes):,}")
    print("  combined teacher-text manifest size:", f"{combined_manifest_rows:,}")


### Stage H-Batch — Batch Execution for Role-Specific Teacher-Text Seeds

This is the **normal production path** for teacher-text seed generation. It applies the frozen role-specific policy using the Batch API so the run is cheaper, more resumable, and easier to document than the old synchronous route.

Role-specific production mapping used in this section:
- `validation_diagnosis` on `gpt-5.4`
- `mutation_robustness` on `gpt-5.4-mini`

What this section does end to end:
- prepares role-specific batch request files
- creates the batch jobs
- waits for completion and downloads outputs
- materializes the combined teacher-text artifact into the canonical PQID JSONL file
- audits the resulting artifact locally

Interpretation rule:
- if everything behaves normally, stay entirely inside `H-B1` to `H-B5`
- if the validation batch fails at the request-file validation boundary, move to `Stage H-BR` and then return to `H-B4` and `H-B5` afterward


#### How To Use Stage H-Batch

This subsection defines the **canonical order of execution** for the teacher-text batch path. The numbering is meant to be read literally: each step depends on the artifacts produced by the previous one.

Mandatory normal path:
1. `H-B1` prepares request files for each role and reports how many rows are new versus skipped from existing outputs.
2. `H-B2` uploads those request files and creates one saved batch-state file per role.
3. `H-B3` uses the saved batch ids to wait for completion and download each role-specific output/error file.
4. `H-B4` materializes the downloaded role-specific batch outputs into the single canonical teacher-text seed artifact.
5. `H-B5` audits the materialized artifact and checks whether coverage matches the manifest expectations.

Optional helper:
- `H-B2a` only inspects saved state files. It is informative, not generative. It should never be confused with `H-B2`.

Failure-handling rule:
- if `H-B3` reports that the validation branch failed during validation before accepting any requests, stop the normal path and move to `H-BR1` through `H-BR4`
- after recovery, return to `H-B4` and `H-B5`


##### Step H-B1 — Prepare Role-Specific Teacher-Text Batch Request Files

Purpose:
- refresh the role-specific manifests if needed
- generate one batch request JSONL for `validation_diagnosis` and one for `mutation_robustness`
- skip rows that are already present in the existing teacher-text output artifact so interrupted work is preserved

What to inspect in the output:
- `requests written`: how many still need generation
- `requests skipped from existing output`: how much prior work is being preserved
- `request file size`: whether the request file is close to or above the Batch API file-size limit

Operational rule:
- do not continue to `H-B2` until both role-specific request files have been reported
- if the validation request file is oversized, continue to `H-B2` only if it is still under the safe threshold; otherwise expect to use `Stage H-BR`


In [ ]:
import json
import subprocess

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}


def ensure_teacher_text_role_manifests(policy):
    manifest_files = [cfg["manifest_file"] for cfg in policy.values()]
    if all(path.exists() for path in manifest_files):
        return

    role_rows = {role: [] for role in policy}
    with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            role = row.get("seed_role")
            if role in role_rows:
                role_rows[role].append(row)

    for role, cfg in policy.items():
        manifest_file = cfg["manifest_file"]
        manifest_file.parent.mkdir(parents=True, exist_ok=True)
        with manifest_file.open("w", encoding="utf-8") as handle:
            for row in role_rows[role]:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()
ensure_teacher_text_role_manifests(TEACHER_TEXT_ROLE_POLICY)

TEACHER_TEXT_BATCH_MAX_RECORDS_PER_ROLE = None

print("teacher-text output artifact:", PRODUCTION_TEACHER_TEXT_SEEDS)

for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    print("\nrole:", role)
    print("  manifest:", cfg["manifest_file"])
    print("  request file:", cfg["batch_request_file"])
    print("  model:", cfg["model"])
    print("  temperature:", cfg["temperature"])

    cmd = [
        sys.executable,
        str(PREPARE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(cfg["manifest_file"]),
        "--source-file", str(ENRICHED_CORPUS),
        "--request-file", str(cfg["batch_request_file"]),
        "--existing-output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
        "--model", cfg["model"],
        "--temperature", str(cfg["temperature"]),
    ]

    if TEACHER_TEXT_BATCH_MAX_RECORDS_PER_ROLE is not None:
        cmd.extend(["--max-records", str(TEACHER_TEXT_BATCH_MAX_RECORDS_PER_ROLE)])

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print(f"teacher-text seed batch preparation completed for {role} with return code:", result.returncode)


##### Optional Step H-B2a — Inspect Role-Specific Teacher-Text Batch State Files

This is a **read-only helper**. It does not upload request files, does not create jobs, and does not wait for anything. It only prints the currently saved state for each role-specific batch job if those state files already exist.

Use this when:
- you are resuming after a restart and want to see whether batch ids were already created
- you want to confirm the saved output/error-file paths

Do not mistake this for `H-B2`. If the state files say `no saved state yet`, that means the jobs have **not** been created.


In [ ]:
import json

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}

TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()


print("role-specific teacher-text batch state files")
for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    print("\nrole:", role)
    print("  request file:", cfg["batch_request_file"])
    print("  state file:", cfg["batch_state_file"])
    print("  output file:", cfg["batch_output_file"])
    print("  error file:", cfg["batch_error_file"])
    if cfg["batch_state_file"].exists():
        state = json.loads(cfg["batch_state_file"].read_text(encoding="utf-8"))
        print("  batch id:", state.get("batch_id"))
        print("  status:", state.get("status"))
    else:
        print("  status: no saved state yet")


#### Fixed-Path Teacher-Text Batch Continuation

The cells below separate **batch creation** from **waiting/downloading** so the notebook remains easier to resume after interruptions. This is especially important for long teacher-text runs where you may come back hours later or after a restart.

Interpretation:
- `H-B2` is the creation boundary
- `H-B3` is the monitoring/download boundary

If a batch fails before requests are accepted, that is usually a request-file-level issue. In that case, stop here and use `Stage H-BR`.


##### Step H-B2 — Create Role-Specific Teacher-Text Batch Jobs

This step uploads each prepared request file to OpenAI and creates one batch job per role. It should also write one saved state file per role.

What success looks like:
- each role prints an uploaded request file id
- each role prints a batch id
- each role writes a saved state file
- initial status is usually `validating` or another early nonterminal state

What failure means here:
- if the job fails before any requests are accepted, the problem is usually with request-file size or request-file validity rather than the model outputs themselves
- that is the exact case handled by `Stage H-BR` for the validation branch


In [ ]:
import json
import subprocess

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}

TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()

for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    if not cfg["batch_request_file"].exists():
        print(f"{role}: prepare the request file first.")
        continue

    print("\nrole:", role)
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(cfg["batch_request_file"]),
        "--state-file", str(cfg["batch_state_file"]),
        "--download-output-file", str(cfg["batch_output_file"]),
        "--download-error-file", str(cfg["batch_error_file"]),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print(f"teacher-text seed batch creation completed for {role} with return code:", result.returncode)
    if cfg["batch_state_file"].exists():
        state = json.loads(cfg["batch_state_file"].read_text(encoding="utf-8"))
        print("batch id:", state.get("batch_id"))
        print("batch status:", state.get("status"))


##### Step H-B3 — Wait For Completion And Download Role-Specific Teacher-Text Batch Files

This step resumes from the saved batch ids, waits for each role-specific batch to reach a terminal status, and downloads the output/error files that will later be materialized into the canonical artifact.

How to read the statuses:
- `in_progress`: requests are being processed
- `finalizing`: requests are done and OpenAI is assembling output files
- `completed`: the batch finished successfully and should have an output file
- `failed`: the batch did not complete successfully; inspect whether this happened before any requests were accepted or after some processing

Special rule for this notebook:
- if `validation_diagnosis` fails with `total = 0` and no output/error file id, treat it as a validation-boundary failure and move to `Stage H-BR`
- if `mutation_robustness` completes successfully, keep its result and do not rerun it unless there is evidence of corruption


In [ ]:
import json
import subprocess

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}

TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()

for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    if not cfg["batch_state_file"].exists():
        print(f"{role}: no batch state file yet. Run the create cell first.")
        continue

    state = json.loads(cfg["batch_state_file"].read_text(encoding="utf-8"))
    batch_id = state.get("batch_id", "")
    if not batch_id:
        print(f"{role}: no batch_id recorded in the saved state file.")
        continue

    print("\nrole:", role)
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", str(cfg["batch_state_file"]),
        "--wait",
        "--download-output-file", str(cfg["batch_output_file"]),
        "--download-error-file", str(cfg["batch_error_file"]),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print(f"teacher-text seed batch wait/download completed for {role} with return code:", result.returncode)


## Stage H-BR — Validation-Diagnosis Oversized Batch Recovery

This recovery block exists for one very specific failure mode: the `validation_diagnosis` request file was too large for the Batch API input-file limit, so the validation batch failed **before accepting any requests**.

Why this recovery block is separate:
- the failure is operational, not conceptual
- the already successful `mutation_robustness` batch should be preserved
- only the oversized validation request file needs intervention

What the recovery block does:
1. shard the oversized validation request JSONL into smaller deterministic parts
2. create a batch job for each shard
3. wait for those shard jobs and download their outputs
4. merge the shard outputs back into the canonical validation batch-output/error files expected by `H-B4`

After `H-BR4`, return to the normal path at `H-B4` and `H-B5`.


##### Step H-BR1 — Shard The Oversized Validation-Diagnosis Request File

This step splits the oversized validation request file into deterministic shard files below the Batch API size limit. It also writes an index file that records every shard request file plus the derivative state/output/error paths that will be used by the later recovery steps.

What to inspect:
- shard count
- rows per shard
- bytes per shard

Do not continue until the shard summary looks plausible and every shard file is comfortably below the Batch API hard limit.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get("VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE", BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl")
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get("VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl")
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get("VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl")
SHARD_BATCH_REQUEST_SCRIPT = globals().get("SHARD_BATCH_REQUEST_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/shard_openai_batch_request_file.py")
VALIDATION_DIAGNOSIS_SHARD_REQUEST_PREFIX = globals().get("VALIDATION_DIAGNOSIS_SHARD_REQUEST_PREFIX", BATCH_DIR / "teacher_text_validation_seed_requests_v1_shard")
VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE = globals().get("VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_shards_v1.json")
VALIDATION_DIAGNOSIS_SHARD_MAX_BYTES = globals().get("VALIDATION_DIAGNOSIS_SHARD_MAX_BYTES", 190 * 1024 * 1024)

cmd = [
    sys.executable,
    str(SHARD_BATCH_REQUEST_SCRIPT),
    "--input-file", str(VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE),
    "--request-prefix", str(VALIDATION_DIAGNOSIS_SHARD_REQUEST_PREFIX),
    "--index-file", str(VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE),
    "--max-bytes", str(VALIDATION_DIAGNOSIS_SHARD_MAX_BYTES),
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.rstrip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.rstrip())
print("validation request sharding completed with return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Validation request sharding failed.")


##### Step H-BR2 — Create Validation-Diagnosis Shard Batch Jobs

This creates one validation batch job per shard. The notebook reuses an existing shard state file if it already contains a `batch_id`, which makes this step restart-safe.

Expected output per shard:
- uploaded request file
- input file id
- batch id
- initial batch status


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py")
VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE = globals().get("VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_shards_v1.json")

shard_payload = json.loads(VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE.read_text(encoding="utf-8"))
for shard in shard_payload.get("shards", []):
    state_file = Path(shard["state_file"])
    if state_file.exists():
        state = json.loads(state_file.read_text(encoding="utf-8"))
        if state.get("batch_id"):
            print(f"part{shard['part']:03d}: existing batch id {state['batch_id']} (status={state.get('status')})")
            continue
    print(f"\nvalidation shard part{shard['part']:03d}")
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", shard["request_file"],
        "--state-file", shard["state_file"],
        "--download-output-file", shard["output_file"],
        "--download-error-file", shard["error_file"],
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"validation shard batch creation completed for part{shard['part']:03d} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Validation shard batch creation failed.")


##### Step H-BR3 — Wait For Completion And Download Validation-Diagnosis Shard Batch Files

This waits on every validation shard batch and downloads each shard output/error file. It is restart-safe because it reads the shard state index and saved batch ids.

Operational note:
- one shard failing does not automatically invalidate the others
- but do not merge until you understand the status of every shard


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py")
VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE = globals().get("VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_shards_v1.json")

shard_payload = json.loads(VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE.read_text(encoding="utf-8"))
for shard in shard_payload.get("shards", []):
    state_file = Path(shard["state_file"])
    if not state_file.exists():
        raise RuntimeError(f"Missing state file for validation shard part{shard['part']:03d}: {state_file}")
    state = json.loads(state_file.read_text(encoding="utf-8"))
    batch_id = state.get("batch_id")
    if not batch_id:
        raise RuntimeError(f"Missing batch_id in state file for validation shard part{shard['part']:03d}")
    print(f"\nvalidation shard part{shard['part']:03d}")
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", shard["state_file"],
        "--wait",
        "--download-output-file", shard["output_file"],
        "--download-error-file", shard["error_file"],
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"validation shard batch wait/download completed for part{shard['part']:03d} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Validation shard batch wait/download failed.")


##### Step H-BR4 — Merge Validation-Diagnosis Shard Outputs Into The Canonical Batch Files

This reconstructs the standard validation batch output/error JSONL files from the shard downloads so the ordinary `H-B4` materialization cell can run unchanged. In other words, this step converts the recovery path back into the canonical interface expected by the rest of the notebook.

Once this step succeeds, leave the recovery block and return to the normal path:
- `H-B4` materialization
- `H-B5` audit


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
MERGE_BATCH_SHARDS_SCRIPT = globals().get("MERGE_BATCH_SHARDS_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/merge_openai_batch_shard_outputs.py")
VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE = globals().get("VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_shards_v1.json")
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get("VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl")
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get("VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE", BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl")

cmd = [
    sys.executable,
    str(MERGE_BATCH_SHARDS_SCRIPT),
    "--index-file", str(VALIDATION_DIAGNOSIS_SHARD_INDEX_FILE),
    "--merged-output-file", str(VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE),
    "--merged-error-file", str(VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE),
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.rstrip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.rstrip())
print("validation shard merge completed with return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Validation shard merge failed.")


##### Step H-B4 — Materialize Combined Teacher-Text Seed Artifact

Run this after the wait/download step. It materializes the two role-specific batch outputs into the shared PQID teacher-text artifact, while skipping any role that is already fully covered in the existing output file.


In [ ]:
import json
import subprocess
from collections import Counter

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)


def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}


def ensure_teacher_text_role_manifests(policy):
    manifest_files = [cfg["manifest_file"] for cfg in policy.values()]
    if all(path.exists() for path in manifest_files):
        return

    role_rows = {role: [] for role in policy}
    with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            role = row.get("seed_role")
            if role in role_rows:
                role_rows[role].append(row)

    for role, cfg in policy.items():
        manifest_file = cfg["manifest_file"]
        manifest_file.parent.mkdir(parents=True, exist_ok=True)
        with manifest_file.open("w", encoding="utf-8") as handle:
            for row in role_rows[role]:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()

def load_completed_role_counts(path):
    counts = Counter()
    if not path.exists():
        return counts
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            meta = row.get("metadata", {})
            counts[meta.get("seed_role", "<missing>")] += 1
    return counts

completed_role_counts = load_completed_role_counts(PRODUCTION_TEACHER_TEXT_SEEDS)

for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
    expected_rows = sum(1 for _ in open(cfg["manifest_file"], encoding="utf-8") if _.strip()) if cfg["manifest_file"].exists() else 0
    completed_rows = completed_role_counts.get(role, 0)
    if expected_rows and completed_rows >= expected_rows:
        print(f"{role}: already materialized {completed_rows:,}/{expected_rows:,} rows; skipping.")
        continue
    if not cfg["batch_output_file"].exists():
        print(f"{role}: no downloaded batch output yet.")
        continue

    print("\nrole:", role)
    cmd = [
        sys.executable,
        str(MATERIALIZE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(cfg["manifest_file"]),
        "--source-file", str(ENRICHED_CORPUS),
        "--batch-output-file", str(cfg["batch_output_file"]),
        "--batch-error-file", str(cfg["batch_error_file"]),
        "--output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
        "--log-file", str(PRODUCTION_TEACHER_TEXT_ERRORS),
        "--model", cfg["model"],
        "--temperature", str(cfg["temperature"]),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print(f"teacher-text seed batch materialization completed for {role} with return code:", result.returncode)
    completed_role_counts = load_completed_role_counts(PRODUCTION_TEACHER_TEXT_SEEDS)


##### Step H-B5 — Audit Teacher-Text Seed Artifact

Run this immediately after materialization. This is a local duplicate of the Stage H teacher-text audit so the whole teacher-text batch path is self-contained.

Interpretation rule:
- if the combined teacher-text rows equal the combined teacher-text manifest size, Stage H is closed
- if coverage is still short, do **not** continue to Stage I or Stage J yet; move immediately into `Stage H-Retry` below and close the residual gap first


In [ ]:
import json

from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
TEACHER_TEXT_MANIFEST_FILE = globals().get("TEACHER_TEXT_MANIFEST_FILE", PROCESSED_DIR / "seed_role_manifest_v1_teacher_text.jsonl")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_requests_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_v1.json",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
)
VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE = globals().get(
    "VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_requests_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_v1.json",
)
MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
)
MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE = globals().get(
    "MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
)
GENERATE_DRAFTS_SCRIPT = globals().get(
    "GENERATE_DRAFTS_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/generate_seed_drafts_quality_aware.py",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)

from collections import Counter

def teacher_text_role_policy():
    return {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "batch_request_file": VALIDATION_DIAGNOSIS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": VALIDATION_DIAGNOSIS_SEED_BATCH_STATE_FILE,
        "batch_output_file": VALIDATION_DIAGNOSIS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": VALIDATION_DIAGNOSIS_SEED_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "batch_request_file": MUTATION_ROBUSTNESS_SEED_BATCH_REQUEST_FILE,
        "batch_state_file": MUTATION_ROBUSTNESS_SEED_BATCH_STATE_FILE,
        "batch_output_file": MUTATION_ROBUSTNESS_SEED_BATCH_OUTPUT_FILE,
        "batch_error_file": MUTATION_ROBUSTNESS_SEED_BATCH_ERROR_FILE,
    },
}


def ensure_teacher_text_role_manifests(policy):
    manifest_files = [cfg["manifest_file"] for cfg in policy.values()]
    if all(path.exists() for path in manifest_files):
        return

    role_rows = {role: [] for role in policy}
    with open(TEACHER_TEXT_MANIFEST_FILE, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            role = row.get("seed_role")
            if role in role_rows:
                role_rows[role].append(row)

    for role, cfg in policy.items():
        manifest_file = cfg["manifest_file"]
        manifest_file.parent.mkdir(parents=True, exist_ok=True)
        with manifest_file.open("w", encoding="utf-8") as handle:
            for row in role_rows[role]:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")


TEACHER_TEXT_ROLE_POLICY = globals().get("TEACHER_TEXT_ROLE_POLICY") or teacher_text_role_policy()

if not PRODUCTION_TEACHER_TEXT_SEEDS.exists():
    print("No teacher-text seed draft file yet.")
    if PRODUCTION_TEACHER_TEXT_ERRORS.exists():
        print("\nRecent error-log preview:\n")
        with open(PRODUCTION_TEACHER_TEXT_ERRORS, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                row = json.loads(line)
                print(json.dumps({
                    "error_type": row.get("error_type"),
                    "error_message": row.get("error_message"),
                    "seed_role": row.get("seed_role"),
                }, ensure_ascii=False, indent=2))
                print()
    else:
        print("Run the production teacher-text seed cell to materialize outputs.")
else:
    role_counts = Counter()
    mode_counts = Counter()
    prompt_type_counts = Counter()
    temperature_counts = Counter()
    model_role_counts = Counter()
    unique_hashes = set()
    rows = 0

    with open(PRODUCTION_TEACHER_TEXT_SEEDS, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            meta = row.get("metadata", {})
            role = meta.get("seed_role", "<missing>")
            model = meta.get("generation_model", "<missing>")
            role_counts[role] += 1
            mode_counts[meta.get("seed_target_supervision_mode", "<missing>")] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            temperature_counts[str(meta.get("seed_generation_temperature", "<missing>"))] += 1
            model_role_counts[f"{role} | {model}"] += 1
            unique_hashes.add(meta.get("circuit_hash", ""))

    print("Teacher-text seed counts by role")
    for key, value in role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nPrompt types")
    for key, value in prompt_type_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nTarget supervision modes")
    for key, value in mode_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nSeed draft temperatures present")
    for key, value in temperature_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nGeneration models by role")
    for key, value in model_role_counts.most_common():
        print(f"  {key}: {value:,}")

    print("\nTeacher-text manifest sizes by role")
    combined_manifest_rows = 0
    for role, cfg in TEACHER_TEXT_ROLE_POLICY.items():
        role_rows = sum(1 for _ in open(cfg["manifest_file"], encoding="utf-8") if _.strip()) if cfg["manifest_file"].exists() else 0
        combined_manifest_rows += role_rows
        print(f"  {role}: {role_rows:,}")

    print("\nTeacher-text seed coverage")
    print("  rows:", f"{rows:,}")
    print("  unique circuit_hash values:", f"{len(unique_hashes):,}")
    print("  combined teacher-text manifest size:", f"{combined_manifest_rows:,}")


## Stage H-Retry — Residual Teacher-Text Gap Closure

Use this subsection **only if `H-B5` still shows missing teacher-text rows** after the normal Stage H batch path.

This is the post-materialization cleanup path. It exists because a batch can complete at the request level while still leaving a smaller residual gap after materialization, for example due to:

- malformed or truncated JSON in otherwise usable downloaded responses
- parser/materialization failures against responses that already exist on disk
- true request failures such as quota or incomplete-response errors

Important order of operations in this subsection:
1. `H-R1` identifies the live gap from the canonical teacher-text artifact.
2. `H-R2A` performs an **offline recovery pass** from the downloaded Stage H batch outputs you already have. This costs nothing and should always be tried before any paid retry.
3. `H-R2B` re-audits coverage after that offline recovery.
4. Only if rows are still missing after `H-R2B` should you continue to the paid retry path in `H-R2` through `H-R6`.

For the current teacher-text run, this distinction matters: most of the residual gap can be recovered locally from downloaded batch files, while only the true unseen/quota-hit rows should need a paid retry.


##### Step H-R1 — Build Teacher-Text Retry Manifests From Still-Missing Rows

Run this immediately after `H-B5` if coverage is still short.

This compares each teacher-text role manifest against the **current materialized teacher-text output artifact** and writes one retry manifest per role containing only the rows that are still missing.

Expected interpretation:
- if a retry manifest has `0` rows, that role is already closed
- if a retry manifest has `> 0` rows, that role still needs a targeted retry pass

These retry manifests are the authoritative inputs for the rest of `Stage H-Retry`. They are rebuilt from the live artifact, so they are safe to rerun after another retry pass or after a restart.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT = globals().get(
    "BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/build_missing_seed_retry_manifest.py",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_retry.jsonl",
)
TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_retry.jsonl",
)

TEACHER_TEXT_RETRY_POLICY = {
    "validation_diagnosis": {
        "manifest_file": TEACHER_TEXT_VALIDATION_MANIFEST_FILE,
        "retry_manifest_file": TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE,
        "model": "gpt-5.4",
        "temperature": 0.1,
        "batch_request_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_validation_seed_retry_requests_v1.jsonl",
        "batch_state_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_validation_seed_retry_batch_v1.json",
        "batch_output_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_validation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_validation_seed_retry_batch_error_v1.jsonl",
    },
    "mutation_robustness": {
        "manifest_file": TEACHER_TEXT_MUTATION_MANIFEST_FILE,
        "retry_manifest_file": TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE,
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "batch_request_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_mutation_seed_retry_requests_v1.jsonl",
        "batch_state_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_mutation_seed_retry_batch_v1.json",
        "batch_output_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_mutation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": PROCESSED_DIR / "openai_batch_jobs/teacher_text_mutation_seed_retry_batch_error_v1.jsonl",
    },
}
globals()["TEACHER_TEXT_RETRY_POLICY"] = TEACHER_TEXT_RETRY_POLICY

for role, cfg in TEACHER_TEXT_RETRY_POLICY.items():
    print("\nrole:", role)
    cmd = [
        sys.executable,
        str(BUILD_MISSING_SEED_RETRY_MANIFEST_SCRIPT),
        "--manifest-file", str(cfg["manifest_file"]),
        "--output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
        "--retry-manifest-file", str(cfg["retry_manifest_file"]),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"teacher-text retry manifest build completed for {role} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Teacher-text retry manifest build failed for {role}.")


##### Step H-R2A — Recover Missing Teacher-Text Rows From Existing Downloaded Batch Outputs

Run this immediately after `H-R1`, before any paid retry.

This is the **offline recovery pass**. It reuses the Stage H batch outputs you already downloaded and tries to materialize any still-missing rows from those files using a more tolerant parser and rerun-safe materialization.

This step is the financially preferred path because it does **not** create new API requests. It is specifically meant to recover rows that were already generated but failed local materialization because of malformed/truncated JSON formatting or parser sensitivity.

Expected interpretation:
- if many rows are recovered here, that is good and expected
- rows still missing after this step are much more likely to be true unseen/quota-hit cases that really need a paid retry


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_retry.jsonl",
)
TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_retry.jsonl",
)

DEFAULT_TEACHER_TEXT_OFFLINE_RECOVERY_POLICY = {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "retry_manifest_file": TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE,
        "batch_output_file": BATCH_DIR / "teacher_text_validation_seed_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_validation_seed_batch_error_v1.jsonl",
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "retry_manifest_file": TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE,
        "batch_output_file": BATCH_DIR / "teacher_text_mutation_seed_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_mutation_seed_batch_error_v1.jsonl",
    },
}
TEACHER_TEXT_OFFLINE_RECOVERY_POLICY = DEFAULT_TEACHER_TEXT_OFFLINE_RECOVERY_POLICY
globals()["TEACHER_TEXT_OFFLINE_RECOVERY_POLICY"] = TEACHER_TEXT_OFFLINE_RECOVERY_POLICY


def load_completed_keys(path: Path):
    completed = set()
    if not path.exists():
        return completed
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            meta = row.get("metadata", {})
            completed.add((meta.get("circuit_hash"), meta.get("seed_role")))
    return completed


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def iter_jsonl(path: Path):
    if not path.exists():
        return
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def custom_id_for_manifest_row(row: dict) -> str:
    return f"seed::{row['source_record']['circuit_hash']}::{row['seed_role']}"


completed_keys = load_completed_keys(PRODUCTION_TEACHER_TEXT_SEEDS)

for role, cfg in TEACHER_TEXT_OFFLINE_RECOVERY_POLICY.items():
    retry_manifest_file = cfg["retry_manifest_file"]
    batch_output_file = cfg["batch_output_file"]
    batch_error_file = cfg["batch_error_file"]
    print("\nrole:", role)
    if not retry_manifest_file.exists():
        print("  no retry manifest yet")
        continue
    retry_rows = [row for row in iter_jsonl(retry_manifest_file)]
    pending_rows = [
        row for row in retry_rows
        if (row["source_record"]["circuit_hash"], row["seed_role"]) not in completed_keys
    ]
    print("  retry manifest rows:", f"{len(retry_rows):,}")
    print("  still-missing rows before offline recovery:", f"{len(pending_rows):,}")
    if not pending_rows:
        print("  all rows from this retry manifest are already materialized; skipping")
        continue
    if not batch_output_file.exists() and not batch_error_file.exists():
        print("  no downloaded main Stage H batch output or error files are available for this role")
        continue

    pending_ids = {custom_id_for_manifest_row(row) for row in pending_rows}
    pending_manifest_file = retry_manifest_file.with_name(retry_manifest_file.stem + "_offline_recovery_pending.jsonl")
    pending_output_file = batch_output_file.with_name(batch_output_file.stem + "_offline_recovery_pending.jsonl")
    pending_error_file = batch_error_file.with_name(batch_error_file.stem + "_offline_recovery_pending.jsonl")

    filtered_output_rows = [row for row in iter_jsonl(batch_output_file) if row.get("custom_id") in pending_ids] if batch_output_file.exists() else []
    filtered_error_rows = [row for row in iter_jsonl(batch_error_file) if row.get("custom_id") in pending_ids] if batch_error_file.exists() else []
    print("  matching downloaded output rows:", f"{len(filtered_output_rows):,}")
    print("  matching downloaded error rows:", f"{len(filtered_error_rows):,}")
    if not filtered_output_rows and not filtered_error_rows:
        print("  no matching downloaded rows exist for this role; a paid retry will still be needed")
        continue

    write_jsonl(pending_manifest_file, pending_rows)
    write_jsonl(pending_output_file, filtered_output_rows)
    write_jsonl(pending_error_file, filtered_error_rows)

    cmd = [
        sys.executable,
        str(MATERIALIZE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(pending_manifest_file),
        "--source-file", str(ENRICHED_CORPUS),
        "--batch-output-file", str(pending_output_file),
        "--batch-error-file", str(pending_error_file),
        "--output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
        "--log-file", str(PRODUCTION_TEACHER_TEXT_ERRORS),
        "--model", cfg["model"],
        "--temperature", str(cfg["temperature"]),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"teacher-text offline recovery completed for {role} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Teacher-text offline recovery failed for {role}.")
    completed_keys = load_completed_keys(PRODUCTION_TEACHER_TEXT_SEEDS)


##### Step H-R2B — Re-Audit Teacher-Text Coverage After Offline Recovery

Run this immediately after `H-R2A`.

This is the decision gate between the free local recovery path and the paid retry path.

Interpretation:
- if total missing rows = `0`, Stage H is fully closed and you can move to Stage I
- if only a very small residual gap remains, continue to `H-R2` through `H-R6` for a paid retry
- if the residual gap is still unexpectedly large, stop and inspect the canonical batch outputs before spending more


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)

manifest_counts = {
    'validation_diagnosis': sum(1 for _ in open(TEACHER_TEXT_VALIDATION_MANIFEST_FILE, encoding='utf-8') if _.strip()) if TEACHER_TEXT_VALIDATION_MANIFEST_FILE.exists() else 0,
    'mutation_robustness': sum(1 for _ in open(TEACHER_TEXT_MUTATION_MANIFEST_FILE, encoding='utf-8') if _.strip()) if TEACHER_TEXT_MUTATION_MANIFEST_FILE.exists() else 0,
}
output_counts = Counter()
unique_hashes = set()
if PRODUCTION_TEACHER_TEXT_SEEDS.exists():
    with open(PRODUCTION_TEACHER_TEXT_SEEDS, encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            meta = row.get('metadata', {})
            role = meta.get('seed_role', '<missing>')
            output_counts[role] += 1
            unique_hashes.add(meta.get('circuit_hash', ''))
error_counts = Counter()
if PRODUCTION_TEACHER_TEXT_ERRORS.exists():
    with open(PRODUCTION_TEACHER_TEXT_ERRORS, encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            error_counts[row.get('seed_role', '<missing>')] += 1

missing_counts = {role: manifest_counts[role] - output_counts.get(role, 0) for role in manifest_counts}
total_missing = sum(missing_counts.values())

print('Teacher-text retry audit')
print('  total rows:', f"{sum(output_counts.values()):,}")
print('  unique circuit_hash values:', f"{len(unique_hashes):,}")
print('  combined manifest size:', f"{sum(manifest_counts.values()):,}")
print('  total missing rows:', f"{total_missing:,}")
print('\nby role')
for role in ['validation_diagnosis', 'mutation_robustness']:
    print(f"  {role}: materialized={output_counts.get(role, 0):,} expected={manifest_counts.get(role, 0):,} missing={missing_counts.get(role, 0):,} logged_errors={error_counts.get(role, 0):,}")

if total_missing == 0:
    print('\nTeacher-text retry status: CLOSED')
    print('Stage H is now fully covered. Move on to Stage I.')
else:
    print('\nTeacher-text retry status: STILL INCOMPLETE')
    print('Do not move to Stage I yet. Inspect the retry error files and decide whether to run one more tiny retry pass.')


##### Step H-R2 — Prepare Paid Retry Batch Request Files For Any Residual Gap

Run this **only if `H-R2B` still shows missing teacher-text rows** after the offline recovery pass.

This prepares small role-specific paid retry request files using the same seed-generation contract as the main Stage H path, but with a **higher retry floor** for `max_output_tokens` so we do not replay the same truncation-sensitive failures.

Current paid-retry policy:
- `validation_diagnosis` stays on `gpt-5.4`
- `mutation_robustness` stays on `gpt-5.4-mini`
- retry base `max_output_tokens` is raised to `700`

The existing teacher-text output artifact is passed in again, so the request builder will still skip any rows that were already materialized by the main Stage H path or by the offline recovery pass.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PREPARE_SEED_BATCH_SCRIPT = globals().get(
    "PREPARE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/prepare_seed_drafts_quality_aware_batch.py",
)
TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_retry.jsonl",
)
TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_retry.jsonl",
)
TEACHER_TEXT_VALIDATION_RETRY_BATCH_REQUEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_validation_seed_retry_requests_v1.jsonl",
)
TEACHER_TEXT_VALIDATION_RETRY_BATCH_STATE_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_validation_seed_retry_batch_v1.json",
)
TEACHER_TEXT_VALIDATION_RETRY_BATCH_OUTPUT_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_validation_seed_retry_batch_output_v1.jsonl",
)
TEACHER_TEXT_VALIDATION_RETRY_BATCH_ERROR_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_RETRY_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_validation_seed_retry_batch_error_v1.jsonl",
)
TEACHER_TEXT_MUTATION_RETRY_BATCH_REQUEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_BATCH_REQUEST_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_retry_requests_v1.jsonl",
)
TEACHER_TEXT_MUTATION_RETRY_BATCH_STATE_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_BATCH_STATE_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_retry_batch_v1.json",
)
TEACHER_TEXT_MUTATION_RETRY_BATCH_OUTPUT_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_retry_batch_output_v1.jsonl",
)
TEACHER_TEXT_MUTATION_RETRY_BATCH_ERROR_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_RETRY_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_mutation_seed_retry_batch_error_v1.jsonl",
)
TEACHER_TEXT_RETRY_BASE_MAX_OUTPUT_TOKENS = globals().get("TEACHER_TEXT_RETRY_BASE_MAX_OUTPUT_TOKENS", 700)

DEFAULT_TEACHER_TEXT_RETRY_POLICY = {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "retry_manifest_file": TEACHER_TEXT_VALIDATION_RETRY_MANIFEST_FILE,
        "batch_request_file": TEACHER_TEXT_VALIDATION_RETRY_BATCH_REQUEST_FILE,
        "batch_state_file": TEACHER_TEXT_VALIDATION_RETRY_BATCH_STATE_FILE,
        "batch_output_file": TEACHER_TEXT_VALIDATION_RETRY_BATCH_OUTPUT_FILE,
        "batch_error_file": TEACHER_TEXT_VALIDATION_RETRY_BATCH_ERROR_FILE,
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "retry_manifest_file": TEACHER_TEXT_MUTATION_RETRY_MANIFEST_FILE,
        "batch_request_file": TEACHER_TEXT_MUTATION_RETRY_BATCH_REQUEST_FILE,
        "batch_state_file": TEACHER_TEXT_MUTATION_RETRY_BATCH_STATE_FILE,
        "batch_output_file": TEACHER_TEXT_MUTATION_RETRY_BATCH_OUTPUT_FILE,
        "batch_error_file": TEACHER_TEXT_MUTATION_RETRY_BATCH_ERROR_FILE,
    },
}
existing_retry_policy = globals().get("TEACHER_TEXT_RETRY_POLICY") or {}
TEACHER_TEXT_RETRY_POLICY = {
    role: {**DEFAULT_TEACHER_TEXT_RETRY_POLICY[role], **existing_retry_policy.get(role, {})}
    for role in DEFAULT_TEACHER_TEXT_RETRY_POLICY
}
globals()["TEACHER_TEXT_RETRY_POLICY"] = TEACHER_TEXT_RETRY_POLICY

for role, cfg in TEACHER_TEXT_RETRY_POLICY.items():
    retry_manifest_file = cfg["retry_manifest_file"]
    retry_rows = sum(1 for _ in open(retry_manifest_file, encoding="utf-8") if _.strip()) if retry_manifest_file.exists() else 0
    print("\nrole:", role)
    print("  retry manifest:", retry_manifest_file)
    print("  retry rows:", f"{retry_rows:,}")
    if retry_rows == 0:
        print("  no missing rows remain for this role; skipping request preparation")
        continue
    cmd = [
        sys.executable,
        str(PREPARE_SEED_BATCH_SCRIPT),
        "--manifest-file", str(retry_manifest_file),
        "--source-file", str(ENRICHED_CORPUS),
        "--request-file", str(cfg["batch_request_file"]),
        "--existing-output-file", str(PRODUCTION_TEACHER_TEXT_SEEDS),
        "--model", cfg["model"],
        "--temperature", str(cfg["temperature"]),
        "--max-output-tokens", str(TEACHER_TEXT_RETRY_BASE_MAX_OUTPUT_TOKENS),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"teacher-text retry batch preparation completed for {role} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Teacher-text retry batch preparation failed for {role}.")


##### Step H-R3 — Create Paid Teacher-Text Retry Batch Jobs

Run this **only if `H-R2` prepared non-empty paid retry request files**.

This should create at most two very small paid retry batch jobs:
- one for the remaining `validation_diagnosis` rows,
- one for the remaining `mutation_robustness` rows.

Restart-safe behavior:
- if a retry state file already contains a batch id, this cell reports the saved job instead of silently creating a duplicate one
- delete the retry state file only if you explicitly want to recreate that retry batch from scratch

If you hit a billing-limit error here, that is an account constraint, not a notebook failure. In that case stop here and preserve the current artifact.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)
DEFAULT_TEACHER_TEXT_RETRY_POLICY = {
    "validation_diagnosis": {
        "batch_request_file": BATCH_DIR / "teacher_text_validation_seed_retry_requests_v1.jsonl",
        "batch_state_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_v1.json",
        "batch_output_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_error_v1.jsonl",
    },
    "mutation_robustness": {
        "batch_request_file": BATCH_DIR / "teacher_text_mutation_seed_retry_requests_v1.jsonl",
        "batch_state_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_v1.json",
        "batch_output_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_error_v1.jsonl",
    },
}
existing_retry_policy = globals().get("TEACHER_TEXT_RETRY_POLICY") or {}
TEACHER_TEXT_RETRY_POLICY = {
    role: {**DEFAULT_TEACHER_TEXT_RETRY_POLICY[role], **existing_retry_policy.get(role, {})}
    for role in DEFAULT_TEACHER_TEXT_RETRY_POLICY
}
globals()["TEACHER_TEXT_RETRY_POLICY"] = TEACHER_TEXT_RETRY_POLICY

for role, cfg in TEACHER_TEXT_RETRY_POLICY.items():
    request_file = cfg["batch_request_file"]
    state_file = cfg["batch_state_file"]
    request_rows = sum(1 for _ in open(request_file, encoding="utf-8") if _.strip()) if request_file.exists() else 0
    print("\nrole:", role)
    print("  request file:", request_file)
    print("  request rows:", f"{request_rows:,}")
    if request_rows == 0:
        print("  no retry requests to submit for this role")
        continue
    if state_file.exists():
        state = json.loads(state_file.read_text(encoding="utf-8"))
        if state.get("batch_id"):
            print("  existing retry batch state detected")
            print("  batch id:", state.get("batch_id"))
            print("  batch status:", state.get("status"))
            continue
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(request_file),
        "--state-file", str(state_file),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"teacher-text retry batch creation completed for {role} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Teacher-text retry batch creation failed for {role}.")


##### Step H-R4 — Wait For Completion And Download Paid Retry Batch Files

Run this after `H-R3`.

This waits on the saved paid retry batch ids and downloads the retry output/error files for each role. These downloaded retry files are then used by `H-R5` to append only the still-missing rows back into the canonical teacher-text artifact.

If a role had `0` retry rows, it should simply report nothing to wait for.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py",
)
DEFAULT_TEACHER_TEXT_RETRY_POLICY = {
    "validation_diagnosis": {
        "batch_state_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_v1.json",
        "batch_output_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_error_v1.jsonl",
    },
    "mutation_robustness": {
        "batch_state_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_v1.json",
        "batch_output_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_error_v1.jsonl",
    },
}
existing_retry_policy = globals().get("TEACHER_TEXT_RETRY_POLICY") or {}
TEACHER_TEXT_RETRY_POLICY = {
    role: {**DEFAULT_TEACHER_TEXT_RETRY_POLICY[role], **existing_retry_policy.get(role, {})}
    for role in DEFAULT_TEACHER_TEXT_RETRY_POLICY
}
globals()["TEACHER_TEXT_RETRY_POLICY"] = TEACHER_TEXT_RETRY_POLICY

for role, cfg in TEACHER_TEXT_RETRY_POLICY.items():
    state_file = cfg["batch_state_file"]
    print("\nrole:", role)
    if not state_file.exists():
        print("  no retry batch state file yet")
        continue
    state = json.loads(state_file.read_text(encoding="utf-8"))
    batch_id = state.get("batch_id")
    if not batch_id:
        print("  no retry batch id recorded in the state file")
        continue
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", str(batch_id),
        "--state-file", str(state_file),
        "--wait",
        "--download-output-file", str(cfg["batch_output_file"]),
        "--download-error-file", str(cfg["batch_error_file"]),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.rstrip())
    print(f"teacher-text retry batch wait/download completed for {role} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Teacher-text retry batch wait/download failed for {role}.")


##### Step H-R5 — Materialize Paid Retry Outputs Into The Canonical Artifact

Run this after the paid retry wait/download step.

This cell is designed to be **rerun-safe**:
- it reloads the current teacher-text artifact,
- filters each retry manifest down to the rows that are still missing right now,
- filters the paid retry batch output/error files to only those pending custom ids,
- and then materializes only that still-missing subset.

That means you can rerun this cell without duplicating already materialized paid retry rows.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
ENRICHED_CORPUS = globals().get("ENRICHED_CORPUS", PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl")
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)
MATERIALIZE_SEED_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_SEED_BATCH_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/materialize_seed_drafts_quality_aware_batch.py",
)
DEFAULT_TEACHER_TEXT_RETRY_POLICY = {
    "validation_diagnosis": {
        "model": "gpt-5.4",
        "temperature": 0.1,
        "retry_manifest_file": PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_retry.jsonl",
        "batch_output_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_validation_seed_retry_batch_error_v1.jsonl",
    },
    "mutation_robustness": {
        "model": "gpt-5.4-mini",
        "temperature": 0.1,
        "retry_manifest_file": PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_retry.jsonl",
        "batch_output_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_mutation_seed_retry_batch_error_v1.jsonl",
    },
}
existing_retry_policy = globals().get("TEACHER_TEXT_RETRY_POLICY") or {}
TEACHER_TEXT_RETRY_POLICY = {
    role: {**DEFAULT_TEACHER_TEXT_RETRY_POLICY[role], **existing_retry_policy.get(role, {})}
    for role in DEFAULT_TEACHER_TEXT_RETRY_POLICY
}
globals()["TEACHER_TEXT_RETRY_POLICY"] = TEACHER_TEXT_RETRY_POLICY


def load_completed_keys(path: Path):
    completed = set()
    if not path.exists():
        return completed
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            meta = row.get('metadata', {})
            completed.add((meta.get('circuit_hash'), meta.get('seed_role')))
    return completed


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')


def iter_jsonl(path: Path):
    if not path.exists():
        return
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def custom_id_for_manifest_row(row: dict) -> str:
    return f"seed::{row['source_record']['circuit_hash']}::{row['seed_role']}"

completed_keys = load_completed_keys(PRODUCTION_TEACHER_TEXT_SEEDS)

for role, cfg in TEACHER_TEXT_RETRY_POLICY.items():
    retry_manifest_file = cfg['retry_manifest_file']
    batch_output_file = cfg['batch_output_file']
    batch_error_file = cfg['batch_error_file']
    print("\nrole:", role)
    if not retry_manifest_file.exists():
        print("  no retry manifest yet")
        continue
    retry_rows = [row for row in iter_jsonl(retry_manifest_file)]
    pending_rows = [
        row for row in retry_rows
        if (row['source_record']['circuit_hash'], row['seed_role']) not in completed_keys
    ]
    print("  retry manifest rows:", f"{len(retry_rows):,}")
    print("  still-missing rows before materialization:", f"{len(pending_rows):,}")
    if not pending_rows:
        print("  all rows from this retry manifest are already materialized; skipping")
        continue
    if not batch_output_file.exists() and not batch_error_file.exists():
        print("  no downloaded retry batch output or error files yet")
        continue

    pending_ids = {custom_id_for_manifest_row(row) for row in pending_rows}
    pending_manifest_file = retry_manifest_file.with_name(retry_manifest_file.stem + '_pending_materialization.jsonl')
    pending_output_file = batch_output_file.with_name(batch_output_file.stem + '_pending_materialization.jsonl')
    pending_error_file = batch_error_file.with_name(batch_error_file.stem + '_pending_materialization.jsonl')

    write_jsonl(pending_manifest_file, pending_rows)
    write_jsonl(
        pending_output_file,
        [row for row in iter_jsonl(batch_output_file) if row.get('custom_id') in pending_ids] if batch_output_file.exists() else [],
    )
    write_jsonl(
        pending_error_file,
        [row for row in iter_jsonl(batch_error_file) if row.get('custom_id') in pending_ids] if batch_error_file.exists() else [],
    )

    cmd = [
        sys.executable,
        str(MATERIALIZE_SEED_BATCH_SCRIPT),
        '--manifest-file', str(pending_manifest_file),
        '--source-file', str(ENRICHED_CORPUS),
        '--batch-output-file', str(pending_output_file),
        '--batch-error-file', str(pending_error_file),
        '--output-file', str(PRODUCTION_TEACHER_TEXT_SEEDS),
        '--log-file', str(PRODUCTION_TEACHER_TEXT_ERRORS),
        '--model', cfg['model'],
        '--temperature', str(cfg['temperature']),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print('stderr')
        print(result.stderr.rstrip())
    print(f"teacher-text retry batch materialization completed for {role} with return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Teacher-text retry materialization failed for {role}.")
    completed_keys = load_completed_keys(PRODUCTION_TEACHER_TEXT_SEEDS)


##### Step H-R6 — Re-Audit Teacher-Text Coverage After Paid Retry

Run this immediately after `H-R5`.

This is the closure check for the paid retry branch of `Stage H-Retry`.

Interpretation:
- if total missing rows = `0`, Stage H is fully closed and you can move to Stage I
- if missing rows remain, stop and inspect the paid retry error files before deciding whether to launch another tiny retry pass


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TEACHER_TEXT_VALIDATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_VALIDATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_validation_diagnosis.jsonl",
)
TEACHER_TEXT_MUTATION_MANIFEST_FILE = globals().get(
    "TEACHER_TEXT_MUTATION_MANIFEST_FILE",
    PROCESSED_DIR / "seed_role_manifest_v1_teacher_text_mutation_robustness.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_ERRORS = globals().get(
    "PRODUCTION_TEACHER_TEXT_ERRORS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1_errors.jsonl",
)

manifest_counts = {
    'validation_diagnosis': sum(1 for _ in open(TEACHER_TEXT_VALIDATION_MANIFEST_FILE, encoding='utf-8') if _.strip()) if TEACHER_TEXT_VALIDATION_MANIFEST_FILE.exists() else 0,
    'mutation_robustness': sum(1 for _ in open(TEACHER_TEXT_MUTATION_MANIFEST_FILE, encoding='utf-8') if _.strip()) if TEACHER_TEXT_MUTATION_MANIFEST_FILE.exists() else 0,
}
output_counts = Counter()
unique_hashes = set()
if PRODUCTION_TEACHER_TEXT_SEEDS.exists():
    with open(PRODUCTION_TEACHER_TEXT_SEEDS, encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            meta = row.get('metadata', {})
            role = meta.get('seed_role', '<missing>')
            output_counts[role] += 1
            unique_hashes.add(meta.get('circuit_hash', ''))
error_counts = Counter()
if PRODUCTION_TEACHER_TEXT_ERRORS.exists():
    with open(PRODUCTION_TEACHER_TEXT_ERRORS, encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            error_counts[row.get('seed_role', '<missing>')] += 1

missing_counts = {role: manifest_counts[role] - output_counts.get(role, 0) for role in manifest_counts}
total_missing = sum(missing_counts.values())

print('Teacher-text retry audit')
print('  total rows:', f"{sum(output_counts.values()):,}")
print('  unique circuit_hash values:', f"{len(unique_hashes):,}")
print('  combined manifest size:', f"{sum(manifest_counts.values()):,}")
print('  total missing rows:', f"{total_missing:,}")
print('\nby role')
for role in ['validation_diagnosis', 'mutation_robustness']:
    print(f"  {role}: materialized={output_counts.get(role, 0):,} expected={manifest_counts.get(role, 0):,} missing={missing_counts.get(role, 0):,} logged_errors={error_counts.get(role, 0):,}")

if total_missing == 0:
    print('\nTeacher-text retry status: CLOSED')
    print('Stage H is now fully covered. Move on to Stage I.')
else:
    print('\nTeacher-text retry status: STILL INCOMPLETE')
    print('Do not move to Stage I yet. Inspect the retry error files and decide whether to run one more tiny retry pass.')


## Stage I — Full-Corpus Seed Coverage Audit

This audit treats the two branches as one routed corpus again.

The target is simple:

- `source_code` seed coverage
- plus `teacher_text` seed coverage
- should equal the full routing manifest size (`91,719` under the current rebuild snapshot)

If it does not, the corpus is not yet fully covered, even if one branch has finished.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
MANIFEST_FILE = PROCESSED_DIR / "seed_role_manifest_v1.jsonl"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_TEACHER_TEXT_SEEDS = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"

manifest_total = sum(1 for line in open(MANIFEST_FILE, encoding="utf-8") if line.strip()) if MANIFEST_FILE.exists() else 0
source_total = sum(1 for line in open(PRODUCTION_SEED_DRAFTS, encoding="utf-8") if line.strip()) if PRODUCTION_SEED_DRAFTS.exists() else 0
teacher_text_total = sum(1 for line in open(PRODUCTION_TEACHER_TEXT_SEEDS, encoding="utf-8") if line.strip()) if PRODUCTION_TEACHER_TEXT_SEEDS.exists() else 0
combined_total = source_total + teacher_text_total
coverage_ratio = (combined_total / manifest_total) if manifest_total else 0.0

print("full routed manifest rows:", f"{manifest_total:,}")
print("source-code seed rows:", f"{source_total:,}")
print("teacher-text seed rows:", f"{teacher_text_total:,}")
print("combined generated seed rows:", f"{combined_total:,}")
print("coverage ratio:", f"{coverage_ratio:.4f}")
print("remaining rows:", f"{manifest_total - combined_total:,}")


## Stage J — Quality-Aware Paraphrase Generation Across Both Branches

This stage expands the generated quality-aware seeds into paraphrase variants **across the full routed corpus**.

Important methodological note:

- the **seed-draft** temperature is now frozen by the documented calibration ladder
- the **paraphrase** stage has not yet undergone an equivalent dedicated calibration study
- therefore the paraphrase model and temperature are exposed as explicit notebook parameters and should be reported whenever paraphrase outputs are used downstream

Why the current paraphrase defaults differ from the seed-draft defaults:

- seed drafting uses `gpt-5.4` because it must convert circuit code plus readiness-conditioned role metadata into a pedagogically aligned instruction
- paraphrasing is a narrower reformulation task over an already grounded seed, so the current operational model is `gpt-5.4-mini`
- paraphrasing also needs slightly more surface-form variation than seed drafting, which is why the current operational temperature is `0.2` rather than the seed-draft `0.1`
- this is a documented operational rationale, not yet a calibration-closed conclusion

Current documented operational policy:

- branch inputs:
  - audited quality-aware `source_code` seeds
  - audited quality-aware `teacher_text` seeds
- current default model: `gpt-5.4-mini`
- current paraphrase count: `5` per seed
- current output is lineage-preserving and keeps the original seed role metadata

Once a later critique/rewrite pass exists, this stage should point to the reviewed seed artifacts rather than to raw draft seeds.

For the full-corpus production run, prefer the adjacent **Batch API** section once the prompts and settings are frozen. Keep the synchronous run cell for spot checks, pilot reruns, and narrow recovery work.


In [ ]:
from openai import OpenAI
from pathlib import Path
import os

key_path = Path(os.environ["OPENAI_API_KEY_FILE"])
api_key = key_path.read_text(encoding="utf-8").strip()

client = OpenAI(api_key=api_key)

resp = client.responses.create(
    model="gpt-5.4-mini",
    input="Reply with exactly: ok"
)

print(resp.output_text)


In [ ]:
PARAPHRASE_MODEL = "gpt-5.4-mini"
PARAPHRASE_TEMPERATURE = 0.2
PARAPHRASE_COUNT = 5
PARAPHRASE_MAX_SEEDS = None
PARAPHRASE_DRY_RUN = False

PARAPHRASE_JOBS = [
    {
        "label": "source_code",
        "seed_file": PRODUCTION_SEED_DRAFTS if PRODUCTION_SEED_DRAFTS.exists() else SEED_DRAFTS,
        "output_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASES,
        "log_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS,
    },
    {
        "label": "teacher_text",
        "seed_file": PRODUCTION_TEACHER_TEXT_SEEDS,
        "output_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES,
        "log_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS,
    },
]

for job in PARAPHRASE_JOBS:
    print("\nparaphrase branch:", job["label"])
    print("  source file:", job["seed_file"])
    print("  output file:", job["output_file"])
    print("  model:", PARAPHRASE_MODEL)
    print("  temperature:", PARAPHRASE_TEMPERATURE)
    print("  paraphrases per seed:", PARAPHRASE_COUNT)

    if not Path(job["seed_file"]).exists():
        print("  skipping missing source seed file")
        continue

    cmd = [
        sys.executable,
        str(GENERATE_PARAPHRASES_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--output-file", str(job["output_file"]),
        "--log-file", str(job["log_file"]),
        "--model", PARAPHRASE_MODEL,
        "--temperature", str(PARAPHRASE_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_COUNT),
    ]

    if PARAPHRASE_MAX_SEEDS is not None:
        cmd.extend(["--max-seeds", str(PARAPHRASE_MAX_SEEDS)])

    if PARAPHRASE_DRY_RUN:
        cmd.append("--dry-run")

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  branch completed with return code:", result.returncode)

In [ ]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
QUALITY_AWARE_SOURCE_CODE_PARAPHRASES = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl"
QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl"
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl"
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_TEACHER_TEXT_SEEDS = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"


def normalize_prompt_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


branch_specs = [
    {
        "label": "source_code",
        "seed_file": PRODUCTION_SEED_DRAFTS,
        "paraphrase_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASES,
        "log_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS,
    },
    {
        "label": "teacher_text",
        "seed_file": PRODUCTION_TEACHER_TEXT_SEEDS,
        "paraphrase_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES,
        "log_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS,
    },
]

all_rows = 0
all_seed_rows = 0
for spec in branch_specs:
    print(f"\nBranch: {spec['label']}")
    if not spec["paraphrase_file"].exists():
        print("  No quality-aware paraphrase file yet.")
        if spec["log_file"].exists():
            print("  Recent error-log preview:")
            with open(spec["log_file"], encoding="utf-8") as f:
                for i, line in enumerate(f):
                    if i >= 3:
                        break
                    row = json.loads(line)
                    print("   ", json.dumps({
                        "error_type": row.get("error_type"),
                        "error_message": row.get("error_message"),
                        "seed_role": row.get("seed_role"),
                    }, ensure_ascii=False))
        continue

    role_counts = Counter()
    prompt_type_counts = Counter()
    source_counts = Counter()
    normalized_duplicates = Counter()
    grouped_examples = defaultdict(list)

    with open(spec["paraphrase_file"], encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            text = row.get("input", "")
            meta = row.get("metadata", {})
            role = meta.get("seed_role", "<missing>")
            source_id = meta.get("paraphrase_source_content_hash") or meta.get("paraphrase_source") or meta.get("circuit_hash")
            role_counts[role] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            source_counts[source_id] += 1
            normalized_duplicates[normalize_prompt_text(text)] += 1
            if source_id and len(grouped_examples[source_id]) < 2:
                grouped_examples[source_id].append(text)

    seed_rows = sum(1 for line in open(spec["seed_file"], encoding="utf-8") if line.strip()) if spec["seed_file"].exists() else 0
    row_count = sum(source_counts.values())
    all_rows += row_count
    all_seed_rows += seed_rows
    duplicate_count = sum(1 for value in normalized_duplicates.values() if value > 1)
    source_distribution = Counter(source_counts.values())

    print("  paraphrase rows:", f"{row_count:,}")
    print("  source seed rows:", f"{seed_rows:,}")
    print("  exact normalized duplicates:", duplicate_count)
    print("  role counts:")
    for key, value in role_counts.most_common():
        print(f"    {key}: {value:,}")
    print("  prompt types:")
    for key, value in prompt_type_counts.most_common():
        print(f"    {key}: {value:,}")
    print("  paraphrase coverage by source seed:")
    for key, value in sorted(source_distribution.items()):
        print(f"    {key} paraphrases: {value:,} source seeds")

print("\nFull-corpus paraphrase totals")
print("  branch seed rows:", f"{all_seed_rows:,}")
print("  branch paraphrase rows:", f"{all_rows:,}")


### Stage J-Batch — Batch Execution for Quality-Aware Paraphrases

For full production paraphrase expansion, Batch is the preferred path. It preserves the same paraphrase prompt contract while cutting cost for the largest-volume stage after teacher-text seed generation. The branch split remains explicit so source-code and teacher-text paraphrases can be tracked and audited separately.

To preserve notebook reproducibility, the fixed-purpose create/wait cells below can be used instead of editing the manual submit cell in place.

Visible run order in this notebook section:

1. **Prepare branch request files**
2. **Optional manual submit cell**
3. **Fixed-path create cell**
4. **Fixed-path wait/download cell**
5. **Materialize standard PQID output**


#### How To Use Stage J-Batch

This section sits **immediately below** `### Stage J-Batch — Batch Execution for Quality-Aware Paraphrases`.

Recommended fixed-path order:

1. **Prepare branch request files**
2. **Create both branch batch jobs**
3. **Wait for completion and download files for both branches**
4. **Materialize the normal PQID paraphrase files**
5. **Run the local audit cell directly below**


##### Step J-B1 — Prepare Paraphrase Batch Request Files

Run this first. It prepares separate batch request JSONL files for the `source_code` paraphrases and the `teacher_text` paraphrases.


In [ ]:
PARAPHRASE_BATCH_MODEL = "gpt-5.4-mini"
PARAPHRASE_BATCH_TEMPERATURE = 0.2
PARAPHRASE_BATCH_COUNT = 5
PARAPHRASE_BATCH_MAX_SEEDS = None

PARAPHRASE_BATCH_JOBS = [
    {
        "label": "source_code",
        "seed_file": PRODUCTION_SEED_DRAFTS if PRODUCTION_SEED_DRAFTS.exists() else SEED_DRAFTS,
        "request_file": SOURCE_CODE_PARAPHRASE_BATCH_REQUEST_FILE,
        "existing_output_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASES,
    },
    {
        "label": "teacher_text",
        "seed_file": PRODUCTION_TEACHER_TEXT_SEEDS,
        "request_file": TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE,
        "existing_output_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES,
    },
]

for job in PARAPHRASE_BATCH_JOBS:
    print("\npreparing paraphrase batch branch:", job["label"])
    if not Path(job["seed_file"]).exists():
        print("  skipping missing seed file")
        continue

    cmd = [
        sys.executable,
        str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--request-file", str(job["request_file"]),
        "--existing-output-file", str(job["existing_output_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
    ]

    if PARAPHRASE_BATCH_MAX_SEEDS is not None:
        cmd.extend(["--max-seeds", str(PARAPHRASE_BATCH_MAX_SEEDS)])

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  branch preparation completed with return code:", result.returncode)

##### Optional Step J-B2a — Manual Submit / Inspect Cell

This is the original flexible control cell for paraphrase batches. The fixed-path create and wait cells below are easier to follow and reproduce.


In [ ]:
PARAPHRASE_BATCH_CREATE = False
PARAPHRASE_BATCH_WAIT = False
PARAPHRASE_BATCH_IDS = {
    "source_code": "",
    "teacher_text": "",
}

PARAPHRASE_BATCH_SUBMIT_JOBS = [
    {
        "label": "source_code",
        "request_file": SOURCE_CODE_PARAPHRASE_BATCH_REQUEST_FILE,
        "state_file": SOURCE_CODE_PARAPHRASE_BATCH_STATE_FILE,
        "output_file": SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE,
        "error_file": SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE,
    },
    {
        "label": "teacher_text",
        "request_file": TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE,
        "state_file": TEACHER_TEXT_PARAPHRASE_BATCH_STATE_FILE,
        "output_file": TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE,
        "error_file": TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE,
    },
]

for job in PARAPHRASE_BATCH_SUBMIT_JOBS:
    batch_id = PARAPHRASE_BATCH_IDS.get(job["label"], "")
    if not PARAPHRASE_BATCH_CREATE and not batch_id:
        print("\nparaphrase branch:", job["label"])
        print("  set PARAPHRASE_BATCH_CREATE = True or provide PARAPHRASE_BATCH_IDS[label] to continue")
        continue

    cmd = [sys.executable, str(RUN_BATCH_JOB_SCRIPT)]
    if PARAPHRASE_BATCH_CREATE:
        cmd.extend([
            "--request-file", str(job["request_file"]),
            "--state-file", str(job["state_file"]),
        ])
    else:
        cmd.extend(["--batch-id", batch_id])

    if PARAPHRASE_BATCH_WAIT:
        cmd.append("--wait")

    cmd.extend([
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ])

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("\nparaphrase branch completed:", job["label"], "return code:", result.returncode)

#### Fixed-Path Paraphrase Batch Continuation

These cells create or wait on the paraphrase branch batch jobs using the saved state files, so the notebook does not depend on manual boolean toggling or copied batch IDs.


##### Step J-B2 — Create Paraphrase Batch Jobs

Run this after preparation. It creates the `source_code` and `teacher_text` paraphrase batch jobs and stores their batch ids in the corresponding state files.


In [ ]:
PARAPHRASE_BATCH_CREATE_JOBS = [
    {
        "label": "source_code",
        "request_file": SOURCE_CODE_PARAPHRASE_BATCH_REQUEST_FILE,
        "state_file": SOURCE_CODE_PARAPHRASE_BATCH_STATE_FILE,
        "output_file": SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE,
        "error_file": SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE,
    },
    {
        "label": "teacher_text",
        "request_file": TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE,
        "state_file": TEACHER_TEXT_PARAPHRASE_BATCH_STATE_FILE,
        "output_file": TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE,
        "error_file": TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE,
    },
]

for job in PARAPHRASE_BATCH_CREATE_JOBS:
    print("\ncreating paraphrase batch branch:", job["label"])
    if not Path(job["request_file"]).exists():
        print("  request file missing; run the paraphrase batch prepare cell first")
        continue
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(job["request_file"]),
        "--state-file", str(job["state_file"]),
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  branch creation completed with return code:", result.returncode)
    if Path(job["state_file"]).exists():
        state = json.loads(Path(job["state_file"]).read_text(encoding="utf-8"))
        print("  batch id:", state.get("batch_id"))
        print("  batch status:", state.get("status"))


## Stage J-BR — Teacher-Text Paraphrase Oversized Batch Recovery

Use this block **only if the `teacher_text` paraphrase branch cannot be submitted cleanly through Step J-B2**. The current recovery path is designed for the three practical batch limits that can appear here: input-file size, maximum requests per batch, and organization-level enqueued-token caps.

The `teacher_text` paraphrase request file is large enough that a pure file-size split is not sufficient. This recovery block mirrors Stage H-BR, but with stricter operational controls: it shards the oversized file, caps rows per shard so no shard exceeds the Batch API request-count limit, runs shard jobs serially so the org enqueue cap is not exhausted, and then merges the shard outputs back into the canonical `teacher_text_paraphrase_batch_output_v1.jsonl` and `teacher_text_paraphrase_batch_error_v1.jsonl` filenames expected by Step J-B4.

After J-BR4 completes, skip Step J-B3 for the `teacher_text` branch (it will self-skip if no state file is present) and proceed directly to Step J-B4.

Important mixed-path note: `J-B3` still belongs to the normal path for the `source_code` paraphrase branch. If `source_code` was created successfully in `J-B2` but `teacher_text` needs recovery, the practical execution order is: run `J-B3` once to finish `source_code`, then run `J-BR1` to `J-BR4` for `teacher_text`, then continue with `J-B4` and `J-B5`.

##### Step J-BR1 — Shard The Oversized Teacher-Text Paraphrase Request File

Run this first. It splits `teacher_text_paraphrase_requests_v1.jsonl` into deterministic shards that satisfy both a byte cap and a row-count cap. The resulting shard index file is then consumed by the serialized recovery steps below.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE", BATCH_DIR / "teacher_text_paraphrase_requests_v1.jsonl")
SHARD_BATCH_REQUEST_SCRIPT = globals().get("SHARD_BATCH_REQUEST_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/shard_openai_batch_request_file.py")
TEACHER_TEXT_PARAPHRASE_SHARD_REQUEST_PREFIX = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_REQUEST_PREFIX", BATCH_DIR / "teacher_text_paraphrase_requests_v1_shard")
TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_shards_v1.json")
TEACHER_TEXT_PARAPHRASE_SHARD_MAX_BYTES = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_MAX_BYTES", 150 * 1024 * 1024)
TEACHER_TEXT_PARAPHRASE_SHARD_MAX_ROWS = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_MAX_ROWS", 1200)

cmd = [
    sys.executable,
    str(SHARD_BATCH_REQUEST_SCRIPT),
    "--input-file", str(TEACHER_TEXT_PARAPHRASE_BATCH_REQUEST_FILE),
    "--request-prefix", str(TEACHER_TEXT_PARAPHRASE_SHARD_REQUEST_PREFIX),
    "--index-file", str(TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE),
    "--max-bytes", str(TEACHER_TEXT_PARAPHRASE_SHARD_MAX_BYTES),
    "--max-rows", str(TEACHER_TEXT_PARAPHRASE_SHARD_MAX_ROWS),
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.rstrip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.rstrip())
print("teacher-text paraphrase request sharding completed with return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Teacher-text paraphrase request sharding failed.")


##### Step J-BR2 — Run Teacher-Text Paraphrase Shards Serially

Run this after J-BR1. It processes the teacher-text paraphrase shards **one at a time**: create batch, wait for completion, download outputs, then move to the next shard. This keeps the recovery path below the org enqueue cap and makes the whole step restart-safe.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py")
TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_shards_v1.json")

shard_payload = json.loads(TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE.read_text(encoding="utf-8"))
for shard in shard_payload.get("shards", []):
    state_file = Path(shard["state_file"])
    output_file = Path(shard["output_file"])
    error_file = Path(shard["error_file"])
    part_label = f"part{shard['part']:03d}"
    batch_id = None
    if state_file.exists():
        state = json.loads(state_file.read_text(encoding="utf-8"))
        if state.get("status") == "completed" and output_file.exists():
            print(f"{part_label}: already completed; skipping")
            continue
        if state.get("status") in {"failed", "cancelled", "expired"}:
            state_file.unlink()
            if output_file.exists():
                output_file.unlink()
            if error_file.exists():
                error_file.unlink()
        else:
            batch_id = state.get("batch_id")
            if batch_id:
                print(f"\nteacher-text paraphrase shard {part_label} REUSE")
                print("existing batch id:", batch_id)
    if not batch_id:
        print(f"\nteacher-text paraphrase shard {part_label} CREATE")
        create_cmd = [
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--request-file", shard["request_file"],
            "--state-file", shard["state_file"],
            "--download-output-file", shard["output_file"],
            "--download-error-file", shard["error_file"],
        ]
        create_result = subprocess.run(create_cmd, capture_output=True, text=True)
        if create_result.stdout.strip():
            print(create_result.stdout.rstrip())
        if create_result.stderr.strip():
            print("stderr")
            print(create_result.stderr.rstrip())
        print(f"teacher-text paraphrase shard batch creation completed for {part_label} with return code:", create_result.returncode)
        if create_result.returncode != 0:
            raise RuntimeError(f"Teacher-text paraphrase shard batch creation failed for {part_label}.")
        state = json.loads(state_file.read_text(encoding="utf-8"))
        batch_id = state.get("batch_id")
        if not batch_id:
            raise RuntimeError(f"Missing batch_id after creation for {part_label}.")
    print(f"\nteacher-text paraphrase shard {part_label} WAIT")
    wait_cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", shard["state_file"],
        "--wait",
        "--download-output-file", shard["output_file"],
        "--download-error-file", shard["error_file"],
    ]
    wait_result = subprocess.run(wait_cmd, capture_output=True, text=True)
    if wait_result.stdout.strip():
        print(wait_result.stdout.rstrip())
    if wait_result.stderr.strip():
        print("stderr")
        print(wait_result.stderr.rstrip())
    print(f"teacher-text paraphrase shard batch wait/download completed for {part_label} with return code:", wait_result.returncode)
    if wait_result.returncode != 0:
        raise RuntimeError(f"Teacher-text paraphrase shard batch wait/download failed for {part_label}.")
    final_state = json.loads(state_file.read_text(encoding="utf-8"))
    if final_state.get("status") != "completed":
        raise RuntimeError(f"Teacher-text paraphrase shard {part_label} ended in status={final_state.get('status')}")


##### Step J-BR3 — Verify Teacher-Text Paraphrase Shard Completion Before Merge

Run this after J-BR2. It audits the shard state files and confirms that every shard finished successfully and produced a local output file before you merge anything.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
RUN_BATCH_JOB_SCRIPT = globals().get("RUN_BATCH_JOB_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/run_openai_batch_job.py")
TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_shards_v1.json")

shard_payload = json.loads(TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE.read_text(encoding="utf-8"))
problems = []
for shard in shard_payload.get("shards", []):
    part_label = f"part{shard['part']:03d}"
    state_file = Path(shard["state_file"])
    output_file = Path(shard["output_file"])
    print(f"\nteacher-text paraphrase shard {part_label}")
    if not state_file.exists():
        problems.append(f"{part_label}: missing state file")
        print("  status: missing state file")
        continue
    state = json.loads(state_file.read_text(encoding="utf-8"))
    print("  batch id:", state.get("batch_id"))
    print("  status:", state.get("status"))
    print("  output file present:", output_file.exists())
    if state.get("status") != "completed":
        problems.append(f"{part_label}: status={state.get('status')}")
    if state.get("status") == "completed" and not output_file.exists():
        problems.append(f"{part_label}: completed but output file missing")
if problems:
    raise RuntimeError("Teacher-text paraphrase shard verification failed: " + "; ".join(problems))
print("\nTeacher-text paraphrase shard verification: all shards completed and ready to merge.")


##### Step J-BR4 — Merge Teacher-Text Paraphrase Shard Outputs Into The Canonical Batch Files

Run this after J-BR3. It concatenates all shard output and error files into `teacher_text_paraphrase_batch_output_v1.jsonl` and `teacher_text_paraphrase_batch_error_v1.jsonl`. After this step, proceed to Step J-B4 to materialize the paraphrase artifact.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
MERGE_BATCH_SHARDS_SCRIPT = globals().get("MERGE_BATCH_SHARDS_SCRIPT", ROOT / "PQID/scripts/03_instruction_generation/merge_openai_batch_shard_outputs.py")
TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_shards_v1.json")
TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_output_v1.jsonl")
TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_error_v1.jsonl")

cmd = [
    sys.executable,
    str(MERGE_BATCH_SHARDS_SCRIPT),
    "--index-file", str(TEACHER_TEXT_PARAPHRASE_SHARD_INDEX_FILE),
    "--merged-output-file", str(TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE),
    "--merged-error-file", str(TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE),
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.rstrip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.rstrip())
print("teacher-text paraphrase shard merge completed with return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Teacher-text paraphrase shard merge failed.")


##### Step J-B3 — Wait For Completion And Download Paraphrase Batch Files

Run this after the create step. It waits on both paraphrase batch jobs and downloads their output and error files.

If the `teacher_text` branch had to move into `Stage J-BR`, this step still matters for the normal `source_code` branch. In that mixed case, run `J-B3` first to finish `source_code`, then use `J-BR1` to `J-BR4` only for `teacher_text`.

In [ ]:
PARAPHRASE_BATCH_WAIT_JOBS = [
    {
        "label": "source_code",
        "state_file": SOURCE_CODE_PARAPHRASE_BATCH_STATE_FILE,
        "output_file": SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE,
        "error_file": SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE,
    },
    {
        "label": "teacher_text",
        "state_file": TEACHER_TEXT_PARAPHRASE_BATCH_STATE_FILE,
        "output_file": TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE,
        "error_file": TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE,
    },
]

for job in PARAPHRASE_BATCH_WAIT_JOBS:
    print("\nwaiting on paraphrase batch branch:", job["label"])
    if not Path(job["state_file"]).exists():
        print("  state file missing; run the paraphrase batch create cell first")
        continue
    state = json.loads(Path(job["state_file"]).read_text(encoding="utf-8"))
    batch_id = state.get("batch_id", "")
    if not batch_id:
        print("  no batch id recorded yet")
        continue
    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", str(job["state_file"]),
        "--wait",
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  branch wait/download completed with return code:", result.returncode)

##### Step J-B4 — Materialize Standard Paraphrase Artifacts

Run this after the wait/download step. It converts the downloaded paraphrase batch results back into the standard PQID paraphrase JSONL files and logs.


##### Audit Note — Session Restore For Step J-B4

This additive support block exists only to restore notebook variables required by `Step J-B4 — Materialize Standard Paraphrase Artifacts` when the kernel/session has lost earlier state.

Why this block is needed:
- in the mixed-path Stage J recovery flow, `teacher_text` paraphrases are recovered through `J-BR1` to `J-BR4`, while `source_code` can still pass through the normal path
- if the notebook kernel is interrupted or cells are rerun out of order, `J-B4` can fail with `NameError` for variables such as `PARAPHRASE_BATCH_MODEL`, `PARAPHRASE_BATCH_TEMPERATURE`, and `PARAPHRASE_BATCH_COUNT`
- this block restores those values using the documented Stage J defaults and the canonical repo paths

What this block does:
- rehydrates the batch-materialization constants and file-path variables expected by `J-B4`
- uses the same documented Stage J paraphrase settings:
  - model: `gpt-5.4-mini`
  - temperature: `0.2`
  - paraphrases per seed: `5`

What this block does **not** do:
- it does **not** call the API
- it does **not** materialize paraphrases
- it does **not** modify any dataset artifact
- it does **not** change the recovery logic or the publication-facing semantics of Stage J

When to run:
- run this only if `J-B4` fails because required notebook variables are missing after a session reset or non-linear rerun


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)

SEED_DRAFTS = globals().get(
    "SEED_DRAFTS",
    PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
)
PRODUCTION_SEED_DRAFTS = globals().get(
    "PRODUCTION_SEED_DRAFTS",
    PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
)
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get(
    "PRODUCTION_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)

SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE = globals().get(
    "SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE",
    BATCH_DIR / "source_code_paraphrase_batch_output_v1.jsonl",
)
SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE = globals().get(
    "SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE",
    BATCH_DIR / "source_code_paraphrase_batch_error_v1.jsonl",
)
TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE = globals().get(
    "TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE",
    BATCH_DIR / "teacher_text_paraphrase_batch_output_v1.jsonl",
)
TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE = globals().get(
    "TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE",
    BATCH_DIR / "teacher_text_paraphrase_batch_error_v1.jsonl",
)

QUALITY_AWARE_SOURCE_CODE_PARAPHRASES = globals().get(
    "QUALITY_AWARE_SOURCE_CODE_PARAPHRASES",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
)
QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS = globals().get(
    "QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl",
)
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES = globals().get(
    "QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
)
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS = globals().get(
    "QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl",
)

MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py",
)

print("PARAPHRASE_BATCH_MODEL =", PARAPHRASE_BATCH_MODEL)
print("PARAPHRASE_BATCH_TEMPERATURE =", PARAPHRASE_BATCH_TEMPERATURE)
print("PARAPHRASE_BATCH_COUNT =", PARAPHRASE_BATCH_COUNT)
print("MATERIALIZE_PARAPHRASE_BATCH_SCRIPT exists:", MATERIALIZE_PARAPHRASE_BATCH_SCRIPT.exists())


##### Audit Note — Optional Error-Log Reset Before Canonical J-B4 Rerun

This optional support block clears only the paraphrase materialization error logs before rerunning `Step J-B4 — Materialize Standard Paraphrase Artifacts`.

Why this block may be needed:
- the materialization script appends to the existing error log files
- if `J-B4` is rerun after a partial or diagnostic pass, the error logs can contain duplicated entries from repeated materialization attempts
- for publication and audit clarity, it is preferable to regenerate the error logs cleanly from a single canonical frontend rerun

What this block does:
- deletes only:
  - `seed_paraphrases_quality_aware_source_code_v1_errors.jsonl`
  - `seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl`

What this block does **not** do:
- it does **not** delete the paraphrase artifacts themselves
- it does **not** alter any batch outputs
- it does **not** affect successful materialized paraphrase rows
- it does **not** call the API

When to run:
- run this immediately before a clean frontend rerun of `J-B4` if the error logs have become append-duplicated through repeated materialization attempts


In [ ]:
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

for path in [
    PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl",
]:
    if path.exists():
        path.unlink()
        print("deleted:", path)
    else:
        print("missing:", path)


In [ ]:
ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")
SEED_DRAFTS = globals().get("SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_SEED_DRAFTS = globals().get("PRODUCTION_SEED_DRAFTS", PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl")
PRODUCTION_TEACHER_TEXT_SEEDS = globals().get("PRODUCTION_TEACHER_TEXT_SEEDS", PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl")
SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE = globals().get("SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE", BATCH_DIR / "source_code_paraphrase_batch_output_v1.jsonl")
SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE = globals().get("SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE", BATCH_DIR / "source_code_paraphrase_batch_error_v1.jsonl")
TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_output_v1.jsonl")
TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE = globals().get("TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE", BATCH_DIR / "teacher_text_paraphrase_batch_error_v1.jsonl")
QUALITY_AWARE_SOURCE_CODE_PARAPHRASES = globals().get("QUALITY_AWARE_SOURCE_CODE_PARAPHRASES", PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl")
QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS = globals().get("QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS", PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl")
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES = globals().get("QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES", PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl")
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS = globals().get("QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS", PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl")
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get("MATERIALIZE_PARAPHRASE_BATCH_SCRIPT", SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py")
PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)

PARAPHRASE_BATCH_MATERIALIZE_JOBS = [
    {
        "label": "source_code",
        "seed_file": PRODUCTION_SEED_DRAFTS if PRODUCTION_SEED_DRAFTS.exists() else SEED_DRAFTS,
        "batch_output_file": SOURCE_CODE_PARAPHRASE_BATCH_OUTPUT_FILE,
        "batch_error_file": SOURCE_CODE_PARAPHRASE_BATCH_ERROR_FILE,
        "output_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASES,
        "log_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS,
    },
    {
        "label": "teacher_text",
        "seed_file": PRODUCTION_TEACHER_TEXT_SEEDS,
        "batch_output_file": TEACHER_TEXT_PARAPHRASE_BATCH_OUTPUT_FILE,
        "batch_error_file": TEACHER_TEXT_PARAPHRASE_BATCH_ERROR_FILE,
        "output_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES,
        "log_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS,
    },
]

for job in PARAPHRASE_BATCH_MATERIALIZE_JOBS:
    print("\nmaterializing paraphrase branch:", job["label"])
    if not Path(job["seed_file"]).exists():
        print("  skipping missing seed file")
        continue
    if not Path(job["batch_output_file"]).exists():
        print("  no downloaded batch output yet")
        continue

    cmd = [
        sys.executable,
        str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--batch-output-file", str(job["batch_output_file"]),
        "--batch-error-file", str(job["batch_error_file"]),
        "--output-file", str(job["output_file"]),
        "--log-file", str(job["log_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  branch materialization completed with return code:", result.returncode)

##### Step J-B5 — Audit Paraphrase Artifacts

Run this immediately after materialization. This is a local duplicate of the Stage J paraphrase audit so the paraphrase batch section is complete on its own.


In [ ]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]
PROCESSED_DIR = ROOT / "PQID/data/processed"
QUALITY_AWARE_SOURCE_CODE_PARAPHRASES = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl"
QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl"
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl"
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl"
PRODUCTION_SEED_DRAFTS = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
PRODUCTION_TEACHER_TEXT_SEEDS = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"


def normalize_prompt_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


branch_specs = [
    {
        "label": "source_code",
        "seed_file": PRODUCTION_SEED_DRAFTS,
        "paraphrase_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASES,
        "log_file": QUALITY_AWARE_SOURCE_CODE_PARAPHRASE_ERRORS,
    },
    {
        "label": "teacher_text",
        "seed_file": PRODUCTION_TEACHER_TEXT_SEEDS,
        "paraphrase_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES,
        "log_file": QUALITY_AWARE_TEACHER_TEXT_PARAPHRASE_ERRORS,
    },
]

all_rows = 0
all_seed_rows = 0
for spec in branch_specs:
    print(f"\nBranch: {spec['label']}")
    if not spec["paraphrase_file"].exists():
        print("  No quality-aware paraphrase file yet.")
        if spec["log_file"].exists():
            print("  Recent error-log preview:")
            with open(spec["log_file"], encoding="utf-8") as f:
                for i, line in enumerate(f):
                    if i >= 3:
                        break
                    row = json.loads(line)
                    print("   ", json.dumps({
                        "error_type": row.get("error_type"),
                        "error_message": row.get("error_message"),
                        "seed_role": row.get("seed_role"),
                    }, ensure_ascii=False))
        continue

    role_counts = Counter()
    prompt_type_counts = Counter()
    source_counts = Counter()
    normalized_duplicates = Counter()
    grouped_examples = defaultdict(list)

    with open(spec["paraphrase_file"], encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            text = row.get("input", "")
            meta = row.get("metadata", {})
            role = meta.get("seed_role", "<missing>")
            source_id = meta.get("paraphrase_source_content_hash") or meta.get("paraphrase_source") or meta.get("circuit_hash")
            role_counts[role] += 1
            prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
            source_counts[source_id] += 1
            normalized_duplicates[normalize_prompt_text(text)] += 1
            if source_id and len(grouped_examples[source_id]) < 2:
                grouped_examples[source_id].append(text)

    seed_rows = sum(1 for line in open(spec["seed_file"], encoding="utf-8") if line.strip()) if spec["seed_file"].exists() else 0
    row_count = sum(source_counts.values())
    all_rows += row_count
    all_seed_rows += seed_rows
    duplicate_count = sum(1 for value in normalized_duplicates.values() if value > 1)
    source_distribution = Counter(source_counts.values())

    print("  paraphrase rows:", f"{row_count:,}")
    print("  source seed rows:", f"{seed_rows:,}")
    print("  exact normalized duplicates:", duplicate_count)
    print("  role counts:")
    for key, value in role_counts.most_common():
        print(f"    {key}: {value:,}")
    print("  prompt types:")
    for key, value in prompt_type_counts.most_common():
        print(f"    {key}: {value:,}")
    print("  paraphrase coverage by source seed:")
    for key, value in sorted(source_distribution.items()):
        print(f"    {key} paraphrases: {value:,} source seeds")

print("\nFull-corpus paraphrase totals")
print("  branch seed rows:", f"{all_seed_rows:,}")
print("  branch paraphrase rows:", f"{all_rows:,}")


#### Stage J-Retry — Residual Missing-Paraphrase Recovery

This stage closes only the residual source seeds that remain below the expected `5` materialized paraphrases after the canonical `J-B4` run and the `J-B5` audit.

Use this stage only if `J-B5` or the retry-manifest step below shows a residual shortfall. It keeps the recovery path publication-clean by:
- rebuilding the retry set from the live canonical paraphrase artifacts
- preparing a dedicated retry transport for only the still-missing seeds
- materializing successful retry outputs back into the same canonical artifacts
- preserving a snapshot of the pre-retry canonical error logs before the residual materialization pass

This stage does **not** rerun the full paraphrase generation flow. It only targets the residual gap left after the canonical Stage J batch merge and materialization path.


##### Step J-R1 — Build Residual Paraphrase Retry Manifests

This step identifies source seeds that still have fewer than the expected `5` materialized paraphrases after the canonical `J-B4` run, and writes retry manifests for those residual cases.

What this step does:
- compares each source seed file against the corresponding canonical paraphrase artifact
- counts how many paraphrases were successfully materialized per source seed
- selects only seeds with fewer than `5` materialized paraphrases
- writes dedicated retry manifests:
  - `seed_paraphrase_retry_missing_source_code_v1.jsonl`
  - `seed_paraphrase_retry_missing_teacher_text_v1.jsonl`

What this step does **not** do:
- it does **not** call the API
- it does **not** modify the canonical paraphrase artifacts
- it does **not** perform the retry itself

Run this step again after another retry pass if you need to confirm whether any residual shortfall remains.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

SOURCE_SEED_FILE = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
SOURCE_PARAPHRASE_FILE = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl"
SOURCE_RETRY_FILE = PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl"

TEACHER_SEED_FILE = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"
TEACHER_PARAPHRASE_FILE = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl"
TEACHER_RETRY_FILE = PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl"

EXPECTED_PARAPHRASES = 5


def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("content_hash") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("paraphrase_source_content_hash") or "").strip()
        or str(meta.get("paraphrase_source") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def build_retry(seed_file: Path, paraphrase_file: Path, retry_file: Path, label: str):
    seed_rows = load_jsonl(seed_file)
    paraphrase_rows = load_jsonl(paraphrase_file) if paraphrase_file.exists() else []

    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    seen = Counter()
    for row in paraphrase_rows:
        key = paraphrase_source_key(row)
        if key:
            seen[key] += 1

    missing_rows = []
    missing_distribution = Counter()
    for key, row in seed_map.items():
        have = seen.get(key, 0)
        if have < EXPECTED_PARAPHRASES:
            deficit = EXPECTED_PARAPHRASES - have
            out = dict(row)
            meta = dict(out.get("metadata", {}))
            meta["paraphrase_retry_reason"] = "missing_materialized_paraphrases"
            meta["paraphrase_retry_existing_count"] = have
            meta["paraphrase_retry_expected_count"] = EXPECTED_PARAPHRASES
            meta["paraphrase_retry_missing_count"] = deficit
            out["metadata"] = meta
            missing_rows.append(out)
            missing_distribution[have] += 1

    with retry_file.open("w", encoding="utf-8") as f:
        for row in missing_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nBranch: {label}")
    print("  seed rows:", f"{len(seed_rows):,}")
    print("  paraphrase rows:", f"{len(paraphrase_rows):,}")
    print("  retry seed rows:", f"{len(missing_rows):,}")
    print("  retry file:", retry_file)
    print("  existing paraphrase coverage among retry seeds:")
    for have, count in sorted(missing_distribution.items()):
        print(f"    {have} existing: {count:,} seeds")


build_retry(SOURCE_SEED_FILE, SOURCE_PARAPHRASE_FILE, SOURCE_RETRY_FILE, "source_code")
build_retry(TEACHER_SEED_FILE, TEACHER_PARAPHRASE_FILE, TEACHER_RETRY_FILE, "teacher_text")


##### Step J-R2 — Prepare Residual Retry Batch Request Files

Run this after `J-R1`. It builds dedicated retry request files for the still-missing seeds only.

What this step does:
- consumes the retry manifests written by `J-R1`
- uses the current canonical paraphrase artifacts as the deduplication baseline
- writes new retry batch request files under `PQID/data/processed/openai_batch_jobs`
- skips any seed that is already closed by the time this step runs

This step does **not** submit any API job. It only prepares the retry request transport.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

PREPARE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "PREPARE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "prepare_paraphrases_quality_aware_batch.py",
)

PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)

RETRY_PARAPHRASE_BATCH_JOBS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "request_file": BATCH_DIR / "source_code_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_v1.json",
        "output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "existing_output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "request_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_v1.json",
        "output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "existing_output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
    },
]

globals()["RETRY_PARAPHRASE_BATCH_JOBS"] = RETRY_PARAPHRASE_BATCH_JOBS

for job in RETRY_PARAPHRASE_BATCH_JOBS:
    print("preparing retry paraphrase batch branch:", job["label"])
    if not job["seed_file"].exists():
        print("  missing retry manifest; run Step J-R1 first")
        continue

    for stale_path in [job["request_file"], job["state_file"], job["output_file"], job["error_file"]]:
        if stale_path.exists():
            stale_path.unlink()

    cmd = [
        sys.executable,
        str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--request-file", str(job["request_file"]),
        "--existing-output-file", str(job["existing_output_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  retry branch preparation completed with return code:", result.returncode)


##### Step J-R3 — Create Residual Retry Batch Jobs

Run this after `J-R2`. It submits the retry request files as dedicated OpenAI batch jobs and writes one state file per branch.

What this step does:
- reads the retry request files from `J-R2`
- creates one batch job for `source_code` and one for `teacher_text`
- stores the resulting batch metadata in branch-specific state files for later waiting and download

This step submits jobs, but it does **not** wait for completion.


In [ ]:
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    SCRIPTS_DIR / "run_openai_batch_job.py",
)

for job in RETRY_PARAPHRASE_BATCH_JOBS:
    print("creating retry paraphrase batch branch:", job["label"])
    if not job["request_file"].exists():
        print("  request file missing; run Step J-R2 first")
        continue

    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(job["request_file"]),
        "--state-file", str(job["state_file"]),
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  retry branch create completed with return code:", result.returncode)


##### Step J-R4 — Wait For Completion And Download Residual Retry Batch Files

Run this after `J-R3`. It waits for each retry batch job to finish and downloads the resulting output and error files.

What this step does:
- reads the saved batch IDs from the retry state files
- waits on each retry batch job until it reaches a terminal status
- downloads the branch-specific output and error files needed for retry materialization

This step does **not** modify the canonical paraphrase artifacts yet.


In [ ]:
import json

for job in RETRY_PARAPHRASE_BATCH_JOBS:
    print("waiting on retry paraphrase batch branch:", job["label"])
    if not job["state_file"].exists():
        print("  state file missing; run Step J-R3 first")
        continue

    state = json.loads(job["state_file"].read_text(encoding="utf-8"))
    batch_id = state.get("batch_id")
    if not batch_id:
        print("  missing batch_id in state file")
        continue

    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", str(job["state_file"]),
        "--wait",
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  retry branch wait/download completed with return code:", result.returncode)


##### Step J-R5 — Snapshot And Clear Canonical Error Logs Before Retry Materialization

The retry materialization script appends to the canonical paraphrase error logs. Run this step immediately before `J-R6` if you want the retry-pass error logs to be cleanly attributable to the residual recovery only.

What this step does:
- snapshots the current canonical paraphrase error logs
- clears the live canonical error-log files before the retry materialization pass

What this step does **not** do:
- it does **not** delete the canonical paraphrase artifacts
- it does **not** delete the downloaded retry batch files
- it does **not** call the API


In [ ]:
import shutil

SOURCE_CANONICAL_LOG = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl"
TEACHER_CANONICAL_LOG = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl"

for path in [SOURCE_CANONICAL_LOG, TEACHER_CANONICAL_LOG]:
    if path.exists():
        snapshot = path.with_name(path.stem + "_preretry_snapshot.jsonl")
        shutil.copy2(path, snapshot)
        path.unlink()
        print("archived:", snapshot)
        print("cleared:", path)
    else:
        print("missing:", path)


##### Step J-R6 — Materialize Retry Outputs Into The Canonical Paraphrase Artifacts

Run this after `J-R4` and, if you want a clean retry-only error record, after `J-R5`. It materializes the successful retry outputs back into the same canonical paraphrase artifacts used by `J-B4`.

What this step does:
- reads the retry batch output and error files from `J-R4`
- appends any newly recovered paraphrases into the canonical paraphrase artifacts
- writes retry-pass materialization failures into the canonical error logs

After this step, rerun `Step J-B5 — Audit Paraphrase Artifacts` and then rerun `J-R1` if you need to confirm whether any residual shortfall remains.


In [ ]:
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py",
)

RETRY_MATERIALIZE_JOBS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "batch_output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "batch_output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl",
    },
]

for job in RETRY_MATERIALIZE_JOBS:
    print("materializing retry paraphrase branch:", job["label"])
    if not job["batch_output_file"].exists():
        print("  missing retry batch output; run Step J-R4 first")
        continue

    cmd = [
        sys.executable,
        str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--batch-output-file", str(job["batch_output_file"]),
        "--batch-error-file", str(job["batch_error_file"]),
        "--output-file", str(job["output_file"]),
        "--log-file", str(job["log_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  retry materialization completed with return code:", result.returncode)


#### Stage J-Retry-2 — Independent Second Residual Retry Wave

This stage exists for cases where `J-R6` improves coverage but does not close the remaining residual gap. It mirrors the first retry stage, but writes a **new second-wave retry transport** so the next recovery pass is documented independently rather than by rerunning `J-R1` to `J-R6`.

Use this stage only after reviewing the post-`J-R6` state. It keeps the audit trail clear by:
- rebuilding the residual retry set from the live canonical paraphrase artifacts after retry wave 1,
- overwriting the same retry working files with the current residual set,
- snapshotting the canonical error logs again before second-wave materialization,
- and materializing the second-wave outputs back into the same canonical paraphrase artifacts.

This stage does **not** change the paraphrase model or prompt semantics. It only documents a second residual-recovery wave using a fresh transport and fresh bookkeeping files.


##### Step J-R7 — Build Second-Wave Residual Retry Manifests

Run this after the first retry wave if the canonical paraphrase artifacts still remain short of the expected `5` paraphrases per seed.

What this step does:
- recomputes the residual gap from the live canonical paraphrase artifacts after `J-R6`
- overwrites the existing retry manifests with the current second-wave residual set
- keeps the second-wave logic independent in the notebook without multiplying working files

This step does **not** call the API or modify the canonical paraphrase artifacts.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

SOURCE_SEED_FILE = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
SOURCE_PARAPHRASE_FILE = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl"
SOURCE_RETRY_FILE = PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl"

TEACHER_SEED_FILE = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"
TEACHER_PARAPHRASE_FILE = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl"
TEACHER_RETRY_FILE = PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl"

EXPECTED_PARAPHRASES = 5


def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("content_hash") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("paraphrase_source_content_hash") or "").strip()
        or str(meta.get("paraphrase_source") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def build_retry_v2(seed_file: Path, paraphrase_file: Path, retry_file: Path, label: str):
    seed_rows = load_jsonl(seed_file)
    paraphrase_rows = load_jsonl(paraphrase_file) if paraphrase_file.exists() else []

    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    seen = Counter()
    for row in paraphrase_rows:
        key = paraphrase_source_key(row)
        if key:
            seen[key] += 1

    missing_rows = []
    missing_distribution = Counter()
    for key, row in seed_map.items():
        have = seen.get(key, 0)
        if have < EXPECTED_PARAPHRASES:
            deficit = EXPECTED_PARAPHRASES - have
            out = dict(row)
            meta = dict(out.get("metadata", {}))
            meta["paraphrase_retry_reason"] = "missing_materialized_paraphrases_second_wave"
            meta["paraphrase_retry_wave"] = 2
            meta["paraphrase_retry_existing_count"] = have
            meta["paraphrase_retry_expected_count"] = EXPECTED_PARAPHRASES
            meta["paraphrase_retry_missing_count"] = deficit
            out["metadata"] = meta
            missing_rows.append(out)
            missing_distribution[have] += 1

    with retry_file.open("w", encoding="utf-8") as f:
        for row in missing_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nBranch: {label}")
    print("  seed rows:", f"{len(seed_rows):,}")
    print("  paraphrase rows:", f"{len(paraphrase_rows):,}")
    print("  retry seed rows:", f"{len(missing_rows):,}")
    print("  retry file:", retry_file)
    print("  existing paraphrase coverage among retry seeds:")
    for have, count in sorted(missing_distribution.items()):
        print(f"    {have} existing: {count:,} seeds")


build_retry_v2(SOURCE_SEED_FILE, SOURCE_PARAPHRASE_FILE, SOURCE_RETRY_FILE, "source_code")
build_retry_v2(TEACHER_SEED_FILE, TEACHER_PARAPHRASE_FILE, TEACHER_RETRY_FILE, "teacher_text")


##### Step J-R8 — Prepare Second-Wave Retry Batch Request Files

Run this after `J-R7`. It rebuilds the retry transport for the current second-wave residual set by reusing the same retry working files.

This step intentionally overwrites the prior retry transport files so the working set stays compact.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

PREPARE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "PREPARE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "prepare_paraphrases_quality_aware_batch.py",
)

PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)

RETRY2_PARAPHRASE_BATCH_JOBS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "request_file": BATCH_DIR / "source_code_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_v1.json",
        "output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "existing_output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "request_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_v1.json",
        "output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "existing_output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
    },
]

globals()["RETRY2_PARAPHRASE_BATCH_JOBS"] = RETRY2_PARAPHRASE_BATCH_JOBS

for job in RETRY2_PARAPHRASE_BATCH_JOBS:
    print("preparing second-wave retry paraphrase batch branch:", job["label"])
    if not job["seed_file"].exists():
        print("  missing second-wave retry manifest; run Step J-R7 first")
        continue

    for stale_path in [job["request_file"], job["state_file"], job["output_file"], job["error_file"]]:
        if stale_path.exists():
            stale_path.unlink()

    cmd = [
        sys.executable,
        str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--request-file", str(job["request_file"]),
        "--existing-output-file", str(job["existing_output_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  second-wave retry branch preparation completed with return code:", result.returncode)


##### Step J-R9 — Create Second-Wave Retry Batch Jobs

Run this after `J-R8`. It creates a new batch job per branch for the second residual-recovery wave.


In [ ]:
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    SCRIPTS_DIR / "run_openai_batch_job.py",
)

for job in RETRY2_PARAPHRASE_BATCH_JOBS:
    print("creating second-wave retry paraphrase batch branch:", job["label"])
    if not job["request_file"].exists():
        print("  request file missing; run Step J-R8 first")
        continue

    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(job["request_file"]),
        "--state-file", str(job["state_file"]),
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  second-wave retry branch create completed with return code:", result.returncode)


##### Step J-R10 — Wait For Completion And Download Second-Wave Retry Batch Files

Run this after `J-R9`. It waits on the new second-wave retry jobs and downloads their output and error files.


In [ ]:
import json

for job in RETRY2_PARAPHRASE_BATCH_JOBS:
    print("waiting on second-wave retry paraphrase batch branch:", job["label"])
    if not job["state_file"].exists():
        print("  state file missing; run Step J-R9 first")
        continue

    state = json.loads(job["state_file"].read_text(encoding="utf-8"))
    batch_id = state.get("batch_id")
    if not batch_id:
        print("  missing batch_id in state file")
        continue

    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", str(job["state_file"]),
        "--wait",
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  second-wave retry branch wait/download completed with return code:", result.returncode)


##### Step J-R11 — Snapshot And Clear Canonical Error Logs Before Second-Wave Materialization

Run this immediately before `J-R12` if you want the second-wave materialization errors to be attributable only to this independent retry wave.

This step overwrites the same `_preretry_snapshot` files so the working set stays compact.


In [ ]:
import shutil

SOURCE_CANONICAL_LOG = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl"
TEACHER_CANONICAL_LOG = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl"

for path in [SOURCE_CANONICAL_LOG, TEACHER_CANONICAL_LOG]:
    if path.exists():
        snapshot = path.with_name(path.stem + "_preretry_snapshot.jsonl")
        shutil.copy2(path, snapshot)
        path.unlink()
        print("archived:", snapshot)
        print("cleared:", path)
    else:
        print("missing:", path)


##### Step J-R12 — Materialize Second-Wave Retry Outputs Into The Canonical Paraphrase Artifacts

Run this after `J-R10` and, if you want clean second-wave error logs, after `J-R11`. It materializes the second-wave retry outputs back into the same canonical paraphrase artifacts used by `J-B4`, `J-R6`, and the rest of Stage J.

After this step, rerun `Step J-B5 — Audit Paraphrase Artifacts` to verify whether the residual gap is now fully closed.


In [ ]:
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py",
)

RETRY2_MATERIALIZE_JOBS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "batch_output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "batch_output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl",
    },
]

for job in RETRY2_MATERIALIZE_JOBS:
    print("materializing second-wave retry paraphrase branch:", job["label"])
    if not job["batch_output_file"].exists():
        print("  missing second-wave retry batch output; run Step J-R10 first")
        continue

    cmd = [
        sys.executable,
        str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--batch-output-file", str(job["batch_output_file"]),
        "--batch-error-file", str(job["batch_error_file"]),
        "--output-file", str(job["output_file"]),
        "--log-file", str(job["log_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  second-wave retry materialization completed with return code:", result.returncode)


#### Stage J-Retry-3 — Token-Safe Single-Paraphrase Recovery

This stage is designed for the stubborn residual tail where repeated retry waves still fail because the response is truncated before the JSON object closes. It uses the same retry working files, but changes the recovery strategy:
- each residual seed is asked for only **one** paraphrase in that request,
- the retry manifest records that one-paraphrase target explicitly,
- and the materializer accepts that partial recovery instead of requiring all remaining paraphrases in one shot.

Use this stage after `J-R12` if the residual error pattern is dominated by `max_output_tokens` / truncated JSON behavior.


##### Step J-R13 — Build Token-Safe Single-Paraphrase Retry Manifests

Run this after `J-R12` if the remaining residual failures are still dominated by truncated JSON responses.

What this step changes:
- it rebuilds the retry manifests from the live canonical paraphrase artifacts,
- it marks each residual seed with `paraphrase_retry_requested_count = 1`,
- and it overwrites the same retry manifests so the working set stays compact.

This does **not** call the API or modify the canonical paraphrase artifacts.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

SOURCE_SEED_FILE = PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl"
SOURCE_PARAPHRASE_FILE = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl"
SOURCE_RETRY_FILE = PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl"

TEACHER_SEED_FILE = PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl"
TEACHER_PARAPHRASE_FILE = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl"
TEACHER_RETRY_FILE = PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl"

EXPECTED_PARAPHRASES = 5
TARGETED_RETRY_REQUEST_COUNT = 1


def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("content_hash") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("paraphrase_source_content_hash") or "").strip()
        or str(meta.get("paraphrase_source") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def build_retry(seed_file: Path, paraphrase_file: Path, retry_file: Path, label: str):
    seed_rows = load_jsonl(seed_file)
    paraphrase_rows = load_jsonl(paraphrase_file) if paraphrase_file.exists() else []

    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    seen = Counter()
    for row in paraphrase_rows:
        key = paraphrase_source_key(row)
        if key:
            seen[key] += 1

    missing_rows = []
    missing_distribution = Counter()
    for key, row in seed_map.items():
        have = seen.get(key, 0)
        if have < EXPECTED_PARAPHRASES:
            deficit = EXPECTED_PARAPHRASES - have
            out = dict(row)
            meta = dict(out.get("metadata", {}))
            meta["paraphrase_retry_reason"] = "missing_materialized_paraphrases_token_safe_single"
            meta["paraphrase_retry_wave"] = 3
            meta["paraphrase_retry_existing_count"] = have
            meta["paraphrase_retry_expected_count"] = EXPECTED_PARAPHRASES
            meta["paraphrase_retry_missing_count"] = deficit
            meta["paraphrase_retry_requested_count"] = min(TARGETED_RETRY_REQUEST_COUNT, deficit)
            out["metadata"] = meta
            missing_rows.append(out)
            missing_distribution[have] += 1

    with retry_file.open("w", encoding="utf-8") as f:
        for row in missing_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nBranch: {label}")
    print("  seed rows:", f"{len(seed_rows):,}")
    print("  paraphrase rows:", f"{len(paraphrase_rows):,}")
    print("  retry seed rows:", f"{len(missing_rows):,}")
    print("  retry file:", retry_file)
    print("  requested paraphrases per residual seed:", TARGETED_RETRY_REQUEST_COUNT)
    print("  existing paraphrase coverage among retry seeds:")
    for have, count in sorted(missing_distribution.items()):
        print(f"    {have} existing: {count:,} seeds")


build_retry(SOURCE_SEED_FILE, SOURCE_PARAPHRASE_FILE, SOURCE_RETRY_FILE, "source_code")
build_retry(TEACHER_SEED_FILE, TEACHER_PARAPHRASE_FILE, TEACHER_RETRY_FILE, "teacher_text")


This is a more conservative approach to ensure higher quality outputs. 

In [ ]:
PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS = 700
AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS = 10

##### Step J-R14 — Prepare Token-Safe Single-Paraphrase Retry Batch Request Files

Run this after `J-R13`. It rebuilds the retry request files using the same working paths, but now each request asks for only one paraphrase per residual seed.

This is the main tactical change for the truncation-dominated residual tail.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

PREPARE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "PREPARE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "prepare_paraphrases_quality_aware_batch.py",
)

PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)
PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS = globals().get("PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS", 500)
PARAPHRASE_BATCH_MAX_PER_REQUEST = 1

RETRY3_PARAPHRASE_BATCH_JOBS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "request_file": BATCH_DIR / "source_code_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_v1.json",
        "output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "existing_output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "request_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_v1.json",
        "output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "existing_output_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
    },
]

globals()["RETRY3_PARAPHRASE_BATCH_JOBS"] = RETRY3_PARAPHRASE_BATCH_JOBS

for job in RETRY3_PARAPHRASE_BATCH_JOBS:
    print("preparing token-safe retry paraphrase batch branch:", job["label"])
    if not job["seed_file"].exists():
        print("  missing token-safe retry manifest; run Step J-R13 first")
        continue

    for stale_path in [job["request_file"], job["state_file"], job["output_file"], job["error_file"]]:
        if stale_path.exists():
            stale_path.unlink()

    cmd = [
        sys.executable,
        str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--request-file", str(job["request_file"]),
        "--existing-output-file", str(job["existing_output_file"]),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
        "--max-output-tokens", str(PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS),
        "--max-paraphrases-per-request", str(PARAPHRASE_BATCH_MAX_PER_REQUEST),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  token-safe retry branch preparation completed with return code:", result.returncode)


##### Step J-R15 — Create Token-Safe Retry Batch Jobs

Run this after `J-R14`. It creates a new retry batch job per branch for the token-safe one-paraphrase recovery wave.


In [ ]:
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    SCRIPTS_DIR / "run_openai_batch_job.py",
)

for job in RETRY3_PARAPHRASE_BATCH_JOBS:
    print("creating token-safe retry paraphrase batch branch:", job["label"])
    if not job["request_file"].exists():
        print("  request file missing; run Step J-R14 first")
        continue

    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--request-file", str(job["request_file"]),
        "--state-file", str(job["state_file"]),
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  token-safe retry branch create completed with return code:", result.returncode)


##### Step J-R16 — Wait For Completion And Download Token-Safe Retry Batch Files

Run this after `J-R15`. It waits on the token-safe retry jobs and downloads the resulting output and error files.


In [ ]:
import json

for job in RETRY3_PARAPHRASE_BATCH_JOBS:
    print("waiting on token-safe retry paraphrase batch branch:", job["label"])
    if not job["state_file"].exists():
        print("  state file missing; run Step J-R15 first")
        continue

    state = json.loads(job["state_file"].read_text(encoding="utf-8"))
    batch_id = state.get("batch_id")
    if not batch_id:
        print("  missing batch_id in state file")
        continue

    cmd = [
        sys.executable,
        str(RUN_BATCH_JOB_SCRIPT),
        "--batch-id", batch_id,
        "--state-file", str(job["state_file"]),
        "--wait",
        "--download-output-file", str(job["output_file"]),
        "--download-error-file", str(job["error_file"]),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  token-safe retry branch wait/download completed with return code:", result.returncode)


##### Step J-R17 — Snapshot And Clear Canonical Error Logs Before Token-Safe Materialization

Run this immediately before `J-R18` if you want the token-safe third-wave materialization errors to be attributable only to this recovery wave.

This step still overwrites the same `_preretry_snapshot` files to avoid additional file sprawl.


In [ ]:
import shutil

SOURCE_CANONICAL_LOG = PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl"
TEACHER_CANONICAL_LOG = PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl"

for path in [SOURCE_CANONICAL_LOG, TEACHER_CANONICAL_LOG]:
    if path.exists():
        snapshot = path.with_name(path.stem + "_preretry_snapshot.jsonl")
        shutil.copy2(path, snapshot)
        path.unlink()
        print("archived:", snapshot)
        print("cleared:", path)
    else:
        print("missing:", path)


##### Step J-R18 — Materialize Token-Safe Retry Outputs Into The Canonical Paraphrase Artifacts

Run this after `J-R16` and, if desired, after `J-R17`. It materializes the token-safe retry outputs back into the canonical paraphrase artifacts.

Because the retry manifests mark each residual seed with `paraphrase_retry_requested_count = 1`, the materializer will now accept one recovered paraphrase per seed instead of insisting on all remaining paraphrases in a single response.

After this step, rerun `Step J-B5 — Audit Paraphrase Artifacts` to measure how much of the residual tail closed.


In [ ]:
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py",
)

for job in RETRY3_PARAPHRASE_BATCH_JOBS:
    print("materializing token-safe retry paraphrase branch:", job["label"])
    if not job["output_file"].exists():
        print("  missing token-safe retry batch output; run Step J-R16 first")
        continue

    cmd = [
        sys.executable,
        str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
        "--seed-file", str(job["seed_file"]),
        "--batch-output-file", str(job["output_file"]),
        "--batch-error-file", str(job["error_file"]),
        "--output-file", str(job["existing_output_file"]),
        "--log-file", str(PROCESSED_DIR / f'seed_paraphrases_quality_aware_{job["label"]}_v1_errors.jsonl'),
        "--model", PARAPHRASE_BATCH_MODEL,
        "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
        "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
        "--max-output-tokens", str(PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS),
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    print("  token-safe retry materialization completed with return code:", result.returncode)


#### Stage J-Retry-Auto — Automated Token-Safe Residual Closure Loop

This optional orchestrator cell runs the token-safe retry wave repeatedly until one of the following stopping conditions is reached:
- the residual paraphrase-slot gap closes completely,
- a round produces zero newly materialized paraphrases,
- or the configured maximum number of rounds is reached.

What this cell does:
- rebuilds the token-safe retry manifests before each round,
- reuses the same retry working files to avoid file sprawl,
- submits, waits, downloads, snapshots/clears logs, and materializes in one loop,
- and prints a round-by-round closure summary.

Run this instead of manually repeating `J-R13` to `J-R18` when the residual tail is clearly in token-safe closure mode.


In [ ]:
import json
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

PREPARE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "PREPARE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "prepare_paraphrases_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    SCRIPTS_DIR / "run_openai_batch_job.py",
)
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py",
)

PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)
AUTO_TOKEN_SAFE_RETRY_MAX_OUTPUT_TOKENS = globals().get("PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS", 700)
AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS = globals().get("AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS", 10)
AUTO_TOKEN_SAFE_RETRY_REQUEST_COUNT = 1
EXPECTED_PARAPHRASES = 5

BRANCH_SPECS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
        "retry_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "request_file": BATCH_DIR / "source_code_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_v1.json",
        "batch_output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
        "retry_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "request_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_v1.json",
        "batch_output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl",
    },
]


def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("content_hash") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return (
        str(meta.get("paraphrase_source_content_hash") or "").strip()
        or str(meta.get("paraphrase_source") or "").strip()
        or str(meta.get("circuit_hash") or "").strip()
    )


def summarize_branch(seed_file: Path, artifact_file: Path) -> dict:
    seed_rows = load_jsonl(seed_file)
    artifact_rows = load_jsonl(artifact_file)
    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    seen = Counter()
    for row in artifact_rows:
        key = paraphrase_source_key(row)
        if key:
            seen[key] += 1

    represented = 0
    residual_seeds = 0
    residual_slots = 0
    for key in seed_map:
        have = seen.get(key, 0)
        if have > 0:
            represented += 1
        if have < EXPECTED_PARAPHRASES:
            residual_seeds += 1
            residual_slots += EXPECTED_PARAPHRASES - have

    return {
        "seed_rows": len(seed_map),
        "artifact_rows": len(artifact_rows),
        "represented_seeds": represented,
        "residual_seeds": residual_seeds,
        "residual_slots": residual_slots,
    }


def build_token_safe_retry_manifest(spec: dict) -> int:
    seed_rows = load_jsonl(spec["seed_file"])
    artifact_rows = load_jsonl(spec["artifact_file"])

    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    seen = Counter()
    for row in artifact_rows:
        key = paraphrase_source_key(row)
        if key:
            seen[key] += 1

    missing_rows = []
    for key, row in seed_map.items():
        have = seen.get(key, 0)
        if have < EXPECTED_PARAPHRASES:
            deficit = EXPECTED_PARAPHRASES - have
            out = dict(row)
            meta = dict(out.get("metadata", {}))
            meta["paraphrase_retry_reason"] = "missing_materialized_paraphrases_token_safe_auto"
            meta["paraphrase_retry_wave"] = "auto"
            meta["paraphrase_retry_existing_count"] = have
            meta["paraphrase_retry_expected_count"] = EXPECTED_PARAPHRASES
            meta["paraphrase_retry_missing_count"] = deficit
            meta["paraphrase_retry_requested_count"] = min(AUTO_TOKEN_SAFE_RETRY_REQUEST_COUNT, deficit)
            out["metadata"] = meta
            missing_rows.append(out)

    with spec["retry_file"].open("w", encoding="utf-8") as f:
        for row in missing_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    return len(missing_rows)


def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    return result


def total_gap(stats_by_label: dict[str, dict]) -> int:
    return sum(stats["residual_slots"] for stats in stats_by_label.values())


print("AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS =", AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS)
print("AUTO_TOKEN_SAFE_RETRY_MAX_OUTPUT_TOKENS =", AUTO_TOKEN_SAFE_RETRY_MAX_OUTPUT_TOKENS)
print("AUTO_TOKEN_SAFE_RETRY_REQUEST_COUNT =", AUTO_TOKEN_SAFE_RETRY_REQUEST_COUNT)

for round_idx in range(1, AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS + 1):
    print(f"\n===== Auto Token-Safe Retry Round {round_idx} =====")

    before = {spec["label"]: summarize_branch(spec["seed_file"], spec["artifact_file"]) for spec in BRANCH_SPECS}
    gap_before = total_gap(before)
    print("residual slots before round:", gap_before)
    for label, stats in before.items():
        print(
            f"  {label}: rows={stats['artifact_rows']:,}, represented={stats['represented_seeds']:,}/{stats['seed_rows']:,}, residual_seeds={stats['residual_seeds']:,}, residual_slots={stats['residual_slots']:,}"
        )

    if gap_before == 0:
        print("Residual gap already closed. Stopping.")
        break

    active_specs = []
    for spec in BRANCH_SPECS:
        retry_count = build_token_safe_retry_manifest(spec)
        print(f"  retry seeds for {spec['label']}: {retry_count:,}")
        if retry_count > 0:
            active_specs.append(spec)

    if not active_specs:
        print("No active retry seeds remain. Stopping.")
        break

    for spec in active_specs:
        print(f"\nPreparing {spec['label']}...")
        for stale_path in [spec["request_file"], spec["state_file"], spec["batch_output_file"], spec["batch_error_file"]]:
            if stale_path.exists():
                stale_path.unlink()
        run_cmd([
            sys.executable,
            str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
            "--seed-file", str(spec["retry_file"]),
            "--request-file", str(spec["request_file"]),
            "--existing-output-file", str(spec["artifact_file"]),
            "--model", PARAPHRASE_BATCH_MODEL,
            "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
            "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
            "--max-output-tokens", str(AUTO_TOKEN_SAFE_RETRY_MAX_OUTPUT_TOKENS),
            "--max-paraphrases-per-request", str(AUTO_TOKEN_SAFE_RETRY_REQUEST_COUNT),
        ])

    for spec in active_specs:
        print(f"\nCreating batch job for {spec['label']}...")
        run_cmd([
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--request-file", str(spec["request_file"]),
            "--state-file", str(spec["state_file"]),
            "--download-output-file", str(spec["batch_output_file"]),
            "--download-error-file", str(spec["batch_error_file"]),
        ])

    for spec in active_specs:
        print(f"\nWaiting on batch for {spec['label']}...")
        state = json.loads(spec["state_file"].read_text(encoding="utf-8"))
        batch_id = state.get("batch_id")
        if not batch_id:
            raise RuntimeError(f"Missing batch_id in {spec['state_file']}")
        run_cmd([
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--batch-id", batch_id,
            "--state-file", str(spec["state_file"]),
            "--wait",
            "--download-output-file", str(spec["batch_output_file"]),
            "--download-error-file", str(spec["batch_error_file"]),
        ])

    for spec in active_specs:
        if spec["log_file"].exists():
            snapshot = spec["log_file"].with_name(spec["log_file"].stem + "_preretry_snapshot.jsonl")
            shutil.copy2(spec["log_file"], snapshot)
            spec["log_file"].unlink()
            print("archived:", snapshot)
            print("cleared:", spec["log_file"])

    for spec in active_specs:
        print(f"\nMaterializing {spec['label']}...")
        run_cmd([
            sys.executable,
            str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
            "--seed-file", str(spec["retry_file"]),
            "--batch-output-file", str(spec["batch_output_file"]),
            "--batch-error-file", str(spec["batch_error_file"]),
            "--output-file", str(spec["artifact_file"]),
            "--log-file", str(spec["log_file"]),
            "--model", PARAPHRASE_BATCH_MODEL,
            "--temperature", str(PARAPHRASE_BATCH_TEMPERATURE),
            "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
            "--max-output-tokens", str(AUTO_TOKEN_SAFE_RETRY_MAX_OUTPUT_TOKENS),
        ])

    after = {spec["label"]: summarize_branch(spec["seed_file"], spec["artifact_file"]) for spec in BRANCH_SPECS}
    gap_after = total_gap(after)
    rows_added = sum(after[label]["artifact_rows"] - before[label]["artifact_rows"] for label in after)
    print(f"\nRound {round_idx} summary:")
    print("  rows added:", f"{rows_added:,}")
    print("  residual slots:", f"{gap_before:,} -> {gap_after:,}")
    for label in after:
        print(
            f"  {label}: rows {before[label]['artifact_rows']:,} -> {after[label]['artifact_rows']:,}; residual_slots {before[label]['residual_slots']:,} -> {after[label]['residual_slots']:,}"
        )

    if gap_after == 0:
        print("Residual gap fully closed. Stopping.")
        break
    if rows_added == 0:
        print("No progress in this round. Stopping to avoid a blind infinite loop.")
        break
else:
    print(f"Reached AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS={AUTO_TOKEN_SAFE_RETRY_MAX_ROUNDS} before full closure.")

final_stats = {spec["label"]: summarize_branch(spec["seed_file"], spec["artifact_file"]) for spec in BRANCH_SPECS}
print("\nFinal automated-loop status:")
for label, stats in final_stats.items():
    print(
        f"  {label}: rows={stats['artifact_rows']:,}, represented={stats['represented_seeds']:,}/{stats['seed_rows']:,}, residual_seeds={stats['residual_seeds']:,}, residual_slots={stats['residual_slots']:,}"
    )
print("\nRerun Step J-B5 if you want the formal audit cell output recorded separately.")


#### Stage J-Finalize — Global Duplicate Remediation And Canonical Closure

This finalization loop handles the last two issues together:
- any remaining under-covered source seeds,
- and any **cross-seed exact normalized duplicate** paraphrases within a branch.

How it works:
- it computes a **virtual canonical selection** that keeps at most `5` paraphrases per source seed and keeps each normalized paraphrase text only once per branch,
- any source seed that would fall below `5` under that canonical selection is assigned a remediation deficit,
- the cell then runs token-safe one-paraphrase retry rounds for only those deficits,
- if a round stalls, it can automatically escalate through a configured temperature fallback schedule,
- and once the virtual canonical selection reaches `5` globally unique paraphrases for every seed, it rewrites the canonical paraphrase files to that final deduplicated selection.

Important:
- this loop intentionally keeps the current full artifact as the candidate pool during remediation rounds,
- so dropped duplicate rows are still remembered while the model searches for replacements,
- and the canonical artifact is only rewritten once the branch is actually closed.


In [ ]:
import json
import re
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
BATCH_DIR = globals().get("BATCH_DIR", PROCESSED_DIR / "openai_batch_jobs")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

PREPARE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "PREPARE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "prepare_paraphrases_quality_aware_batch.py",
)
RUN_BATCH_JOB_SCRIPT = globals().get(
    "RUN_BATCH_JOB_SCRIPT",
    SCRIPTS_DIR / "run_openai_batch_job.py",
)
MATERIALIZE_PARAPHRASE_BATCH_SCRIPT = globals().get(
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    SCRIPTS_DIR / "materialize_paraphrases_quality_aware_batch.py",
)

PARAPHRASE_BATCH_MODEL = globals().get("PARAPHRASE_BATCH_MODEL", "gpt-5.4-mini")
PARAPHRASE_BATCH_TEMPERATURE = globals().get("PARAPHRASE_BATCH_TEMPERATURE", 0.2)
PARAPHRASE_BATCH_COUNT = globals().get("PARAPHRASE_BATCH_COUNT", 5)
FINAL_PARAPHRASE_MAX_OUTPUT_TOKENS = globals().get("PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS", 700)
FINAL_PARAPHRASE_MAX_ROUNDS = globals().get("FINAL_PARAPHRASE_MAX_ROUNDS", 12)
FINAL_PARAPHRASE_TEMPERATURE_SCHEDULE = globals().get(
    "FINAL_PARAPHRASE_TEMPERATURE_SCHEDULE",
    [PARAPHRASE_BATCH_TEMPERATURE, 0.4, 0.6],
)
FINAL_PARAPHRASE_REQUEST_COUNT = 1
EXPECTED_PARAPHRASES = 5

BRANCH_SPECS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
        "retry_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_source_code_v1.jsonl",
        "request_file": BATCH_DIR / "source_code_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_v1.json",
        "batch_output_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "source_code_paraphrase_retry_missing_batch_error_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_errors.jsonl",
        "snapshot_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_precanonical_snapshot.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
        "retry_file": PROCESSED_DIR / "seed_paraphrase_retry_missing_teacher_text_v1.jsonl",
        "request_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_requests_v1.jsonl",
        "state_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_v1.json",
        "batch_output_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_output_v1.jsonl",
        "batch_error_file": BATCH_DIR / "teacher_text_paraphrase_retry_missing_batch_error_v1.jsonl",
        "log_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_errors.jsonl",
        "snapshot_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_precanonical_snapshot.jsonl",
    },
]

NORMALIZE_RE = re.compile(r"\s+")


def normalize_prompt_text(text: str) -> str:
    return NORMALIZE_RE.sub(" ", str(text).strip().lower())


def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def write_jsonl(rows, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return str(meta.get("content_hash") or meta.get("circuit_hash") or "").strip()


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return str(
        meta.get("paraphrase_source_content_hash")
        or meta.get("paraphrase_source")
        or meta.get("circuit_hash")
        or ""
    ).strip()


def analyze_branch(spec: dict) -> dict:
    seed_rows = load_jsonl(spec["seed_file"])
    artifact_rows = load_jsonl(spec["artifact_file"])
    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    global_norm_counts = Counter()
    for row in artifact_rows:
        global_norm_counts[normalize_prompt_text(row.get("input", ""))] += 1

    kept_rows = []
    kept_counts = Counter()
    seen_norms = set()
    dropped_duplicate_rows = 0
    dropped_overflow_rows = 0

    for row in artifact_rows:
        source_id = paraphrase_source_key(row)
        if not source_id:
            continue
        norm = normalize_prompt_text(row.get("input", ""))
        if kept_counts[source_id] >= EXPECTED_PARAPHRASES:
            dropped_overflow_rows += 1
            continue
        if norm in seen_norms:
            dropped_duplicate_rows += 1
            continue
        kept_rows.append(row)
        kept_counts[source_id] += 1
        seen_norms.add(norm)

    deficits = {}
    for source_id in seed_map:
        have = kept_counts.get(source_id, 0)
        if have < EXPECTED_PARAPHRASES:
            deficits[source_id] = EXPECTED_PARAPHRASES - have

    return {
        "seed_map": seed_map,
        "artifact_rows": artifact_rows,
        "kept_rows": kept_rows,
        "kept_counts": kept_counts,
        "deficits": deficits,
        "raw_row_count": len(artifact_rows),
        "canonical_row_count": len(kept_rows),
        "duplicate_groups": sum(1 for v in global_norm_counts.values() if v > 1),
        "duplicate_extra_rows": sum(v - 1 for v in global_norm_counts.values() if v > 1),
        "dropped_duplicate_rows": dropped_duplicate_rows,
        "dropped_overflow_rows": dropped_overflow_rows,
        "represented_seeds": sum(1 for source_id in seed_map if kept_counts.get(source_id, 0) > 0),
    }


def build_remediation_manifest(spec: dict, analysis: dict) -> int:
    rows = []
    for source_id, deficit in analysis["deficits"].items():
        seed_entry = analysis["seed_map"][source_id]
        out = dict(seed_entry)
        meta = dict(out.get("metadata", {}))
        meta["paraphrase_retry_reason"] = "final_global_duplicate_or_missing_remediation"
        meta["paraphrase_retry_wave"] = "finalize"
        meta["paraphrase_retry_existing_count"] = analysis["kept_counts"].get(source_id, 0)
        meta["paraphrase_retry_expected_count"] = EXPECTED_PARAPHRASES
        meta["paraphrase_retry_missing_count"] = deficit
        meta["paraphrase_retry_requested_count"] = min(FINAL_PARAPHRASE_REQUEST_COUNT, deficit)
        out["metadata"] = meta
        rows.append(out)
    write_jsonl(rows, spec["retry_file"])
    return len(rows)


def run_cmd(cmd):
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print("stderr")
        print(result.stderr.strip())
    return result


def total_deficit(analyses: dict[str, dict]) -> int:
    return sum(sum(a["deficits"].values()) for a in analyses.values())


def print_analysis(prefix: str, analyses: dict[str, dict]):
    print(prefix)
    for label, analysis in analyses.items():
        print(
            f"  {label}: raw_rows={analysis['raw_row_count']:,}, canonical_rows={analysis['canonical_row_count']:,}, represented={analysis['represented_seeds']:,}/{len(analysis['seed_map']):,}, duplicate_groups={analysis['duplicate_groups']:,}, duplicate_extra_rows={analysis['duplicate_extra_rows']:,}, residual_slots={sum(analysis['deficits'].values()):,}"
        )


print("FINAL_PARAPHRASE_MAX_ROUNDS =", FINAL_PARAPHRASE_MAX_ROUNDS)
print("FINAL_PARAPHRASE_MAX_OUTPUT_TOKENS =", FINAL_PARAPHRASE_MAX_OUTPUT_TOKENS)
print("FINAL_PARAPHRASE_TEMPERATURE_SCHEDULE =", FINAL_PARAPHRASE_TEMPERATURE_SCHEDULE)

temperature_schedule = [float(value) for value in FINAL_PARAPHRASE_TEMPERATURE_SCHEDULE]
if not temperature_schedule:
    raise RuntimeError("FINAL_PARAPHRASE_TEMPERATURE_SCHEDULE must contain at least one temperature value")
temperature_idx = 0
current_temperature = temperature_schedule[temperature_idx]

closed = False
for round_idx in range(1, FINAL_PARAPHRASE_MAX_ROUNDS + 1):
    print(f"\n===== Final Canonical Remediation Round {round_idx} =====")
    print("current remediation temperature:", current_temperature)
    before = {spec['label']: analyze_branch(spec) for spec in BRANCH_SPECS}
    deficit_before = total_deficit(before)
    print_analysis("State before round:", before)

    if deficit_before == 0:
        closed = True
        print("Canonical closure already achieved. No more remediation needed.")
        break

    active_specs = []
    for spec in BRANCH_SPECS:
        remediation_rows = build_remediation_manifest(spec, before[spec['label']])
        print(f"  remediation seeds for {spec['label']}: {remediation_rows:,}")
        if remediation_rows > 0:
            active_specs.append(spec)

    if not active_specs:
        print("No remediation seeds remain, but canonical deficits were reported. Stopping for safety.")
        break

    for spec in active_specs:
        print(f"\nPreparing remediation batch for {spec['label']}...")
        for stale_path in [spec['request_file'], spec['state_file'], spec['batch_output_file'], spec['batch_error_file']]:
            if stale_path.exists():
                stale_path.unlink()
        run_cmd([
            sys.executable,
            str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
            "--seed-file", str(spec['retry_file']),
            "--request-file", str(spec['request_file']),
            "--existing-output-file", str(spec['artifact_file']),
            "--model", PARAPHRASE_BATCH_MODEL,
            "--temperature", str(current_temperature),
            "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
            "--max-output-tokens", str(FINAL_PARAPHRASE_MAX_OUTPUT_TOKENS),
            "--max-paraphrases-per-request", str(FINAL_PARAPHRASE_REQUEST_COUNT),
        ])

    for spec in active_specs:
        print(f"\nCreating remediation batch for {spec['label']}...")
        run_cmd([
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--request-file", str(spec['request_file']),
            "--state-file", str(spec['state_file']),
            "--download-output-file", str(spec['batch_output_file']),
            "--download-error-file", str(spec['batch_error_file']),
        ])

    for spec in active_specs:
        print(f"\nWaiting on remediation batch for {spec['label']}...")
        state = json.loads(spec['state_file'].read_text(encoding='utf-8'))
        batch_id = state.get('batch_id')
        if not batch_id:
            raise RuntimeError(f"Missing batch_id in {spec['state_file']}")
        run_cmd([
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--batch-id", batch_id,
            "--state-file", str(spec['state_file']),
            "--wait",
            "--download-output-file", str(spec['batch_output_file']),
            "--download-error-file", str(spec['batch_error_file']),
        ])

    for spec in active_specs:
        if spec['log_file'].exists():
            snapshot = spec['log_file'].with_name(spec['log_file'].stem + "_preretry_snapshot.jsonl")
            shutil.copy2(spec['log_file'], snapshot)
            spec['log_file'].unlink()
            print("archived:", snapshot)
            print("cleared:", spec['log_file'])

    for spec in active_specs:
        print(f"\nMaterializing remediation batch for {spec['label']}...")
        run_cmd([
            sys.executable,
            str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
            "--seed-file", str(spec['retry_file']),
            "--batch-output-file", str(spec['batch_output_file']),
            "--batch-error-file", str(spec['batch_error_file']),
            "--output-file", str(spec['artifact_file']),
            "--log-file", str(spec['log_file']),
            "--model", PARAPHRASE_BATCH_MODEL,
            "--temperature", str(current_temperature),
            "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
            "--max-output-tokens", str(FINAL_PARAPHRASE_MAX_OUTPUT_TOKENS),
        ])

    after = {spec['label']: analyze_branch(spec) for spec in BRANCH_SPECS}
    deficit_after = total_deficit(after)
    improvement = deficit_before - deficit_after
    print_analysis("State after round:", after)
    print("  deficit improvement:", f"{deficit_before:,} -> {deficit_after:,}")

    if deficit_after == 0:
        closed = True
        break
    if improvement <= 0:
        if temperature_idx + 1 < len(temperature_schedule):
            temperature_idx += 1
            current_temperature = temperature_schedule[temperature_idx]
            print(
                f"No canonical improvement in this round. Escalating remediation temperature to {current_temperature} and continuing."
            )
            continue
        print("No canonical improvement in this round and no higher fallback temperatures remain. Stopping.")
        break
else:
    print(f"Reached FINAL_PARAPHRASE_MAX_ROUNDS={FINAL_PARAPHRASE_MAX_ROUNDS} before canonical closure.")

final_analyses = {spec['label']: analyze_branch(spec) for spec in BRANCH_SPECS}
final_deficit = total_deficit(final_analyses)
print_analysis("\nFinal virtual canonical state:", final_analyses)

if final_deficit == 0:
    print("\nCanonical closure achieved. Rewriting canonical paraphrase files to the deduplicated final selection...")
    for spec in BRANCH_SPECS:
        analysis = final_analyses[spec['label']]
        shutil.copy2(spec['artifact_file'], spec['snapshot_file'])
        write_jsonl(analysis['kept_rows'], spec['artifact_file'])
        print("  snapshot:", spec['snapshot_file'])
        print("  rewritten canonical artifact:", spec['artifact_file'])
        print("  canonical rows:", f"{len(analysis['kept_rows']):,}")
else:
    print("\nCanonical closure not yet achieved. The full candidate pool was left unchanged so another finalization run can continue from the same state.")

print("\nRerun Step J-B5 after a successful rewrite if you want the final audited canonical totals recorded in the notebook.")


#### Stage J-Final-Tail — Anti-Template Residual Closure

This final-tail recovery cell is intended only for the tiny residual teacher-text gap that can remain after `Stage J-Finalize` stalls under the normal remediation prompt.

Why this cell exists:
- sometimes the remaining responses are valid and complete, but they are still filtered out because they collapse back into the same generic paraphrase template family,
- in that situation, more temperature alone is often not enough,
- so this cell switches the retry request into an explicit `anti_template` prompt mode and forces a rotating set of rhetorical surface forms.

What this cell does:
- targets only one branch at a time, defaulting to `teacher_text`,
- reuses the existing retry working files to avoid file sprawl,
- annotates each residual seed with an anti-template surface-form request,
- prepares, submits, waits, downloads, snapshots logs, and materializes one anti-template tail round,
- and reports the before/after canonical deficit for the target branch.

What this cell does not do:
- it does not rewrite the final canonical artifact on its own,
- it does not replace `Stage J-Finalize`,
- it does not broaden the retry scope back to already-closed branches.

When to run:
- run this only after `Stage J-Finalize` has stalled with a very small residual gap and the remaining outputs appear to be valid-but-filtered rather than malformed.


In [ ]:
FINAL_TAIL_TARGET_BRANCH = "teacher_text"
FINAL_TAIL_PROMPT_MODE = "anti_template"
FINAL_TAIL_TEMPERATURE = 0.9
FINAL_TAIL_MAX_OUTPUT_TOKENS = 700


In [ ]:
import json
import shutil
import sys
from collections import Counter

FINAL_TAIL_TARGET_BRANCH = globals().get("FINAL_TAIL_TARGET_BRANCH", "teacher_text")
FINAL_TAIL_PROMPT_MODE = globals().get("FINAL_TAIL_PROMPT_MODE", "anti_template")
FINAL_TAIL_TEMPERATURE = globals().get("FINAL_TAIL_TEMPERATURE", 0.9)
FINAL_TAIL_MAX_OUTPUT_TOKENS = globals().get(
    "FINAL_TAIL_MAX_OUTPUT_TOKENS",
    globals().get("PARAPHRASE_BATCH_MAX_OUTPUT_TOKENS", 700),
)
FINAL_TAIL_REQUEST_COUNT = 1
FINAL_TAIL_STYLE_FAMILIES = globals().get(
    "FINAL_TAIL_STYLE_FAMILIES",
    [
        "direct_question",
        "troubleshooting_request",
        "code_review_request",
        "repair_plan_request",
        "risk_assessment_request",
    ],
)

required_names = [
    "BRANCH_SPECS",
    "EXPECTED_PARAPHRASES",
    "PREPARE_PARAPHRASE_BATCH_SCRIPT",
    "RUN_BATCH_JOB_SCRIPT",
    "MATERIALIZE_PARAPHRASE_BATCH_SCRIPT",
    "PARAPHRASE_BATCH_MODEL",
    "PARAPHRASE_BATCH_COUNT",
    "analyze_branch",
    "write_jsonl",
    "print_analysis",
    "run_cmd",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Stage J-Finalize first so the canonical analysis helpers are loaded. Missing: "
        + ", ".join(missing)
    )

spec = next((item for item in BRANCH_SPECS if item["label"] == FINAL_TAIL_TARGET_BRANCH), None)
if spec is None:
    raise RuntimeError(f"Unknown FINAL_TAIL_TARGET_BRANCH: {FINAL_TAIL_TARGET_BRANCH}")

before = analyze_branch(spec)
deficit_before = sum(before["deficits"].values())
print("FINAL_TAIL_TARGET_BRANCH =", FINAL_TAIL_TARGET_BRANCH)
print("FINAL_TAIL_PROMPT_MODE =", FINAL_TAIL_PROMPT_MODE)
print("FINAL_TAIL_TEMPERATURE =", FINAL_TAIL_TEMPERATURE)
print("FINAL_TAIL_MAX_OUTPUT_TOKENS =", FINAL_TAIL_MAX_OUTPUT_TOKENS)
print("FINAL_TAIL_STYLE_FAMILIES =", FINAL_TAIL_STYLE_FAMILIES)
print_analysis("State before anti-template tail round:", {spec["label"]: before})

if deficit_before == 0:
    print("No residual canonical slots remain for the target branch.")
else:
    rows = []
    style_counts = Counter()
    for index, (source_id, deficit) in enumerate(sorted(before["deficits"].items())):
        seed_entry = before["seed_map"][source_id]
        out = dict(seed_entry)
        meta = dict(out.get("metadata", {}))
        style = FINAL_TAIL_STYLE_FAMILIES[index % len(FINAL_TAIL_STYLE_FAMILIES)]
        meta["paraphrase_retry_reason"] = "final_tail_anti_template_remediation"
        meta["paraphrase_retry_wave"] = "finalize_tail"
        meta["paraphrase_retry_existing_count"] = before["kept_counts"].get(source_id, 0)
        meta["paraphrase_retry_expected_count"] = EXPECTED_PARAPHRASES
        meta["paraphrase_retry_missing_count"] = deficit
        meta["paraphrase_retry_requested_count"] = min(FINAL_TAIL_REQUEST_COUNT, deficit)
        meta["paraphrase_retry_prompt_mode"] = FINAL_TAIL_PROMPT_MODE
        meta["paraphrase_retry_surface_form"] = style
        out["metadata"] = meta
        rows.append(out)
        style_counts[style] += 1

    write_jsonl(rows, spec["retry_file"])
    print("anti-template remediation seeds:", f"{len(rows):,}")
    print("retry manifest:", spec["retry_file"])
    print("surface-form distribution:")
    for key, value in style_counts.items():
        print(f"  {key}: {value:,}")

    for stale_path in [spec["request_file"], spec["state_file"], spec["batch_output_file"], spec["batch_error_file"]]:
        if stale_path.exists():
            stale_path.unlink()

    print(f"\nPreparing anti-template tail batch for {spec['label']}...")
    run_cmd(
        [
            sys.executable,
            str(PREPARE_PARAPHRASE_BATCH_SCRIPT),
            "--seed-file", str(spec["retry_file"]),
            "--request-file", str(spec["request_file"]),
            "--existing-output-file", str(spec["artifact_file"]),
            "--model", PARAPHRASE_BATCH_MODEL,
            "--temperature", str(FINAL_TAIL_TEMPERATURE),
            "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
            "--max-output-tokens", str(FINAL_TAIL_MAX_OUTPUT_TOKENS),
            "--max-paraphrases-per-request", "1",
            "--prompt-mode", FINAL_TAIL_PROMPT_MODE,
        ]
    )

    print(f"\nCreating anti-template tail batch for {spec['label']}...")
    run_cmd(
        [
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--request-file", str(spec["request_file"]),
            "--state-file", str(spec["state_file"]),
            "--download-output-file", str(spec["batch_output_file"]),
            "--download-error-file", str(spec["batch_error_file"]),
        ]
    )

    batch_state = json.loads(spec["state_file"].read_text(encoding="utf-8"))
    batch_id = batch_state.get("batch_id")
    if not batch_id:
        raise RuntimeError("anti-template tail batch state file is missing batch_id")

    print(f"\nWaiting on anti-template tail batch for {spec['label']}...")
    run_cmd(
        [
            sys.executable,
            str(RUN_BATCH_JOB_SCRIPT),
            "--batch-id", batch_id,
            "--state-file", str(spec["state_file"]),
            "--wait",
            "--download-output-file", str(spec["batch_output_file"]),
            "--download-error-file", str(spec["batch_error_file"]),
        ]
    )

    snapshot = spec["log_file"].with_name(spec["log_file"].stem + "_pretail_snapshot.jsonl")
    if spec["log_file"].exists():
        shutil.copy2(spec["log_file"], snapshot)
        spec["log_file"].unlink()
        print("archived:", snapshot)
        print("cleared:", spec["log_file"])
    else:
        print("no existing log file to snapshot:", spec["log_file"])

    print(f"\nMaterializing anti-template tail batch for {spec['label']}...")
    run_cmd(
        [
            sys.executable,
            str(MATERIALIZE_PARAPHRASE_BATCH_SCRIPT),
            "--seed-file", str(spec["retry_file"]),
            "--batch-output-file", str(spec["batch_output_file"]),
            "--batch-error-file", str(spec["batch_error_file"]),
            "--output-file", str(spec["artifact_file"]),
            "--log-file", str(spec["log_file"]),
            "--model", PARAPHRASE_BATCH_MODEL,
            "--temperature", str(FINAL_TAIL_TEMPERATURE),
            "--num-paraphrases", str(PARAPHRASE_BATCH_COUNT),
            "--max-output-tokens", str(FINAL_TAIL_MAX_OUTPUT_TOKENS),
        ]
    )

    after = analyze_branch(spec)
    deficit_after = sum(after["deficits"].values())
    print_analysis("State after anti-template tail round:", {spec["label"]: after})
    print(f"deficit improvement: {deficit_before:,} -> {deficit_after:,}")
    if deficit_after == 0:
        print("Target branch canonical closure achieved under anti-template tail mode.")
    else:
        print("If residual slots remain, rerun this cell to try another anti-template tail round.")


#### Stage J-Canonicalize — Snapshot And Rewrite Final Paraphrase Artifacts

This final housekeeping cell rewrites the paraphrase artifacts from the current raw candidate pool into the lean canonical form.

What it does:
- recomputes the canonical paraphrase selection for each branch
- keeps at most `5` paraphrases per source seed
- keeps each normalized paraphrase text only once per branch
- refuses to rewrite anything unless canonical closure is already complete
- snapshots the current raw candidate-pool files before rewriting

Why this cell exists:
- after final-tail recovery, the artifacts can still contain duplicate or overflow rows because the full candidate pool is intentionally preserved during remediation
- this cell performs only the final cleanup rewrite, so the artifact becomes easier to inspect and audit

What this cell does not do:
- it does not call the API
- it does not generate new paraphrases
- it does not change the methodological temperature rationale


In [ ]:
import json
import re
import shutil
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

EXPECTED_PARAPHRASES = 5
NORMALIZE_RE = re.compile(r"\s+")

BRANCH_SPECS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
        "snapshot_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1_precanonical_snapshot.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
        "snapshot_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1_precanonical_snapshot.jsonl",
    },
]


def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def write_jsonl(rows, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def normalize_prompt_text(text: str) -> str:
    return NORMALIZE_RE.sub(" ", str(text).strip().lower())


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return str(meta.get("content_hash") or meta.get("circuit_hash") or "").strip()


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return str(
        meta.get("paraphrase_source_content_hash")
        or meta.get("paraphrase_source")
        or meta.get("circuit_hash")
        or ""
    ).strip()


def analyze_branch(spec: dict) -> dict:
    seed_rows = load_jsonl(spec["seed_file"])
    artifact_rows = load_jsonl(spec["artifact_file"])

    seed_map = {}
    for row in seed_rows:
        key = seed_key(row)
        if key:
            seed_map[key] = row

    global_norm_counts = Counter()
    for row in artifact_rows:
        global_norm_counts[normalize_prompt_text(row.get("input", ""))] += 1

    kept_rows = []
    kept_counts = Counter()
    seen_norms = set()
    dropped_duplicate_rows = 0
    dropped_overflow_rows = 0

    for row in artifact_rows:
        source_id = paraphrase_source_key(row)
        if not source_id:
            continue
        norm = normalize_prompt_text(row.get("input", ""))
        if kept_counts[source_id] >= EXPECTED_PARAPHRASES:
            dropped_overflow_rows += 1
            continue
        if norm in seen_norms:
            dropped_duplicate_rows += 1
            continue
        kept_rows.append(row)
        kept_counts[source_id] += 1
        seen_norms.add(norm)

    deficits = {}
    for source_id in seed_map:
        have = kept_counts.get(source_id, 0)
        if have < EXPECTED_PARAPHRASES:
            deficits[source_id] = EXPECTED_PARAPHRASES - have

    return {
        "seed_map": seed_map,
        "artifact_rows": artifact_rows,
        "kept_rows": kept_rows,
        "kept_counts": kept_counts,
        "deficits": deficits,
        "raw_row_count": len(artifact_rows),
        "canonical_row_count": len(kept_rows),
        "duplicate_groups": sum(1 for v in global_norm_counts.values() if v > 1),
        "duplicate_extra_rows": sum(v - 1 for v in global_norm_counts.values() if v > 1),
        "dropped_duplicate_rows": dropped_duplicate_rows,
        "dropped_overflow_rows": dropped_overflow_rows,
        "represented_seeds": sum(1 for source_id in seed_map if kept_counts.get(source_id, 0) > 0),
    }


analyses = {spec["label"]: analyze_branch(spec) for spec in BRANCH_SPECS}

print("Pre-rewrite canonical check:")
for spec in BRANCH_SPECS:
    label = spec["label"]
    analysis = analyses[label]
    print(
        f"  {label}: raw_rows={analysis['raw_row_count']:,}, "
        f"canonical_rows={analysis['canonical_row_count']:,}, "
        f"represented={analysis['represented_seeds']:,}/{len(analysis['seed_map']):,}, "
        f"duplicate_groups={analysis['duplicate_groups']:,}, "
        f"duplicate_extra_rows={analysis['duplicate_extra_rows']:,}, "
        f"residual_slots={sum(analysis['deficits'].values()):,}"
    )

remaining_deficit = sum(sum(a["deficits"].values()) for a in analyses.values())
if remaining_deficit != 0:
    raise RuntimeError(
        f"Canonical closure is not complete yet. Residual slots remaining: {remaining_deficit:,}. "
        "Do not rewrite the artifacts yet."
    )

print("\nCanonical closure confirmed. Rewriting lean canonical artifacts...")

for spec in BRANCH_SPECS:
    analysis = analyses[spec["label"]]
    shutil.copy2(spec["artifact_file"], spec["snapshot_file"])
    write_jsonl(analysis["kept_rows"], spec["artifact_file"])
    print(f"  snapshot: {spec['snapshot_file']}")
    print(f"  rewritten: {spec['artifact_file']}")
    print(f"  canonical rows: {len(analysis['kept_rows']):,}")
    print(f"  dropped duplicate rows: {analysis['dropped_duplicate_rows']:,}")
    print(f"  dropped overflow rows: {analysis['dropped_overflow_rows']:,}")

print("\nFinal canonical artifact sizes:")
for spec in BRANCH_SPECS:
    analysis = analyses[spec["label"]]
    print(f"  {spec['label']}: {analysis['canonical_row_count']:,}")

print("\nNext: rerun Step J-B5 to record the final audited canonical totals.")


#### Stage J-Audit-Final — Final Canonical Paraphrase Audit

This additive audit cell reruns the final paraphrase summary directly after the lean canonical rewrite, so the notebook records the final branch totals without requiring a scroll back to the earlier `J-B5` cell.

What this cell does:
- audits the canonical `source_code` and `teacher_text` paraphrase artifacts
- reports exact normalized duplicates
- reports per-seed paraphrase coverage
- reports full-corpus paraphrase totals

What this cell does not do:
- it does not call the API
- it does not generate or materialize any paraphrases
- it does not modify any artifact


In [ ]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

NORMALIZE_RE = re.compile(r"\s+")

FINAL_AUDIT_SPECS = [
    {
        "label": "source_code",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
    },
    {
        "label": "teacher_text",
        "seed_file": PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
        "artifact_file": PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
    },
]


def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def normalize_text(text: str) -> str:
    return NORMALIZE_RE.sub(" ", str(text).strip().lower())


def seed_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return str(meta.get("content_hash") or meta.get("circuit_hash") or "").strip()


def paraphrase_source_key(row: dict) -> str:
    meta = row.get("metadata", {})
    return str(
        meta.get("paraphrase_source_content_hash")
        or meta.get("paraphrase_source")
        or meta.get("circuit_hash")
        or ""
    ).strip()


def audit_branch(spec: dict) -> dict:
    seed_rows = load_jsonl(spec["seed_file"])
    artifact_rows = load_jsonl(spec["artifact_file"])

    role_counts = Counter()
    prompt_type_counts = Counter()
    coverage = Counter()
    by_source = defaultdict(list)
    norm_counts = Counter()

    for row in artifact_rows:
        meta = row.get("metadata", {})
        role_counts[meta.get("seed_role", "<missing>")] += 1
        prompt_type_counts[meta.get("prompt_type", "<missing>")] += 1
        source_id = paraphrase_source_key(row)
        if source_id:
            by_source[source_id].append(row)
        norm_counts[normalize_text(row.get("input", ""))] += 1

    for row in seed_rows:
        coverage[len(by_source.get(seed_key(row), []))] += 1

    return {
        "seed_rows": len(seed_rows),
        "artifact_rows": len(artifact_rows),
        "role_counts": role_counts,
        "prompt_type_counts": prompt_type_counts,
        "coverage": coverage,
        "exact_normalized_duplicates": sum(v - 1 for v in norm_counts.values() if v > 1),
    }


full_seed_rows = 0
full_artifact_rows = 0

for spec in FINAL_AUDIT_SPECS:
    summary = audit_branch(spec)
    full_seed_rows += summary["seed_rows"]
    full_artifact_rows += summary["artifact_rows"]

    print(f"Branch: {spec['label']}")
    print(f"  paraphrase rows: {summary['artifact_rows']:,}")
    print(f"  source seed rows: {summary['seed_rows']:,}")
    print(f"  exact normalized duplicates: {summary['exact_normalized_duplicates']:,}")

    print("  role counts:")
    for key, value in summary["role_counts"].most_common():
        print(f"    {key}: {value:,}")

    print("  prompt types:")
    for key, value in summary["prompt_type_counts"].most_common():
        print(f"    {key}: {value:,}")

    print("  paraphrase coverage by source seed:")
    for key, value in sorted(summary["coverage"].items()):
        print(f"    {key} paraphrases: {value:,} source seeds")

    print()

print("Full-corpus paraphrase totals")
print(f"  branch seed rows: {full_seed_rows:,}")
print(f"  branch paraphrase rows: {full_artifact_rows:,}")


## Stage K — Critique / Rewrite And Acceptance Gate

This stage starts the post-Stage-J review path without changing the canonical seed or paraphrase artifacts.

What this stage does first:
- builds one unified acceptance-gate manifest from the canonical seed and paraphrase artifacts
- records compact provenance needed for later critique / rewrite review
- audits the resulting review corpus by branch, instruction kind, role, supervision mode, and prompt provenance

What this stage does not do yet:
- it does not call the API
- it does not accept or reject rows yet
- it does not rewrite the Stage J canonical artifacts
- it does not refresh paraphrases from reviewed seeds yet


#### Step K1 — Build Unified Acceptance-Gate Manifest

This additive local step builds a single review-ready manifest from the canonical `source_code` and `teacher_text` seed/paraphrase artifacts.

Why this step exists:
- Stage J closes transport, materialization, duplicate remediation, and paraphrase-slot coverage
- the next methodological object is no longer a branch-specific batch file, but a unified review corpus for critique / rewrite and acceptance decisions
- this step keeps that transition explicit and reproducible

Outputs:
- `instruction_acceptance_gate_manifest_v1.jsonl`
- `instruction_acceptance_gate_manifest_v1_summary.json`

What this step does not do:
- it does not call the API
- it does not modify the canonical Stage J artifacts


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", ROOT / "PQID/scripts/03_instruction_generation")

BUILD_ACCEPTANCE_GATE_MANIFEST_SCRIPT = globals().get(
    "BUILD_ACCEPTANCE_GATE_MANIFEST_SCRIPT",
    SCRIPTS_DIR / "build_instruction_acceptance_gate_manifest.py",
)
INSTRUCTION_ACCEPTANCE_GATE_MANIFEST = globals().get(
    "INSTRUCTION_ACCEPTANCE_GATE_MANIFEST",
    PROCESSED_DIR / "instruction_acceptance_gate_manifest_v1.jsonl",
)
INSTRUCTION_ACCEPTANCE_GATE_SUMMARY = globals().get(
    "INSTRUCTION_ACCEPTANCE_GATE_SUMMARY",
    PROCESSED_DIR / "instruction_acceptance_gate_manifest_v1_summary.json",
)

cmd = [
    sys.executable,
    str(BUILD_ACCEPTANCE_GATE_MANIFEST_SCRIPT),
    "--manifest-file", str(INSTRUCTION_ACCEPTANCE_GATE_MANIFEST),
    "--summary-file", str(INSTRUCTION_ACCEPTANCE_GATE_SUMMARY),
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("acceptance-gate manifest build completed with return code:", result.returncode)

globals()["BUILD_ACCEPTANCE_GATE_MANIFEST_SCRIPT"] = BUILD_ACCEPTANCE_GATE_MANIFEST_SCRIPT
globals()["INSTRUCTION_ACCEPTANCE_GATE_MANIFEST"] = INSTRUCTION_ACCEPTANCE_GATE_MANIFEST
globals()["INSTRUCTION_ACCEPTANCE_GATE_SUMMARY"] = INSTRUCTION_ACCEPTANCE_GATE_SUMMARY


#### Step K2 — Audit Acceptance-Gate Manifest

This local audit step records the size and composition of the review corpus produced by `K1`.

What this step reports:
- total acceptance-gate rows
- branch totals
- seed versus paraphrase counts
- role and supervision-mode distribution
- prompt-type and paraphrase prompt-mode distribution

Why this matters:
- the acceptance-gate stage should be auditable as its own corpus object
- late-stage repair provenance such as `anti_template` paraphrase closure should remain visible rather than silently folded into the ordinary bulk paraphrase population


In [ ]:
import json
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
INSTRUCTION_ACCEPTANCE_GATE_SUMMARY = globals().get(
    "INSTRUCTION_ACCEPTANCE_GATE_SUMMARY",
    PROCESSED_DIR / "instruction_acceptance_gate_manifest_v1_summary.json",
)

if not INSTRUCTION_ACCEPTANCE_GATE_SUMMARY.exists():
    raise FileNotFoundError(
        f"Acceptance-gate summary file not found: {INSTRUCTION_ACCEPTANCE_GATE_SUMMARY}"
    )

summary = json.loads(INSTRUCTION_ACCEPTANCE_GATE_SUMMARY.read_text(encoding="utf-8"))

print("Acceptance-gate manifest")
print(f"  manifest version: {summary['manifest_version']}")
print(f"  acceptance gate version: {summary['acceptance_gate_version']}")
print(f"  manifest file: {summary['manifest_file']}")
print(f"  total rows: {summary['total_rows']:,}")

print("  branch counts:")
for key, value in summary["branch_counts"].items():
    print(f"    {key}: {value:,}")

print("  instruction kinds:")
for key, value in summary["instruction_kind_counts"].items():
    print(f"    {key}: {value:,}")

print("  branch × instruction kind:")
for branch, counts in summary["branch_instruction_kind_counts"].items():
    joined = ", ".join(f"{kind}={value:,}" for kind, value in counts.items())
    print(f"    {branch}: {joined}")

print("  role counts:")
for key, value in summary["role_counts"].items():
    print(f"    {key}: {value:,}")

print("  supervision modes:")
for key, value in summary["supervision_mode_counts"].items():
    print(f"    {key}: {value:,}")

print("  prompt types:")
for key, value in summary["prompt_type_counts"].items():
    print(f"    {key}: {value:,}")

print("  paraphrase prompt modes:")
for key, value in summary["prompt_mode_counts"].items():
    print(f"    {key}: {value:,}")


#### Step K3 — Build Stratified Acceptance-Gate Pilot

This local support cell builds a small, deterministic pilot review set from the full acceptance-gate manifest.

Why this step comes before a full critique/rewrite pass:
- the full manifest contains `550,314` rows and is too large to review blindly as the first acceptance-gate action
- the pilot lets us test the acceptance rubric across branch, instruction kind, and role before committing to a larger review strategy
- it force-includes the rare `anti_template` final-tail rows so late-stage repair provenance is explicitly represented

Default sampling rule:
- up to `25` rows per observed `(source_branch, instruction_kind, seed_role)` stratum
- deterministic random seed `42`
- all `anti_template` rows are included even if already sampled

What this cell does not do:
- it does not call the API
- it does not decide accept/rewrite/reject outcomes
- it only creates a pilot manifest for later manual or model-assisted acceptance review


In [ ]:
import json
import random
from collections import Counter, defaultdict
from copy import deepcopy
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
INSTRUCTION_ACCEPTANCE_GATE_MANIFEST = globals().get(
    "INSTRUCTION_ACCEPTANCE_GATE_MANIFEST",
    PROCESSED_DIR / "instruction_acceptance_gate_manifest_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_SUMMARY = globals().get(
    "ACCEPTANCE_GATE_PILOT_SUMMARY",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1_summary.json",
)

ACCEPTANCE_GATE_PILOT_PER_STRATUM = globals().get(
    "ACCEPTANCE_GATE_PILOT_PER_STRATUM",
    25,
)
ACCEPTANCE_GATE_PILOT_RANDOM_SEED = globals().get(
    "ACCEPTANCE_GATE_PILOT_RANDOM_SEED",
    42,
)

if not INSTRUCTION_ACCEPTANCE_GATE_MANIFEST.exists():
    raise FileNotFoundError(
        f"Acceptance-gate manifest not found: {INSTRUCTION_ACCEPTANCE_GATE_MANIFEST}"
    )

rng = random.Random(ACCEPTANCE_GATE_PILOT_RANDOM_SEED)
reservoirs = defaultdict(list)
stratum_seen = Counter()
forced_rows = {}
manifest_rows = 0

def review_context(row: dict) -> dict:
    return row.get("review_context", {}) or {}

def seed_role(row: dict) -> str:
    return str(review_context(row).get("seed_role") or "<missing>")

def prompt_mode(row: dict) -> str:
    return str(review_context(row).get("paraphrase_generation_prompt_mode") or "<none>")

def stratum_key(row: dict) -> tuple[str, str, str]:
    return (
        str(row.get("source_branch") or "<missing>"),
        str(row.get("instruction_kind") or "<missing>"),
        seed_role(row),
    )

def instruction_key(row: dict) -> str:
    return str(row.get("instruction_key") or "")

with INSTRUCTION_ACCEPTANCE_GATE_MANIFEST.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        row = json.loads(line)
        manifest_rows += 1
        key = stratum_key(row)
        stratum_seen[key] += 1

        bucket = reservoirs[key]
        seen = stratum_seen[key]
        if len(bucket) < ACCEPTANCE_GATE_PILOT_PER_STRATUM:
            bucket.append(row)
        else:
            replacement_index = rng.randrange(seen)
            if replacement_index < ACCEPTANCE_GATE_PILOT_PER_STRATUM:
                bucket[replacement_index] = row

        if prompt_mode(row) == "anti_template":
            forced_rows[instruction_key(row)] = row

selected = {}
selection_reasons = {}
for key, rows in reservoirs.items():
    for row in rows:
        row_key = instruction_key(row)
        selected[row_key] = row
        selection_reasons[row_key] = "stratified_reservoir_sample"

for row_key, row in forced_rows.items():
    selected[row_key] = row
    selection_reasons[row_key] = (
        "forced_anti_template_tail"
        if selection_reasons.get(row_key) is None
        else selection_reasons[row_key] + "+forced_anti_template_tail"
    )

pilot_rows = []
for row_key, row in selected.items():
    out = deepcopy(row)
    context = out.setdefault("pilot_context", {})
    stratum = stratum_key(out)
    context.update({
        "acceptance_gate_pilot_version": "instruction_acceptance_gate_pilot_v1",
        "pilot_selection_reason": selection_reasons[row_key],
        "pilot_stratum_source_branch": stratum[0],
        "pilot_stratum_instruction_kind": stratum[1],
        "pilot_stratum_seed_role": stratum[2],
        "pilot_sample_per_stratum": ACCEPTANCE_GATE_PILOT_PER_STRATUM,
        "pilot_random_seed": ACCEPTANCE_GATE_PILOT_RANDOM_SEED,
    })
    out.setdefault("acceptance_decision", None)
    out.setdefault("acceptance_decision_reason", "")
    out.setdefault("acceptance_rewrite_required", None)
    out.setdefault("acceptance_reviewer_notes", "")
    pilot_rows.append(out)

pilot_rows.sort(
    key=lambda row: (
        row.get("source_branch", ""),
        row.get("instruction_kind", ""),
        seed_role(row),
        instruction_key(row),
    )
)

branch_counts = Counter(row.get("source_branch", "<missing>") for row in pilot_rows)
kind_counts = Counter(row.get("instruction_kind", "<missing>") for row in pilot_rows)
role_counts = Counter(seed_role(row) for row in pilot_rows)
prompt_mode_counts = Counter(prompt_mode(row) for row in pilot_rows)
stratum_counts = Counter("|".join(stratum_key(row)) for row in pilot_rows)
selection_counts = Counter(
    row.get("pilot_context", {}).get("pilot_selection_reason", "<missing>")
    for row in pilot_rows
)

with ACCEPTANCE_GATE_PILOT_FILE.open("w", encoding="utf-8") as handle:
    for row in pilot_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "pilot_version": "instruction_acceptance_gate_pilot_v1",
    "source_manifest": str(INSTRUCTION_ACCEPTANCE_GATE_MANIFEST.relative_to(ROOT)).replace("\\", "/"),
    "pilot_file": str(ACCEPTANCE_GATE_PILOT_FILE.relative_to(ROOT)).replace("\\", "/"),
    "manifest_rows_scanned": manifest_rows,
    "pilot_rows": len(pilot_rows),
    "sample_per_stratum": ACCEPTANCE_GATE_PILOT_PER_STRATUM,
    "random_seed": ACCEPTANCE_GATE_PILOT_RANDOM_SEED,
    "observed_strata": len(stratum_seen),
    "forced_anti_template_rows": len(forced_rows),
    "branch_counts": dict(sorted(branch_counts.items())),
    "instruction_kind_counts": dict(sorted(kind_counts.items())),
    "role_counts": dict(sorted(role_counts.items())),
    "prompt_mode_counts": dict(sorted(prompt_mode_counts.items())),
    "selection_reason_counts": dict(sorted(selection_counts.items())),
    "stratum_counts": dict(sorted(stratum_counts.items())),
}
ACCEPTANCE_GATE_PILOT_SUMMARY.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Acceptance-gate pilot built")
print(f"  source manifest rows scanned: {manifest_rows:,}")
print(f"  observed strata: {len(stratum_seen):,}")
print(f"  sample per stratum: {ACCEPTANCE_GATE_PILOT_PER_STRATUM:,}")
print(f"  forced anti-template rows: {len(forced_rows):,}")
print(f"  pilot rows written: {len(pilot_rows):,}")
print(f"  pilot file: {ACCEPTANCE_GATE_PILOT_FILE}")
print(f"  summary file: {ACCEPTANCE_GATE_PILOT_SUMMARY}")

globals()["ACCEPTANCE_GATE_PILOT_FILE"] = ACCEPTANCE_GATE_PILOT_FILE
globals()["ACCEPTANCE_GATE_PILOT_SUMMARY"] = ACCEPTANCE_GATE_PILOT_SUMMARY
globals()["ACCEPTANCE_GATE_PILOT_PER_STRATUM"] = ACCEPTANCE_GATE_PILOT_PER_STRATUM
globals()["ACCEPTANCE_GATE_PILOT_RANDOM_SEED"] = ACCEPTANCE_GATE_PILOT_RANDOM_SEED


#### Step K4 — Audit Stratified Acceptance-Gate Pilot

This local audit cell reports the pilot review set produced by `K3`.

What this step checks:
- total pilot rows
- branch, role, and instruction-kind coverage
- stratum counts for `(source_branch, instruction_kind, seed_role)`
- whether `anti_template` tail rows are present
- whether all pilot rows remain in `pending` acceptance-review status

This is the final sanity check before designing the actual critique/rewrite or human review pass.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ACCEPTANCE_GATE_PILOT_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_SUMMARY = globals().get(
    "ACCEPTANCE_GATE_PILOT_SUMMARY",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1_summary.json",
)

if not ACCEPTANCE_GATE_PILOT_FILE.exists():
    raise FileNotFoundError(f"Pilot file not found: {ACCEPTANCE_GATE_PILOT_FILE}")
if not ACCEPTANCE_GATE_PILOT_SUMMARY.exists():
    raise FileNotFoundError(f"Pilot summary not found: {ACCEPTANCE_GATE_PILOT_SUMMARY}")

summary = json.loads(ACCEPTANCE_GATE_PILOT_SUMMARY.read_text(encoding="utf-8"))
rows = []
with ACCEPTANCE_GATE_PILOT_FILE.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            rows.append(json.loads(line))

def review_context(row: dict) -> dict:
    return row.get("review_context", {}) or {}

def seed_role(row: dict) -> str:
    return str(review_context(row).get("seed_role") or "<missing>")

def prompt_mode(row: dict) -> str:
    return str(review_context(row).get("paraphrase_generation_prompt_mode") or "<none>")

branch_counts = Counter(row.get("source_branch", "<missing>") for row in rows)
kind_counts = Counter(row.get("instruction_kind", "<missing>") for row in rows)
role_counts = Counter(seed_role(row) for row in rows)
prompt_mode_counts = Counter(prompt_mode(row) for row in rows)
review_status_counts = Counter(row.get("acceptance_review_status", "<missing>") for row in rows)
decision_counts = Counter(str(row.get("acceptance_decision")) for row in rows)
stratum_counts = Counter(
    (
        row.get("source_branch", "<missing>"),
        row.get("instruction_kind", "<missing>"),
        seed_role(row),
    )
    for row in rows
)

print("Acceptance-gate pilot audit")
print(f"  pilot version: {summary['pilot_version']}")
print(f"  source manifest: {summary['source_manifest']}")
print(f"  pilot file: {summary['pilot_file']}")
print(f"  manifest rows scanned: {summary['manifest_rows_scanned']:,}")
print(f"  pilot rows: {len(rows):,}")
print(f"  observed strata: {summary['observed_strata']:,}")
print(f"  sample per stratum: {summary['sample_per_stratum']:,}")
print(f"  random seed: {summary['random_seed']}")

print("  branch counts:")
for key, value in sorted(branch_counts.items()):
    print(f"    {key}: {value:,}")

print("  instruction kind counts:")
for key, value in sorted(kind_counts.items()):
    print(f"    {key}: {value:,}")

print("  role counts:")
for key, value in sorted(role_counts.items()):
    print(f"    {key}: {value:,}")

print("  paraphrase prompt modes:")
for key, value in sorted(prompt_mode_counts.items()):
    print(f"    {key}: {value:,}")

print("  review status counts:")
for key, value in sorted(review_status_counts.items()):
    print(f"    {key}: {value:,}")

print("  acceptance decision placeholders:")
for key, value in sorted(decision_counts.items()):
    print(f"    {key}: {value:,}")

print("  stratum counts:")
for key, value in sorted(stratum_counts.items()):
    print(f"    {key[0]} | {key[1]} | {key[2]}: {value:,}")

if prompt_mode_counts.get("anti_template", 0) == 0:
    raise RuntimeError("Pilot audit failed: anti_template tail rows are missing.")
if review_status_counts.get("pending", 0) != len(rows):
    raise RuntimeError("Pilot audit failed: not all rows are still pending review.")

print("\nPilot audit passed. Next step: design the pilot acceptance rubric / critique pass.")


#### Step K5 — Build Pilot Acceptance Review Sheet

This local support cell converts the pilot manifest into a compact spreadsheet-friendly review sheet.

Default review conventions:
- `acceptance_decision`: `accept`, `rewrite`, `reject`, `defer`
- rubric axes: `pass`, `minor_issue`, `major_issue`, `n_a`
- `teacher_text_answer_quality` should usually be marked `n_a` for `source_code` rows

What this cell writes:
- one UTF-8 CSV review sheet with the pilot rows plus empty rubric columns

What this cell does not do:
- it does not call the API
- it does not fill review decisions automatically
- it only prepares the pilot for manual or later model-assisted review


In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ACCEPTANCE_GATE_PILOT_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_REVIEW_SHEET = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEW_SHEET",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_review_sheet_v1.csv",
)

if not ACCEPTANCE_GATE_PILOT_FILE.exists():
    raise FileNotFoundError(f"Pilot file not found: {ACCEPTANCE_GATE_PILOT_FILE}")

def review_context(row: dict) -> dict:
    return row.get("review_context", {}) or {}

def pilot_context(row: dict) -> dict:
    return row.get("pilot_context", {}) or {}

rows = []
with ACCEPTANCE_GATE_PILOT_FILE.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            rows.append(json.loads(line))

fieldnames = [
    "pilot_row_index",
    "instruction_key",
    "review_group_key",
    "source_branch",
    "instruction_kind",
    "seed_role",
    "seed_target_supervision_mode",
    "expected_response_mode",
    "prompt_type",
    "paraphrase_prompt_mode",
    "review_priority",
    "validation_status",
    "repo_owner",
    "repo_name",
    "file_path",
    "original_url",
    "pilot_selection_reason",
    "input",
    "output",
    "acceptance_review_status",
    "acceptance_decision",
    "acceptance_rewrite_required",
    "role_fidelity",
    "semantic_grounding",
    "confidence_discipline",
    "hallucination_risk",
    "teacher_text_answer_quality",
    "reviewer_notes",
    "rewrite_guidance",
]

review_rows = []
for idx, row in enumerate(rows, start=1):
    ctx = review_context(row)
    pctx = pilot_context(row)
    prompt_mode = str(ctx.get("paraphrase_generation_prompt_mode") or "<none>")
    review_rows.append({
        "pilot_row_index": idx,
        "instruction_key": row.get("instruction_key", ""),
        "review_group_key": row.get("review_group_key", ""),
        "source_branch": row.get("source_branch", ""),
        "instruction_kind": row.get("instruction_kind", ""),
        "seed_role": ctx.get("seed_role", ""),
        "seed_target_supervision_mode": ctx.get("seed_target_supervision_mode", ""),
        "expected_response_mode": ctx.get("expected_response_mode", ""),
        "prompt_type": ctx.get("prompt_type", ""),
        "paraphrase_prompt_mode": prompt_mode,
        "review_priority": "high_tail_case" if prompt_mode == "anti_template" else "normal",
        "validation_status": ctx.get("validation_status", ""),
        "repo_owner": ctx.get("repo_owner", ""),
        "repo_name": ctx.get("repo_name", ""),
        "file_path": ctx.get("file_path", ""),
        "original_url": ctx.get("original_url", ""),
        "pilot_selection_reason": pctx.get("pilot_selection_reason", ""),
        "input": row.get("input", ""),
        "output": row.get("output", ""),
        "acceptance_review_status": "pending",
        "acceptance_decision": "",
        "acceptance_rewrite_required": "",
        "role_fidelity": "",
        "semantic_grounding": "",
        "confidence_discipline": "",
        "hallucination_risk": "",
        "teacher_text_answer_quality": "",
        "reviewer_notes": "",
        "rewrite_guidance": "",
    })

with ACCEPTANCE_GATE_PILOT_REVIEW_SHEET.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(review_rows)

branch_counts = Counter(row["source_branch"] for row in review_rows)
role_counts = Counter(row["seed_role"] for row in review_rows)
priority_counts = Counter(row["review_priority"] for row in review_rows)

print("Acceptance-gate pilot review sheet built")
print(f"  pilot rows loaded: {len(rows):,}")
print(f"  review sheet rows written: {len(review_rows):,}")
print(f"  review sheet: {ACCEPTANCE_GATE_PILOT_REVIEW_SHEET}")
print("  branch counts:")
for key, value in sorted(branch_counts.items()):
    print(f"    {key}: {value:,}")
print("  role counts:")
for key, value in sorted(role_counts.items()):
    print(f"    {key}: {value:,}")
print("  review priorities:")
for key, value in sorted(priority_counts.items()):
    print(f"    {key}: {value:,}")

globals()["ACCEPTANCE_GATE_PILOT_REVIEW_SHEET"] = ACCEPTANCE_GATE_PILOT_REVIEW_SHEET


#### Step K6 — Audit Pilot Review Sheet Readiness

This local audit cell verifies that the CSV review sheet produced by `K5` is structurally ready for review.

What this step checks:
- row count matches the pilot manifest
- required review columns exist
- all rows remain `pending`
- decision and rubric fields are still blank placeholders
- `anti_template` high-priority rows are preserved
- each row still has non-empty `input` and `output`


In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ACCEPTANCE_GATE_PILOT_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_REVIEW_SHEET = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEW_SHEET",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_review_sheet_v1.csv",
)

if not ACCEPTANCE_GATE_PILOT_FILE.exists():
    raise FileNotFoundError(f"Pilot file not found: {ACCEPTANCE_GATE_PILOT_FILE}")
if not ACCEPTANCE_GATE_PILOT_REVIEW_SHEET.exists():
    raise FileNotFoundError(f"Review sheet not found: {ACCEPTANCE_GATE_PILOT_REVIEW_SHEET}")

with ACCEPTANCE_GATE_PILOT_FILE.open("r", encoding="utf-8") as handle:
    pilot_rows = [json.loads(line) for line in handle if line.strip()]

with ACCEPTANCE_GATE_PILOT_REVIEW_SHEET.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    fieldnames = reader.fieldnames or []
    review_rows = list(reader)

required_columns = [
    "pilot_row_index",
    "instruction_key",
    "source_branch",
    "instruction_kind",
    "seed_role",
    "paraphrase_prompt_mode",
    "review_priority",
    "input",
    "output",
    "acceptance_review_status",
    "acceptance_decision",
    "acceptance_rewrite_required",
    "role_fidelity",
    "semantic_grounding",
    "confidence_discipline",
    "hallucination_risk",
    "teacher_text_answer_quality",
    "reviewer_notes",
    "rewrite_guidance",
]
missing_columns = [column for column in required_columns if column not in fieldnames]
if missing_columns:
    raise RuntimeError(f"Review sheet audit failed: missing columns: {missing_columns}")

if len(review_rows) != len(pilot_rows):
    raise RuntimeError(
        f"Review sheet audit failed: row count mismatch ({len(review_rows)} vs {len(pilot_rows)})."
    )

blank_rubric_fields = [
    "acceptance_decision",
    "acceptance_rewrite_required",
    "role_fidelity",
    "semantic_grounding",
    "confidence_discipline",
    "hallucination_risk",
    "teacher_text_answer_quality",
    "reviewer_notes",
    "rewrite_guidance",
]

status_counts = Counter(row.get("acceptance_review_status", "<missing>") for row in review_rows)
decision_counts = Counter((row.get("acceptance_decision", "") or "<blank>") for row in review_rows)
branch_counts = Counter(row.get("source_branch", "<missing>") for row in review_rows)
kind_counts = Counter(row.get("instruction_kind", "<missing>") for row in review_rows)
role_counts = Counter(row.get("seed_role", "<missing>") for row in review_rows)
prompt_mode_counts = Counter(row.get("paraphrase_prompt_mode", "<missing>") for row in review_rows)
priority_counts = Counter(row.get("review_priority", "<missing>") for row in review_rows)
blank_counts = Counter()

for row in review_rows:
    if not (row.get("input", "") or "").strip():
        raise RuntimeError("Review sheet audit failed: empty input field detected.")
    if not (row.get("output", "") or "").strip():
        raise RuntimeError("Review sheet audit failed: empty output field detected.")
    for field in blank_rubric_fields:
        if not (row.get(field, "") or "").strip():
            blank_counts[field] += 1

if status_counts.get("pending", 0) != len(review_rows):
    raise RuntimeError("Review sheet audit failed: not all rows are still pending.")
if prompt_mode_counts.get("anti_template", 0) == 0:
    raise RuntimeError("Review sheet audit failed: anti_template rows are missing.")
for field in blank_rubric_fields:
    if blank_counts[field] != len(review_rows):
        raise RuntimeError(f"Review sheet audit failed: field '{field}' is not blank-initialized.")

print("Acceptance-gate pilot review sheet audit")
print(f"  review sheet: {ACCEPTANCE_GATE_PILOT_REVIEW_SHEET}")
print(f"  rows: {len(review_rows):,}")
print(f"  columns: {len(fieldnames):,}")
print("  branch counts:")
for key, value in sorted(branch_counts.items()):
    print(f"    {key}: {value:,}")
print("  instruction kind counts:")
for key, value in sorted(kind_counts.items()):
    print(f"    {key}: {value:,}")
print("  role counts:")
for key, value in sorted(role_counts.items()):
    print(f"    {key}: {value:,}")
print("  paraphrase prompt modes:")
for key, value in sorted(prompt_mode_counts.items()):
    print(f"    {key}: {value:,}")
print("  review priorities:")
for key, value in sorted(priority_counts.items()):
    print(f"    {key}: {value:,}")
print("  review status counts:")
for key, value in sorted(status_counts.items()):
    print(f"    {key}: {value:,}")
print("  acceptance decision placeholders:")
for key, value in sorted(decision_counts.items()):
    print(f"    {key}: {value:,}")

print("\nReview sheet readiness audit passed. Next step: fill the pilot rubric or build the model-assisted critique pass.")


#### Step K7 — Import Completed Pilot Review Sheet

This local import cell merges the filled CSV review sheet back into a structured JSONL pilot-review artifact.

What this step does:
- reads the manually reviewed pilot CSV from `K5`
- aligns it back to the pilot manifest by `instruction_key`
- stores the review decisions, rubric values, and notes in a structured JSONL file
- writes a compact summary JSON for quick auditing

What this step does not do:
- it does not call the API
- it does not rewrite the full acceptance-gate manifest
- it only imports the pilot review outcomes into an auditable structured layer


In [ ]:
import csv
import json
from collections import Counter
from copy import deepcopy
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ACCEPTANCE_GATE_PILOT_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_REVIEW_SHEET = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEW_SHEET",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_review_sheet_v1.csv",
)
ACCEPTANCE_GATE_PILOT_REVIEWED_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEWED_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_reviewed_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_reviewed_v1_summary.json",
)

if not ACCEPTANCE_GATE_PILOT_FILE.exists():
    raise FileNotFoundError(f"Pilot file not found: {ACCEPTANCE_GATE_PILOT_FILE}")
if not ACCEPTANCE_GATE_PILOT_REVIEW_SHEET.exists():
    raise FileNotFoundError(f"Review sheet not found: {ACCEPTANCE_GATE_PILOT_REVIEW_SHEET}")

def norm(value) -> str:
    return str(value or "").strip()

with ACCEPTANCE_GATE_PILOT_FILE.open("r", encoding="utf-8") as handle:
    pilot_rows = [json.loads(line) for line in handle if line.strip()]
pilot_by_key = {}
for row in pilot_rows:
    key = norm(row.get("instruction_key"))
    if not key:
        raise RuntimeError("Pilot import failed: missing instruction_key in pilot manifest.")
    pilot_by_key[key] = row

review_rows = []
with ACCEPTANCE_GATE_PILOT_REVIEW_SHEET.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        review_rows.append(row)

reviewed_rows = []
decision_counts = Counter()
rewrite_required_counts = Counter()
branch_counts = Counter()
role_counts = Counter()
priority_counts = Counter()
prompt_mode_counts = Counter()
status_counts = Counter()
rubric_value_counts = {
    "role_fidelity": Counter(),
    "semantic_grounding": Counter(),
    "confidence_discipline": Counter(),
    "hallucination_risk": Counter(),
    "teacher_text_answer_quality": Counter(),
}

for csv_row in review_rows:
    key = norm(csv_row.get("instruction_key"))
    if key not in pilot_by_key:
        raise RuntimeError(f"Pilot import failed: unknown instruction_key in review sheet: {key}")
    base = deepcopy(pilot_by_key[key])
    decision = norm(csv_row.get("acceptance_decision")).lower()
    rewrite_required = norm(csv_row.get("acceptance_rewrite_required")).lower()
    review_status = "reviewed" if decision else "pending"

    rubric = {
        "role_fidelity": norm(csv_row.get("role_fidelity")).lower(),
        "semantic_grounding": norm(csv_row.get("semantic_grounding")).lower(),
        "confidence_discipline": norm(csv_row.get("confidence_discipline")).lower(),
        "hallucination_risk": norm(csv_row.get("hallucination_risk")).lower(),
        "teacher_text_answer_quality": norm(csv_row.get("teacher_text_answer_quality")).lower(),
    }

    base["acceptance_review_status"] = review_status
    base["acceptance_decision"] = decision or None
    base["acceptance_rewrite_required"] = rewrite_required or None
    base["acceptance_reviewer_notes"] = norm(csv_row.get("reviewer_notes"))
    base["acceptance_rewrite_guidance"] = norm(csv_row.get("rewrite_guidance"))
    base["acceptance_rubric"] = rubric

    pctx = base.setdefault("pilot_context", {})
    pctx["review_priority"] = norm(csv_row.get("review_priority")) or pctx.get("review_priority", "")
    pctx["review_sheet_version"] = "instruction_acceptance_gate_pilot_review_sheet_v1"

    reviewed_rows.append(base)

    branch = norm(csv_row.get("source_branch")) or "<missing>"
    role = norm(csv_row.get("seed_role")) or "<missing>"
    priority = norm(csv_row.get("review_priority")) or "<missing>"
    prompt_mode = norm(csv_row.get("paraphrase_prompt_mode")) or "<missing>"
    branch_counts[branch] += 1
    role_counts[role] += 1
    priority_counts[priority] += 1
    prompt_mode_counts[prompt_mode] += 1
    status_counts[review_status] += 1
    decision_counts[decision or "<blank>"] += 1
    rewrite_required_counts[rewrite_required or "<blank>"] += 1
    for field, value in rubric.items():
        rubric_value_counts[field][value or "<blank>"] += 1

with ACCEPTANCE_GATE_PILOT_REVIEWED_FILE.open("w", encoding="utf-8") as handle:
    for row in reviewed_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "reviewed_version": "instruction_acceptance_gate_pilot_reviewed_v1",
    "source_pilot_file": str(ACCEPTANCE_GATE_PILOT_FILE.relative_to(ROOT)).replace("\\", "/"),
    "source_review_sheet": str(ACCEPTANCE_GATE_PILOT_REVIEW_SHEET.relative_to(ROOT)).replace("\\", "/"),
    "reviewed_file": str(ACCEPTANCE_GATE_PILOT_REVIEWED_FILE.relative_to(ROOT)).replace("\\", "/"),
    "rows": len(reviewed_rows),
    "review_status_counts": dict(sorted(status_counts.items())),
    "decision_counts": dict(sorted(decision_counts.items())),
    "rewrite_required_counts": dict(sorted(rewrite_required_counts.items())),
    "branch_counts": dict(sorted(branch_counts.items())),
    "role_counts": dict(sorted(role_counts.items())),
    "review_priority_counts": dict(sorted(priority_counts.items())),
    "prompt_mode_counts": dict(sorted(prompt_mode_counts.items())),
    "rubric_value_counts": {field: dict(sorted(counts.items())) for field, counts in rubric_value_counts.items()},
}
ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Acceptance-gate pilot review import completed")
print(f"  review sheet rows imported: {len(reviewed_rows):,}")
print(f"  reviewed file: {ACCEPTANCE_GATE_PILOT_REVIEWED_FILE}")
print(f"  summary file: {ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY}")
print("  review status counts:")
for key, value in sorted(status_counts.items()):
    print(f"    {key}: {value:,}")
print("  acceptance decisions:")
for key, value in sorted(decision_counts.items()):
    print(f"    {key}: {value:,}")

globals()["ACCEPTANCE_GATE_PILOT_REVIEWED_FILE"] = ACCEPTANCE_GATE_PILOT_REVIEWED_FILE
globals()["ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY"] = ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY


#### Step K8 — Audit Pilot Review Outcomes

This local audit cell summarizes the imported pilot review outcomes from `K7`.

What this step reports:
- reviewed versus still-pending pilot rows
- decision distribution (`accept`, `rewrite`, `reject`, `defer`)
- rewrite-required distribution
- rubric value counts by review axis
- whether high-priority `anti_template` tail rows have been reviewed


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
ACCEPTANCE_GATE_PILOT_REVIEWED_FILE = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEWED_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_reviewed_v1.jsonl",
)
ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY = globals().get(
    "ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_reviewed_v1_summary.json",
)

if not ACCEPTANCE_GATE_PILOT_REVIEWED_FILE.exists():
    raise FileNotFoundError(f"Reviewed pilot file not found: {ACCEPTANCE_GATE_PILOT_REVIEWED_FILE}")
if not ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY.exists():
    raise FileNotFoundError(f"Reviewed pilot summary not found: {ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY}")

summary = json.loads(ACCEPTANCE_GATE_PILOT_REVIEWED_SUMMARY.read_text(encoding="utf-8"))
rows = []
with ACCEPTANCE_GATE_PILOT_REVIEWED_FILE.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            rows.append(json.loads(line))

status_counts = Counter(row.get("acceptance_review_status", "<missing>") for row in rows)
decision_counts = Counter(str(row.get("acceptance_decision") or "<blank>") for row in rows)
rewrite_required_counts = Counter(str(row.get("acceptance_rewrite_required") or "<blank>") for row in rows)
priority_counts = Counter(row.get("pilot_context", {}).get("review_priority") or "<missing>" for row in rows)
prompt_mode_counts = Counter(
    (row.get("review_context", {}) or {}).get("paraphrase_generation_prompt_mode") or "<none>"
    for row in rows
)

rubric_fields = [
    "role_fidelity",
    "semantic_grounding",
    "confidence_discipline",
    "hallucination_risk",
    "teacher_text_answer_quality",
]
rubric_counts = {field: Counter() for field in rubric_fields}
reviewed_tail_rows = 0

for row in rows:
    rubric = row.get("acceptance_rubric", {}) or {}
    for field in rubric_fields:
        rubric_counts[field][str(rubric.get(field) or "<blank>")] += 1
    if (
        ((row.get("review_context", {}) or {}).get("paraphrase_generation_prompt_mode") or "") == "anti_template"
        and row.get("acceptance_review_status") == "reviewed"
    ):
        reviewed_tail_rows += 1

print("Acceptance-gate pilot review outcome audit")
print(f"  reviewed version: {summary['reviewed_version']}")
print(f"  source review sheet: {summary['source_review_sheet']}")
print(f"  reviewed file: {summary['reviewed_file']}")
print(f"  rows: {len(rows):,}")
print("  review status counts:")
for key, value in sorted(status_counts.items()):
    print(f"    {key}: {value:,}")
print("  acceptance decisions:")
for key, value in sorted(decision_counts.items()):
    print(f"    {key}: {value:,}")
print("  rewrite required values:")
for key, value in sorted(rewrite_required_counts.items()):
    print(f"    {key}: {value:,}")
print("  review priorities:")
for key, value in sorted(priority_counts.items()):
    print(f"    {key}: {value:,}")
print("  paraphrase prompt modes:")
for key, value in sorted(prompt_mode_counts.items()):
    print(f"    {key}: {value:,}")
print(f"  reviewed anti_template tail rows: {reviewed_tail_rows:,}")
print("  rubric value counts:")
for field in rubric_fields:
    print(f"    {field}:")
    for key, value in sorted(rubric_counts[field].items()):
        print(f"      {key}: {value:,}")

if prompt_mode_counts.get("anti_template", 0) > 0 and reviewed_tail_rows == 0:
    print("\nNote: anti_template tail rows are present but not yet reviewed.")
if status_counts.get("reviewed", 0) == 0:
    print("\nNote: no reviewed rows detected yet; fill the CSV and rerun K7/K8.")


#### Step K9 — Run Model-Assisted Pilot Review Pass

This local support cell runs a resumable model-assisted second-opinion review over the pilot sheet built in `K5`.

What this step does:
- reads the current pilot review sheet
- requests structured review suggestions from an OpenAI model
- keeps the model suggestions separate from the human review sheet
- writes a comparison-friendly CSV and a compact summary JSON
- can be rerun safely because it caches completed `instruction_key` suggestions

What this step does not do:
- it does not overwrite the human review sheet
- it does not rewrite the full acceptance-gate manifest
- it should be interpreted as a supplementary review layer, not as a replacement for human judgment


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
MODEL_ASSISTED_ACCEPTANCE_REVIEW_SCRIPT = globals().get(
    "MODEL_ASSISTED_ACCEPTANCE_REVIEW_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/run_model_assisted_acceptance_pilot_review.py",
)
MODEL_ASSISTED_REVIEW_MODEL = globals().get("MODEL_ASSISTED_REVIEW_MODEL", "gpt-5.4")
MODEL_ASSISTED_REVIEW_TEMPERATURE = globals().get("MODEL_ASSISTED_REVIEW_TEMPERATURE", 0.1)
MODEL_ASSISTED_REVIEW_MAX_OUTPUT_TOKENS = globals().get(
    "MODEL_ASSISTED_REVIEW_MAX_OUTPUT_TOKENS",
    320,
)
MODEL_ASSISTED_REVIEW_CONCURRENCY = globals().get("MODEL_ASSISTED_REVIEW_CONCURRENCY", 8)

cmd = [
    sys.executable,
    str(MODEL_ASSISTED_ACCEPTANCE_REVIEW_SCRIPT),
    "--model",
    str(MODEL_ASSISTED_REVIEW_MODEL),
    "--temperature",
    str(MODEL_ASSISTED_REVIEW_TEMPERATURE),
    "--max-output-tokens",
    str(MODEL_ASSISTED_REVIEW_MAX_OUTPUT_TOKENS),
    "--concurrency",
    str(MODEL_ASSISTED_REVIEW_CONCURRENCY),
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("model-assisted pilot review completed with return code:", result.returncode)

globals()["MODEL_ASSISTED_ACCEPTANCE_REVIEW_SCRIPT"] = MODEL_ASSISTED_ACCEPTANCE_REVIEW_SCRIPT


#### Step K10 — Audit Model-Assisted Pilot Review Suggestions

This local audit cell summarizes the model-assisted suggestion layer produced by `K9`.

What this step reports:
- model review completion status
- model decision distribution
- human/model decision agreement counts when human decisions are present
- sampled disagreement rows for targeted follow-up


In [ ]:
import json
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
MODEL_ASSISTED_REVIEW_SUMMARY_FILE = globals().get(
    "MODEL_ASSISTED_REVIEW_SUMMARY_FILE",
    PROCESSED_DIR / "instruction_acceptance_gate_pilot_model_review_summary_v1.json",
)

if not MODEL_ASSISTED_REVIEW_SUMMARY_FILE.exists():
    raise FileNotFoundError(
        f"Model-assisted review summary not found: {MODEL_ASSISTED_REVIEW_SUMMARY_FILE}"
    )

summary = json.loads(MODEL_ASSISTED_REVIEW_SUMMARY_FILE.read_text(encoding="utf-8"))

print("Model-assisted pilot review audit")
print(f"  model review version: {summary['model_review_version']}")
print(f"  source review sheet: {summary['source_review_sheet']}")
print(f"  suggestion sheet: {summary['suggestion_sheet']}")
print(f"  rows: {summary['rows']:,}")
print(f"  model: {summary['model_review_model']}")
print(f"  temperature: {summary['model_review_temperature']}")
print("  model review status counts:")
for key, value in sorted(summary.get("model_review_status_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  model acceptance decisions:")
for key, value in sorted(summary.get("model_acceptance_decision_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  human/model decision agreement:")
for key, value in sorted(summary.get("human_model_decision_agreement_counts", {}).items()):
    print(f"    {key}: {value:,}")

disagreements = summary.get("decision_disagreement_examples", [])
if disagreements:
    print("  sampled decision disagreements:")
    for row in disagreements[:10]:
        print(
            "    "
            f"pilot_row_index={row['pilot_row_index']}, "
            f"human={row['human_acceptance_decision']}, "
            f"model={row['model_acceptance_decision']}, "
            f"branch={row['source_branch']}, kind={row['instruction_kind']}, role={row['seed_role']}"
        )
else:
    print("  sampled decision disagreements: none")


## Stage K-R - Acceptance-Gate Remediation Pass

This stage closes the non-trivial rewrite tail from the Stage K acceptance pilot. It uses the final human-adjudicated `47` rewrite-required rows plus `235` same-lineage nearest risk-neighbours, yielding a bounded `282`-row remediation sidecar.

This stage does not mutate the canonical `550,314`-row acceptance-gate manifest. It produces a separate reviewable remediation layer for targeted repair, inspection, and closeout.


#### Step K-R1 - Submit Remediation Batch Job

Submit the prepared remediation Batch API request file:

`instruction_acceptance_gate_remediation_batch_requests_v1.jsonl`

This request file contains all `282` remediation candidates. The resulting batch state is saved locally so the job can be resumed, inspected, or downloaded later without recreating the request.


In [ ]:
from pathlib import Path
import subprocess, sys, json

root = Path.cwd()
while not (root / "PQID").exists() and root != root.parent:
    root = root.parent

processed = root / "PQID" / "data" / "processed"
state_file = processed / "instruction_acceptance_gate_remediation_batch_state_v1.json"

subprocess.run([
    sys.executable,
    str(root / "PQID" / "scripts" / "03_instruction_generation" / "run_openai_batch_job.py"),
    "--request-file", str(processed / "instruction_acceptance_gate_remediation_batch_requests_v1.jsonl"),
    "--state-file", str(state_file),
], check=True)

print(json.loads(state_file.read_text(encoding="utf-8"))["batch_id"])


#### Step K-R2 - Wait For Completion And Download Remediation Batch Files

Poll the submitted remediation batch until it reaches a terminal status. When complete, download the raw batch output file and any raw batch error file into `PQID/data/processed/`.

The expected output file is:

`instruction_acceptance_gate_remediation_batch_outputs_v1.jsonl`


In [ ]:
from pathlib import Path
import subprocess, sys, json

root = Path.cwd()
while not (root / "PQID").exists() and root != root.parent:
    root = root.parent

processed = root / "PQID" / "data" / "processed"
state_file = processed / "instruction_acceptance_gate_remediation_batch_state_v1.json"
batch_id = json.loads(state_file.read_text(encoding="utf-8"))["batch_id"]

subprocess.run([
    sys.executable,
    str(root / "PQID" / "scripts" / "03_instruction_generation" / "run_openai_batch_job.py"),
    "--batch-id", batch_id,
    "--state-file", str(state_file),
    "--wait",
    "--poll-interval-seconds", "60",
    "--download-output-file", str(processed / "instruction_acceptance_gate_remediation_batch_outputs_v1.jsonl"),
    "--download-error-file", str(processed / "instruction_acceptance_gate_remediation_batch_errors_raw_v1.jsonl"),
], check=True)


#### Step K-R3 - Materialize Remediation Outputs

Normalize the raw Batch API responses and join them back to the remediation candidate sidecar.

This step writes the reviewable remediation artifacts:

`instruction_acceptance_gate_remediation_outputs_v1.jsonl`

`instruction_acceptance_gate_remediation_outputs_v1.csv`

`instruction_acceptance_gate_remediation_outputs_v1_summary.json`

`instruction_acceptance_gate_remediation_errors_v1.jsonl`


In [ ]:
from pathlib import Path
import subprocess, sys

root = Path.cwd()
while not (root / "PQID").exists() and root != root.parent:
    root = root.parent

subprocess.run([
    sys.executable,
    str(root / "PQID" / "scripts" / "03_instruction_generation" / "materialize_acceptance_remediation_batch.py"),
], check=True)


#### Step K-R4 - Inspect Remediation Completion Summary

Inspect the materialization summary to verify that the remediation pass completed cleanly.

The target closeout condition is:

- `candidate_rows`: `282`
- `result_rows`: `282`
- `missing_output_count`: `0`
- no material parse failures, or only explainable residual errors


In [ ]:
from pathlib import Path
import json

root = Path.cwd()
while not (root / "PQID").exists() and root != root.parent:
    root = root.parent

summary_path = root / "PQID" / "data" / "processed" / "instruction_acceptance_gate_remediation_outputs_v1_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary


#### Step K-R5 - Apply Final Manual Closeout Overlay

Apply the final deterministic manual closeout overlay for the two lineage-neighbor rows that remained outside the normalized `rewrite` decision after batch materialization.

This step updates only the remediation sidecar outputs and summary files. It does not mutate the canonical acceptance-gate manifest or the canonical instruction splits.

The target final state is `282 / 282` materialized remediation rows, `282` final `rewrite` decisions, `2` manual closeout overrides, and `0` remaining manual-review rows.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

root = Path.cwd()
while not (root / "PQID").exists() and root != root.parent:
    root = root.parent

subprocess.run([
    sys.executable,
    str(root / "PQID" / "scripts" / "03_instruction_generation" / "finalize_acceptance_remediation_closeout.py"),
], check=True)

summary_path = root / "PQID" / "data" / "processed" / "instruction_acceptance_gate_remediation_outputs_v1_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
{
    "closeout_status": summary.get("closeout_status"),
    "result_rows": summary.get("result_rows"),
    "decision_counts": summary.get("decision_counts"),
    "manual_closeout_rows": summary.get("manual_closeout_rows"),
    "missing_output_count": summary.get("missing_output_count"),
}


## Stage L — Refresh Canonical Splits And Run Semantic Metrics

This stage refreshes the canonical split layer from the completed quality-aware seed/paraphrase artifacts and then runs the post-generation semantic analyses that were intentionally deferred until canonical closure.

What this stage does:
- rebuilds `train_clean.jsonl`, `validation_clean.jsonl`, and `test_clean.jsonl` from the canonical quality-aware artifacts
- computes per-entry semantic consistency metrics for paraphrases versus their source seeds
- computes group-level paraphrase diversity diagnostics
- audits semantic-metric coverage after enrichment

Why this stage exists now:
- the semantic consistency metrics depend on completed seed/paraphrase lineage, including `original_prompt`
- the old split files in `data/processed/` predate the final quality-aware Stage J corpus and should not be treated as the current semantic-analysis base


#### Step L1 — Refresh Canonical Split Layer From Quality-Aware Artifacts

This step rebuilds the canonical `train_clean`, `validation_clean`, and `test_clean` files directly from the current quality-aware seed and paraphrase artifacts.

Inputs:
- `seed_drafts_quality_aware_source_code_v1.jsonl`
- `seed_drafts_quality_aware_teacher_text_v1.jsonl`
- `seed_paraphrases_quality_aware_source_code_v1.jsonl`
- `seed_paraphrases_quality_aware_teacher_text_v1.jsonl`

What this step does not do:
- it does not call the API
- it does not regenerate any seed or paraphrase text
- it only refreshes the split layer so later semantic analyses operate on the current canonical corpus


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
MERGE_AND_SPLIT_SCRIPT = globals().get(
    "MERGE_AND_SPLIT_SCRIPT",
    ROOT / "PQID/scripts/merge_and_split.py",
)

QUALITY_AWARE_SOURCE_CODE_SEEDS = globals().get(
    "QUALITY_AWARE_SOURCE_CODE_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_source_code_v1.jsonl",
)
QUALITY_AWARE_TEACHER_TEXT_SEEDS = globals().get(
    "QUALITY_AWARE_TEACHER_TEXT_SEEDS",
    PROCESSED_DIR / "seed_drafts_quality_aware_teacher_text_v1.jsonl",
)
QUALITY_AWARE_SOURCE_CODE_PARAPHRASES = globals().get(
    "QUALITY_AWARE_SOURCE_CODE_PARAPHRASES",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_source_code_v1.jsonl",
)
QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES = globals().get(
    "QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES",
    PROCESSED_DIR / "seed_paraphrases_quality_aware_teacher_text_v1.jsonl",
)

TRAIN_CLEAN_FILE = globals().get("TRAIN_CLEAN_FILE", PROCESSED_DIR / "train_clean.jsonl")
VALIDATION_CLEAN_FILE = globals().get(
    "VALIDATION_CLEAN_FILE",
    PROCESSED_DIR / "validation_clean.jsonl",
)
TEST_CLEAN_FILE = globals().get("TEST_CLEAN_FILE", PROCESSED_DIR / "test_clean.jsonl")

cmd = [
    sys.executable,
    str(MERGE_AND_SPLIT_SCRIPT),
    "--seed-file", str(QUALITY_AWARE_SOURCE_CODE_SEEDS),
    "--seed-file", str(QUALITY_AWARE_TEACHER_TEXT_SEEDS),
    "--paraphrase-file", str(QUALITY_AWARE_SOURCE_CODE_PARAPHRASES),
    "--paraphrase-file", str(QUALITY_AWARE_TEACHER_TEXT_PARAPHRASES),
    "--train-file", str(TRAIN_CLEAN_FILE),
    "--validation-file", str(VALIDATION_CLEAN_FILE),
    "--test-file", str(TEST_CLEAN_FILE),
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("quality-aware split refresh completed with return code:", result.returncode)

globals()["MERGE_AND_SPLIT_SCRIPT"] = MERGE_AND_SPLIT_SCRIPT
globals()["TRAIN_CLEAN_FILE"] = TRAIN_CLEAN_FILE
globals()["VALIDATION_CLEAN_FILE"] = VALIDATION_CLEAN_FILE
globals()["TEST_CLEAN_FILE"] = TEST_CLEAN_FILE


#### Step L2 — Enrich Per-Entry Semantic Consistency Metrics

This step runs the late-stage semantic consistency enrichment over the refreshed canonical splits.

Metrics written for paraphrase rows:
- `semantic_similarity_to_seed`
- `bert_score_f1`
- `bleu_score_to_seed`
- `rouge_l_to_seed`
- `normalized_edit_distance`

Operational note:
- the script now runs in chunked, resume-friendly mode and flushes semantic cache progress incrementally
- CPU-safe default: `bert_score_f1` is skipped on the first pass and can be backfilled later if needed
- the script remains resume-safe through `semantic_consistency_cache.jsonl`
- recommended execution strategy: local first pass without `BERTScore`, then optional Google Cloud GPU backfill for `bert_score_f1`
- see `PQID/GCP_BERT_BACKFILL_STRATEGY.md` for the local-plus-cloud execution protocol


In [ ]:
SEMANTIC_COMPUTE_BERT_SCORE = True
SEMANTIC_PAIR_CHUNK_SIZE = 500
SEMANTIC_BATCH_SIZE_ST = 128
SEMANTIC_BATCH_SIZE_BERT = 8


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
ENRICH_SEMANTIC_CONSISTENCY_SCRIPT = globals().get(
    "ENRICH_SEMANTIC_CONSISTENCY_SCRIPT",
    ROOT / "PQID/scripts/enrich_semantic_consistency.py",
)

SEMANTIC_COMPUTE_BERT_SCORE = globals().get("SEMANTIC_COMPUTE_BERT_SCORE", False)
SEMANTIC_PAIR_CHUNK_SIZE = globals().get("SEMANTIC_PAIR_CHUNK_SIZE", 2000)
SEMANTIC_BATCH_SIZE_ST = globals().get("SEMANTIC_BATCH_SIZE_ST", 256)
SEMANTIC_BATCH_SIZE_BERT = globals().get("SEMANTIC_BATCH_SIZE_BERT", 16)

cmd = [
    sys.executable,
    str(ENRICH_SEMANTIC_CONSISTENCY_SCRIPT),
    "--pair-chunk-size", str(SEMANTIC_PAIR_CHUNK_SIZE),
    "--batch-size-st", str(SEMANTIC_BATCH_SIZE_ST),
    "--batch-size-bert", str(SEMANTIC_BATCH_SIZE_BERT),
]
if SEMANTIC_COMPUTE_BERT_SCORE:
    cmd.append("--compute-bert-score")
else:
    cmd.append("--skip-bert-score")

print("semantic consistency mode:")
print("  compute bert score:", SEMANTIC_COMPUTE_BERT_SCORE)
print("  pair chunk size:", SEMANTIC_PAIR_CHUNK_SIZE)
print("  sentence batch size:", SEMANTIC_BATCH_SIZE_ST)
print("  bert batch size:", SEMANTIC_BATCH_SIZE_BERT)

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("semantic consistency enrichment completed with return code:", result.returncode)

globals()["ENRICH_SEMANTIC_CONSISTENCY_SCRIPT"] = ENRICH_SEMANTIC_CONSISTENCY_SCRIPT
globals()["SEMANTIC_COMPUTE_BERT_SCORE"] = SEMANTIC_COMPUTE_BERT_SCORE
globals()["SEMANTIC_PAIR_CHUNK_SIZE"] = SEMANTIC_PAIR_CHUNK_SIZE
globals()["SEMANTIC_BATCH_SIZE_ST"] = SEMANTIC_BATCH_SIZE_ST
globals()["SEMANTIC_BATCH_SIZE_BERT"] = SEMANTIC_BATCH_SIZE_BERT


#### Step L3 — Compute Paraphrase Diversity Diagnostics

This step computes group-level paraphrase diversity diagnostics across the refreshed canonical splits.

Current outputs:
- `paraphrase_diversity.jsonl`
- `paraphrase_diversity_report.txt`

Current diversity summary measures:
- pairwise BLEU-4 mean and minimum per paraphrase group
- mean type-token ratio (TTR)
- within-group instruction-length coefficient of variation


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
COMPUTE_PARAPHRASE_DIVERSITY_SCRIPT = globals().get(
    "COMPUTE_PARAPHRASE_DIVERSITY_SCRIPT",
    ROOT / "PQID/scripts/compute_paraphrase_diversity.py",
)

cmd = [sys.executable, str(COMPUTE_PARAPHRASE_DIVERSITY_SCRIPT)]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("paraphrase diversity computation completed with return code:", result.returncode)

globals()["COMPUTE_PARAPHRASE_DIVERSITY_SCRIPT"] = COMPUTE_PARAPHRASE_DIVERSITY_SCRIPT


#### Step L4 — Audit Semantic-Metric Coverage Across Splits

Run this after `L2`. It verifies that the per-entry semantic fields are populated for paraphrase rows and left null for seed rows, and it records split-level metric coverage for the quality-aware corpus.


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
TRAIN_CLEAN_FILE = globals().get("TRAIN_CLEAN_FILE", PROCESSED_DIR / "train_clean.jsonl")
VALIDATION_CLEAN_FILE = globals().get(
    "VALIDATION_CLEAN_FILE",
    PROCESSED_DIR / "validation_clean.jsonl",
)
TEST_CLEAN_FILE = globals().get("TEST_CLEAN_FILE", PROCESSED_DIR / "test_clean.jsonl")

SPLIT_FILES = {
    "train": TRAIN_CLEAN_FILE,
    "validation": VALIDATION_CLEAN_FILE,
    "test": TEST_CLEAN_FILE,
}
SEMANTIC_EXPECT_BERT_SCORE = globals().get("SEMANTIC_COMPUTE_BERT_SCORE", False)
SEMANTIC_FIELDS = [
    "semantic_similarity_to_seed",
    "bert_score_f1",
    "bleu_score_to_seed",
    "rouge_l_to_seed",
    "normalized_edit_distance",
]

def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

for split_name, path in SPLIT_FILES.items():
    rows = load_jsonl(path)
    prompt_type_counts = Counter()
    paraphrase_rows = 0
    seed_rows = 0
    filled_counts = Counter()
    seed_null_ok = Counter()

    for row in rows:
        meta = row.get("metadata", {})
        prompt_type = meta.get("prompt_type", "<missing>")
        prompt_type_counts[prompt_type] += 1
        is_paraphrase = bool(meta.get("original_prompt"))
        if is_paraphrase:
            paraphrase_rows += 1
            for field in SEMANTIC_FIELDS:
                if meta.get(field) is not None:
                    filled_counts[field] += 1
        else:
            seed_rows += 1
            for field in SEMANTIC_FIELDS:
                if meta.get(field) is None:
                    seed_null_ok[field] += 1

    print(f"Split: {split_name}")
    print(f"  rows: {len(rows):,}")
    print(f"  seed rows: {seed_rows:,}")
    print(f"  paraphrase rows: {paraphrase_rows:,}")
    print("  prompt types:")
    for key, value in prompt_type_counts.most_common():
        print(f"    {key}: {value:,}")
    print("  paraphrase metric coverage:")
    for field in SEMANTIC_FIELDS:
        coverage_note = "required" if (field != "bert_score_f1" or SEMANTIC_EXPECT_BERT_SCORE) else "optional"
        print(f"    {field}: {filled_counts[field]:,} / {paraphrase_rows:,} ({coverage_note})")
    print("  seed null coverage:")
    for field in SEMANTIC_FIELDS:
        print(f"    {field}: {seed_null_ok[field]:,} / {seed_rows:,}")
    print()

print("Semantic consistency mode:")
print(f"  BERTScore expected for this run: {SEMANTIC_EXPECT_BERT_SCORE}")
print("Semantic consistency cache:", PROCESSED_DIR / "semantic_consistency_cache.jsonl")
print("Semantic consistency report:", PROCESSED_DIR / "semantic_consistency_report.txt")
print("Paraphrase diversity report:", PROCESSED_DIR / "paraphrase_diversity_report.txt")
print("Paraphrase diversity metrics:", PROCESSED_DIR / "paraphrase_diversity.jsonl")


### Stage M — Instruction Language Scope Audit

With the semantic layer and the pilot review scaffolding in place, the next local audit should characterize the **human-language distribution** of the instruction corpus.

This stage is intentionally heuristic and should be interpreted as an audit layer, not as a claim of perfect language identification.

Goals:
- estimate how English-dominant the corpus actually is
- detect clearly non-English or mixed-language rows
- distinguish natural-language prompts from multilingual comments/docstrings in `source_code` outputs
- produce joinable sidecar metadata for `input_human_language` and `output_human_language`


#### Step M1 — Audit Instruction Language Distribution

This local support cell builds a heuristic sidecar language-audit layer over the unified acceptance-gate manifest.

What this step writes:
- `instruction_language_audit_v1.jsonl`
- `instruction_language_audit_v1_summary.json`

What this step does not do:
- it does not rewrite the canonical seed/paraphrase artifacts
- it does not claim perfect language identification
- it provides a joinable audit layer keyed by `instruction_key`


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
INSTRUCTION_LANGUAGE_AUDIT_SCRIPT = globals().get(
    "INSTRUCTION_LANGUAGE_AUDIT_SCRIPT",
    ROOT / "PQID/scripts/03_instruction_generation/audit_instruction_language_distribution.py",
)

cmd = [sys.executable, str(INSTRUCTION_LANGUAGE_AUDIT_SCRIPT)]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("instruction language audit completed with return code:", result.returncode)

globals()["INSTRUCTION_LANGUAGE_AUDIT_SCRIPT"] = INSTRUCTION_LANGUAGE_AUDIT_SCRIPT


#### Step M2 — Inspect Language Audit Summary

This local audit cell summarizes the language-audit sidecar and prints sampled non-English or mixed-language examples for inspection.


In [ ]:
import json
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
INSTRUCTION_LANGUAGE_AUDIT_SUMMARY_FILE = globals().get(
    "INSTRUCTION_LANGUAGE_AUDIT_SUMMARY_FILE",
    PROCESSED_DIR / "instruction_language_audit_v1_summary.json",
)

if not INSTRUCTION_LANGUAGE_AUDIT_SUMMARY_FILE.exists():
    raise FileNotFoundError(
        f"Instruction language audit summary not found: {INSTRUCTION_LANGUAGE_AUDIT_SUMMARY_FILE}"
    )

summary = json.loads(INSTRUCTION_LANGUAGE_AUDIT_SUMMARY_FILE.read_text(encoding="utf-8"))
print("Instruction language audit summary")
print(f"  version: {summary['language_audit_version']}")
print(f"  manifest file: {summary['manifest_file']}")
print(f"  audit file: {summary['audit_file']}")
print(f"  rows: {summary['rows']:,}")
print("  input human languages:")
for key, value in sorted(summary.get("input_human_language_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  input human languages (resolved):")
for key, value in sorted(summary.get("input_human_language_resolved_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  input human script buckets:")
for key, value in sorted(summary.get("input_human_script_bucket_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  output human languages:")
for key, value in sorted(summary.get("output_human_language_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  output human languages (resolved):")
for key, value in sorted(summary.get("output_human_language_resolved_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  output human script buckets:")
for key, value in sorted(summary.get("output_human_script_bucket_counts", {}).items()):
    print(f"    {key}: {value:,}")
print("  output human language scopes:")
for key, value in sorted(summary.get("output_human_language_scope_counts", {}).items()):
    print(f"    {key}: {value:,}")
examples = summary.get("non_english_examples", [])
print(f"  sampled non-English or mixed examples: {len(examples):,}")
for row in examples[:10]:
    print(
        "    "
        f"instruction_key={row['instruction_key']}, "
        f"input_lang={row['input_human_language_resolved']}, "
        f"output_lang={row['output_human_language_resolved']}, "
        f"output_script={row['output_human_script_bucket']}, "
        f"scope={row['output_human_language_scope']}"
    )



## Stage N - License-Filtered Public Release Views

This stage makes the public-release decision auditable from inside the notebook. The full `550,314`-row instruction layer remains the construction-complete internal corpus, but public upload should use a license-filtered release view.

Release policy:
- `public_open`: permissive-license rows only
- `license_valid`: permissive plus copyleft plus the 702 manually reviewed `other`-license rows, with obligation-bearing rows kept in `public_open_with_obligations`
- exclude `no_license` and missing-license rows from public release; preserve the 18 missing-license rows in the internal-only manifest

This stage writes release-view files under `PQID/data/processed/release_views/` and keeps attribution manifests beside the exported splits.


#### Step N0 - Apply Q-Bridge MIT License Update

Run this post-freeze license-evidence update before auditing the public release views. Runjia Zeng added a root MIT `LICENSE` file to `runtsang/Q-Bridge` on `2026-04-28`; the update reclassifies the `72,888` Q-Bridge-derived instruction rows from `no_license` to `permissive` and records the evidence commit in row-level metadata. The script is idempotent and writes an evidence sidecar under `data/processed/license_evidence/`.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
qbridge_update_script = ROOT / "PQID/scripts/apply_qbridge_mit_license_update.py"

result = subprocess.run(
    [sys.executable, str(qbridge_update_script)],
    cwd=ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(result.stdout)


#### Step N1 - Audit Canonical Split License Categories

Count `license_category` values in the canonical construction splits before exporting any public release view.

Expected current totals:
- permissive: `311,724`
- copyleft: `7,356`
- no_license: `230,514`
- other: `702`
- missing: `18`


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")

RELEASE_SPLIT_FILES = {
    "train": PROCESSED_DIR / "train_clean.jsonl",
    "validation": PROCESSED_DIR / "validation_clean.jsonl",
    "test": PROCESSED_DIR / "test_clean.jsonl",
}

def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

license_counts_by_split = {}
license_counts_total = Counter()

for split_name, path in RELEASE_SPLIT_FILES.items():
    counts = Counter()
    rows = 0
    for row in iter_jsonl(path):
        rows += 1
        metadata = row.get("metadata") or {}
        category = metadata.get("license_category") or "<missing>"
        counts[category] += 1
        license_counts_total[category] += 1
    license_counts_by_split[split_name] = {"rows": rows, "license_category_counts": dict(sorted(counts.items()))}

print("Canonical split license-category audit")
for split_name, stats in license_counts_by_split.items():
    print(f"  {split_name}: rows={stats['rows']:,} categories={stats['license_category_counts']}")
print("  total:", dict(sorted(license_counts_total.items())))
print("  permissive only:", f"{license_counts_total['permissive']:,}")
print("  permissive + copyleft:", f"{license_counts_total['permissive'] + license_counts_total['copyleft']:,}")
print("  license_valid expected:", f"{license_counts_total['permissive'] + license_counts_total['copyleft'] + license_counts_total['other']:,}")
print("  internal-only missing-license rows:", f"{license_counts_total['<missing>']:,}")


#### Step N2 - Export `public_open` Release View

Export the strict permissive-only fallback package.

Outputs:
- `release_views/pqid_v1_public_open_train.jsonl`
- `release_views/pqid_v1_public_open_validation.jsonl`
- `release_views/pqid_v1_public_open_test.jsonl`
- `release_views/pqid_v1_public_open_summary.json`
- `release_views/pqid_v1_public_open_attribution_manifest.csv`


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
EXPORT_LICENSE_VALID_RELEASE_SCRIPT = globals().get(
    "EXPORT_LICENSE_VALID_RELEASE_SCRIPT",
    ROOT / "PQID/scripts/export_license_valid_release_views.py",
)

cmd = [
    sys.executable,
    str(EXPORT_LICENSE_VALID_RELEASE_SCRIPT),
    "--profile",
    "public_open",
    "--overwrite",
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("public_open release export completed with return code:", result.returncode)

globals()["EXPORT_LICENSE_VALID_RELEASE_SCRIPT"] = EXPORT_LICENSE_VALID_RELEASE_SCRIPT


#### Step N3 - Export Optional `license_valid` Release View

Export the broader license-resolved package. This includes permissive rows, copyleft rows, and the 702 manually reviewed `other`-license rows. Copyleft and reviewed-`other` rows remain explicitly marked as `public_open_with_obligations` and should not be presented as obligation-free.

Outputs:
- `release_views/pqid_v1_license_valid_train.jsonl`
- `release_views/pqid_v1_license_valid_validation.jsonl`
- `release_views/pqid_v1_license_valid_test.jsonl`
- `release_views/pqid_v1_license_valid_summary.json`
- `release_views/pqid_v1_license_valid_attribution_manifest.csv`


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
EXPORT_LICENSE_VALID_RELEASE_SCRIPT = globals().get(
    "EXPORT_LICENSE_VALID_RELEASE_SCRIPT",
    ROOT / "PQID/scripts/export_license_valid_release_views.py",
)

cmd = [
    sys.executable,
    str(EXPORT_LICENSE_VALID_RELEASE_SCRIPT),
    "--profile",
    "license_valid",
    "--overwrite",
]
result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.stderr.strip():
    print("stderr")
    print(result.stderr.strip())
print("license_valid release export completed with return code:", result.returncode)


#### Step N4 - Verify Release-View Integrity

Recount the exported release views and assert that no excluded license categories leaked into the public artifacts.

Closeout checks:
- `public_open` contains only `permissive`
- `license_valid` contains only `permissive`, `copyleft`, and manually reviewed `other`
- no `no_license` or missing-license rows appear in either release view


In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
RELEASE_VIEW_DIR = PROCESSED_DIR / "release_views"

def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

PROFILE_ALLOWED_CATEGORIES = {
    "pqid_v1_public_open": {"permissive"},
    "pqid_v1_license_valid": {"permissive", "copyleft", "other"},
}

release_integrity = {}
for stem, allowed_categories in PROFILE_ALLOWED_CATEGORIES.items():
    summary_path = RELEASE_VIEW_DIR / f"{stem}_summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    profile_counts = Counter()
    bucket_counts = Counter()
    split_counts = {}

    for split in ["train", "validation", "test"]:
        path = RELEASE_VIEW_DIR / f"{stem}_{split}.jsonl"
        rows = 0
        split_license_counts = Counter()
        split_bucket_counts = Counter()
        for row in iter_jsonl(path):
            rows += 1
            metadata = row.get("metadata") or {}
            category = metadata.get("license_category") or "<missing>"
            bucket = metadata.get("public_release_bucket") or "<missing>"
            split_license_counts[category] += 1
            split_bucket_counts[bucket] += 1
            profile_counts[category] += 1
            bucket_counts[bucket] += 1
        split_counts[split] = {
            "rows": rows,
            "license_category_counts": dict(sorted(split_license_counts.items())),
            "public_release_bucket_counts": dict(sorted(split_bucket_counts.items())),
        }

    observed_categories = set(profile_counts)
    disallowed = observed_categories - allowed_categories
    if disallowed:
        raise AssertionError(f"{stem} contains disallowed categories: {sorted(disallowed)}")
    if sum(profile_counts.values()) != summary["total_exported_rows"]:
        raise AssertionError(f"{stem} row count mismatch against summary")

    release_integrity[stem] = {
        "summary_exported_rows": summary["total_exported_rows"],
        "summary_excluded_rows": summary["total_excluded_rows"],
        "license_category_counts": dict(sorted(profile_counts.items())),
        "public_release_bucket_counts": dict(sorted(bucket_counts.items())),
        "splits": split_counts,
    }

release_integrity


#### Step N5 - Inspect Release Summaries And Attribution Manifests

List the release-view files and inspect the machine-readable summaries. The attribution manifests are part of the public package and should travel with any uploaded split files.


In [ ]:
import json
from pathlib import Path

ROOT = globals().get("ROOT", Path.cwd().resolve().parents[2])
PROCESSED_DIR = globals().get("PROCESSED_DIR", ROOT / "PQID/data/processed")
RELEASE_VIEW_DIR = PROCESSED_DIR / "release_views"

for path in sorted(RELEASE_VIEW_DIR.glob("pqid_v1_*")):
    print(f"{path.name:55s} {path.stat().st_size:>14,} bytes")

print("\nRelease summaries")
for stem in ["pqid_v1_public_open", "pqid_v1_license_valid"]:
    summary_path = RELEASE_VIEW_DIR / f"{stem}_summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(f"\n{stem}")
    print(f"  exported rows: {summary['total_exported_rows']:,}")
    print(f"  excluded rows: {summary['total_excluded_rows']:,}")
    print(f"  exported categories: {summary['exported_license_category_counts']}")
    print(f"  excluded categories: {summary['excluded_license_category_counts']}")
    print(f"  attribution manifest: {summary['attribution_manifest_file']}")


## Next Step

Stage K, Stage L, Stage M, and Stage N have now established a pilot-reviewed, fully audited, and license-filtered instruction corpus layer for PQID.

Completed components:
- unified acceptance-gate manifest over the full routed instruction corpus
- stratified pilot review set
- human review sheet and model-assisted second-opinion layer
- disagreement adjudication for the pilot review
- semantic-consistency metrics for all paraphrase rows, including BERTScore
- paraphrase-diversity diagnostics
- instruction-language scope audit
- license-filtered public release views and attribution manifests

What remains is no longer basic pipeline construction, but final packaging and manuscript work:
- preserve the semantic-metric layer as an audit aid rather than a blind acceptance rule
- preserve the acceptance-gate pilot as evidence of review policy and observed failure modes
- treat the full routed instruction corpus, together with its review and audit sidecars, as the internal object of record for publication documentation
- use `pqid_v1_license_valid_*` as the recommended public upload after documenting copyleft and reviewed-`other` obligations in the dataset card; keep `pqid_v1_public_open_*` as a strict permissive-only fallback

At this point, the branch split remains an implementation device. The publication-facing object of record is the routed instruction corpus plus its acceptance-gate, semantic-consistency, diversity, language-audit, and release-view artifacts.
